# MUFASA — Public Reproducibility Notebook

**Manuscript status:** Under review  
**Task:** Multi-step hourly solar radiation forecasting (08:00–18:00; 11 horizons)  
**Sites:** Busan, Daegu, Daejeon, Gwangju, Incheon, and Seoul

This notebook is the public, review-stage reproducibility version of the MUFASA workflow. It is organized as a sequence of short execution blocks rather than a few monolithic cells. The code keeps the original experimental logic but removes workstation-specific paths, clears notebook outputs, and uses concise English comments.

### Experimental boundary

- **2016–2018:** primary training
- **2019:** development, hyperparameter selection, seed policy, and aggregation calibration
- **2020:** locked test year
- **Target-day solar radiation is never used as an input.**
- The **oracle-weather protocol** treats observed target-horizon meteorology as a perfectly accurate weather forecast. Results under this protocol are conditional upper-bound results and should not be interpreted as operational NWP-driven performance.
- Full paper reproduction requires the exact six-site dataset. Paper mode fails closed when any expected city is missing.

> **Review-stage notice:** Results and manuscript-facing labels may be updated during peer review. Do not treat this repository snapshot as the final published software release.


## Before running

The safe default is **smoke mode**, which is intended only to verify data loading, tensor shapes, and execution contracts. It is **not** suitable for reporting paper results.

For the full manuscript configuration, restart the kernel and set `MUFASA_RUN_MODE=paper`, `MUFASA_SPEED_PROFILE=thorough`, and `MUFASA_REQUIRE_CUDA=1` before running the notebook from the top. The full benchmark/HPO workflow is computationally expensive.

The notebook writes generated artifacts under `outputs/`. Some downstream analysis stages recreate their own subdirectories, so do not store manual files inside generated output folders.


In [ ]:
# Safe public default. Override these variables before the setup section for a full paper run.
import os

os.environ.setdefault("MUFASA_RUN_MODE", "smoke")
os.environ.setdefault("MUFASA_SPEED_PROFILE", "fast")
os.environ.setdefault("MUFASA_REQUIRE_CUDA", "0")


## Execution map

Run the notebook **top to bottom** for a complete reproduction. During development, individual stages can be rerun after their upstream dependencies have been executed.

1. Runtime and experiment contract  
2. Data, chronology, and information boundaries  
3. MUFASA nonlinear expert and training objective  
4. Development selection and aggregation  
5. Locked refit and 2020 evaluation  
6. Matched-budget benchmark suite  
7. Ablation, diagnostics, inference, and XAI  
8. Publication artifacts  
9. Oracle-weather manuscript XAI  
10. Multi-horizon statistical analysis

Each code cell is intentionally scoped to a manageable unit. Large model-definition and analysis blocks are split at top-level boundaries without changing their execution order.


## 1. Runtime and experiment contract

Configure deterministic execution, hardware guards, output paths, and the experiment-wide contract. Run this section first after choosing a runtime profile.

**Run note.** Execute the cells in this section in order. Objects created here are consumed by later sections; the notebook intentionally avoids hidden state restoration from unpublished artifacts.


In [ ]:
# Environment, Dependencies, and Reproducibility Controls
import importlib.util
# 2. Imports, reproducibility, profiles, large fonts, and strict boundaries
import copy
import gc
import hashlib
import json
import math
import os
import platform
import random
import re
import time
import traceback
import warnings
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

# Repository-relative paths keep the public notebook portable.
NOTEBOOK_CWD = Path.cwd().resolve()
if (NOTEBOOK_CWD / "data").exists():
    REPO_ROOT = NOTEBOOK_CWD
elif NOTEBOOK_CWD.name == "notebooks" and (NOTEBOOK_CWD.parent / "data").exists():
    REPO_ROOT = NOTEBOOK_CWD.parent
else:
    REPO_ROOT = NOTEBOOK_CWD
DATA_DIR = Path(os.getenv("MUFASA_DATA_DIR", str(REPO_ROOT / "data"))).resolve()
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".cache" / "matplotlib"))
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2, friedmanchisquare, norm, wilcoxon
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")
RUN_MODE = os.getenv("MUFASA_RUN_MODE", "paper").strip().lower()
if RUN_MODE not in {"paper", "smoke"}:
    raise ValueError("MUFASA_RUN_MODE must be 'paper' or 'smoke'.")
SMOKE = RUN_MODE == "smoke"
SPEED_PROFILE = os.getenv("MUFASA_SPEED_PROFILE", "balanced").strip().lower()
if SPEED_PROFILE not in {"fast", "balanced", "thorough"}:
    raise ValueError("MUFASA_SPEED_PROFILE must be fast, balanced, or thorough.")
SEED = 2026
MAX_LOOKBACK_DAYS = 28
TARGET_HOURS = np.arange(8, 19)
N_HORIZONS = len(TARGET_HOURS)
REGIME_NAMES = ("low_variability", "variable", "high_variability")
TRAIN_END = pd.Timestamp("2018-12-31")
VAL_START = pd.Timestamp("2019-01-01")
VAL_SELECT_END = pd.Timestamp("2019-09-30")
VAL_CAL_START = pd.Timestamp("2019-10-01")
VAL_END = pd.Timestamp("2019-12-31")
TEST_START = pd.Timestamp("2020-01-01")
TEST_END = pd.Timestamp("2020-12-31")
EXPECTED_DATES = pd.date_range("2016-01-01", "2020-12-31", freq="D")
GPU_BATCH_SIZE = 64 if SMOKE else int(os.getenv("MUFASA_GPU_BATCH", "256"))
BO_TRIALS = 2 if SMOKE else int(os.getenv(
    "MUFASA_BO_TRIALS", {"fast": 12, "balanced": 24, "thorough": 36}[SPEED_PROFILE]
))
BO_INITIAL_RANDOM = min(5, BO_TRIALS)
BO_POOL_SIZE = 256 if SMOKE else 2048
BO_LOW_EPOCHS = 2 if SMOKE else int(os.getenv(
    "MUFASA_BO_LOW_EPOCHS", {"fast": 32, "balanced": 44, "thorough": 60}[SPEED_PROFILE]
))
BO_LOW_PATIENCE = 2 if SMOKE else max(5, BO_LOW_EPOCHS // 5)
BO_PROMOTE = 1 if SMOKE else int(os.getenv("MUFASA_BO_PROMOTE", "2"))
MUFASA_MAX_EPOCHS = 3 if SMOKE else int(os.getenv("MUFASA_MAX_EPOCHS", "150"))
MUFASA_PATIENCE = 3 if SMOKE else int(os.getenv("MUFASA_PATIENCE", "20"))
MC_PASSES = 1 if SMOKE else int(os.getenv(
    "MUFASA_MC_PASSES", {"fast": 8, "balanced": 16, "thorough": 24}[SPEED_PROFILE]
))
BO_VALIDATION_MC_PASSES = 1 if SMOKE else min(8, MC_PASSES)
BENCHMARK_MAX_EPOCHS = MUFASA_MAX_EPOCHS
BENCHMARK_PATIENCE = MUFASA_PATIENCE
BASELINE_BO_TRIALS = BO_TRIALS
BASELINE_BO_LOW_EPOCHS = BO_LOW_EPOCHS
BASELINE_BO_PROMOTE = BO_PROMOTE
PROPOSED_SEEDS = [11] if SMOKE else [11, 29, 47]
BENCHMARK_SEEDS = PROPOSED_SEEDS.copy()
FIG_DPI = 180 if SMOKE else 600
BOOTSTRAP_REPS = 200 if SMOKE else 5000
RUN_DEEP_BENCHMARKS = os.getenv("MUFASA_RUN_DEEP", "1") == "1"
RUN_ABLATIONS = os.getenv("MUFASA_RUN_ABLATION", "1") == "1"
RUN_XAI = os.getenv("MUFASA_RUN_XAI", "1") == "1"
REQUIRE_CUDA = os.getenv("MUFASA_REQUIRE_CUDA", "0" if SMOKE else "1") == "1"
REQUIRE_COMPLETE_BENCHMARK = os.getenv("MUFASA_REQUIRE_COMPLETE", "0" if SMOKE else "1") == "1"
RUN_SHAP = os.getenv("MUFASA_RUN_SHAP", "1") == "1"
SOLAR_UNIT = os.getenv("MUFASA_SOLAR_UNIT", "MJ_per_m2_per_hour")
EVIDENCE_STATUS = os.getenv("MUFASA_EVIDENCE_STATUS", "under_review_locked_2020_test")
CONFIRMATORY_UNTOUCHED_TEST = EVIDENCE_STATUS == "confirmatory_untouched"
BENCHMARK_CONTRACT_VERSION = "v3_return_shape_finite_fair_hpo_fail_closed"
INVALIDATED_PREVIOUS_DEEP_TRIALS = 3888
INVALIDATED_PREVIOUS_DEEP_SCENARIOS = ("strict_history", "oracle_weather")
FEATURE_DESIGN_VERSION = "literature_compact_v2"
USE_ROLLING_ORIGIN_HPO_AUDIT = os.getenv("MUFASA_ROLLING_HPO_AUDIT", "1") == "1"
ALLOW_TARGET_WEATHER_TEACHER = False
if SMOKE:
    BO_TRIALS = int(os.getenv("MUFASA_SMOKE_BO_TRIALS", str(BO_TRIALS)))
    BO_INITIAL_RANDOM = min(BO_TRIALS, BO_INITIAL_RANDOM)
    BO_LOW_EPOCHS = int(os.getenv("MUFASA_SMOKE_LOW_EPOCHS", str(BO_LOW_EPOCHS)))
    BO_PROMOTE = min(BO_TRIALS, int(os.getenv("MUFASA_SMOKE_PROMOTE", str(BO_PROMOTE))))
    MUFASA_MAX_EPOCHS = int(os.getenv("MUFASA_SMOKE_MAX_EPOCHS", str(MUFASA_MAX_EPOCHS)))
    MUFASA_PATIENCE = min(MUFASA_MAX_EPOCHS, MUFASA_PATIENCE)
    BENCHMARK_MAX_EPOCHS = MUFASA_MAX_EPOCHS
    BENCHMARK_PATIENCE = MUFASA_PATIENCE
    BASELINE_BO_TRIALS = BO_TRIALS
    BASELINE_BO_LOW_EPOCHS = BO_LOW_EPOCHS
    BASELINE_BO_PROMOTE = BO_PROMOTE
    SMOKE_TRAIN_LIMIT = int(os.getenv("MUFASA_SMOKE_TRAIN_LIMIT", "0"))
else:
    SMOKE_TRAIN_LIMIT = 0
OUTPUT_DIR = Path(os.getenv("MUFASA_OUTPUT_DIR", str(REPO_ROOT / "outputs"))).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SITE_COORDINATES = {
    "Busan": (35.1796, 129.0756), "Daegu": (35.8714, 128.6014),
    "Daejeon": (36.3504, 127.3845), "Gwangju": (35.1595, 126.8526),
    "Incheon": (37.4563, 126.7052), "Seoul": (37.5665, 126.9780),
}
FORECAST_WEATHER_COLUMNS = {
    # Example only: "Cloud": "Cloud_Forecast_Issued_Previous_Day"
    # Never map these names to observed target-day Temp/Humi/WS/WD/Solar columns.
}
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": FIG_DPI, "font.size": 19,
    "axes.titlesize": 25, "axes.labelsize": 21, "xtick.labelsize": 17,
    "ytick.labelsize": 17, "legend.fontsize": 15, "figure.titlesize": 27,
    "axes.titleweight": "bold",
})
sns.set_theme(style="whitegrid", context="talk", font_scale=1.16)
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception as exc:
        _torch_seed_note = type(exc).__name__
def save_figure(fig, stem: str) -> None:
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=FIG_DPI, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
def metric_set(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    yt = np.asarray(y_true, dtype=float).ravel()
    yp = np.asarray(y_pred, dtype=float).ravel()
    return {
        "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
        "MAE": float(mean_absolute_error(yt, yp)),
        "R2": float(r2_score(yt, yp)),
        "MBE": float(np.mean(yp - yt)),
        "nRMSE_mean": float(np.sqrt(mean_squared_error(yt, yp)) / max(float(np.mean(yt)), 1e-8)),
    }
def positions(mask: np.ndarray) -> np.ndarray:
    return np.where(np.asarray(mask))[0]
set_seed(SEED)
print(f"RUN_MODE={RUN_MODE}; profile={SPEED_PROFILE}; output={OUTPUT_DIR.resolve()}; Solar unit={SOLAR_UNIT}")
print(f"Python={platform.python_version()}, NumPy={np.__version__}, pandas={pd.__version__}")
TORCH_AVAILABLE = importlib.util.find_spec("torch") is not None
if RUN_MODE == "paper" and not TORCH_AVAILABLE:
    raise ImportError(
        "Paper mode requires CUDA PyTorch. Install the build matching your CUDA driver "
        "from https://pytorch.org/get-started/locally/."
    )
if TORCH_AVAILABLE:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if REQUIRE_CUDA and DEVICE.type != "cuda":
        raise RuntimeError(
            "MUFASA_REQUIRE_CUDA=1 but CUDA is unavailable. This guard prevents an "
            "accidental long CPU run. Set MUFASA_REQUIRE_CUDA=0 only for debugging."
        )
    USE_AMP = bool(DEVICE.type == "cuda" and os.getenv("MUFASA_AMP", "1") == "1")
    if DEVICE.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        try:
            torch.set_float32_matmul_precision("high")
        except Exception as exc:
            _matmul_precision_note = type(exc).__name__
    print(
        f"device={DEVICE}; AMP={USE_AMP}; batch={GPU_BATCH_SIZE}; "
        f"BO trials={BO_TRIALS}; MC passes={MC_PASSES}"
    )
else:
    DEVICE, USE_AMP = "cpu", False
    print("PyTorch unavailable: structural smoke mode uses Ridge only; never report it as MUFASA.")
RUNTIME_ROWS: List[Dict[str, Any]] = []
def record_runtime(stage, site, model, status, seconds, error="", device=None):
    RUNTIME_ROWS.append({
        "Stage": stage, "Site": site, "Model": model,
        "Device": device or (str(DEVICE) if ("deep" in stage or model == "MUFASA") else "CPU-fast"),
        "Status": status, "Seconds": float(seconds), "Error": error,
    })


In [ ]:
# Global Experiment Contract
STRUCTURE_NEAR_BEST_TOL = 0.005
STRUCTURE_TOL_SENSITIVITY = (0.0, 0.0025, 0.005, 0.01)
STRUCTURE_SCREEN_SEEDS = [11, 29, 47]
ACTIVE_STRUCTURE_SEEDS = [11] if SMOKE else STRUCTURE_SCREEN_SEEDS.copy()
FINAL_SEEDS = [11, 29, 47]
ACTIVE_FINAL_SEEDS = [11] if SMOKE else FINAL_SEEDS.copy()
HPO_SEED = 2026
HPO_TOP_K = 2
CANONICAL_CORE_CANDIDATE = "C2-MUFASA-BiGRU"
CORE_SELECTION_POLICY = "canonical_c2_plus_best_2019_challenger"
CANONICAL_CHOICE_RETROSPECTIVE = True

if SPEED_PROFILE == "thorough":
    FINAL_BO_TRIALS, BO_INITIAL_RANDOM, BO_LOW_EPOCHS, BO_PROMOTE = 60, 8, 60, 3
else:
    FINAL_BO_TRIALS, BO_INITIAL_RANDOM, BO_LOW_EPOCHS, BO_PROMOTE = 36, 6, 48, 2
if SMOKE:
    FINAL_BO_TRIALS = int(os.getenv("MUFASA_SMOKE_BO_TRIALS", "2"))
    BO_INITIAL_RANDOM = min(FINAL_BO_TRIALS, 2)
    BO_LOW_EPOCHS = int(os.getenv("MUFASA_SMOKE_LOW_EPOCHS", "2"))
    BO_PROMOTE = 1

FINAL_MAX_EPOCHS = 3 if SMOKE else 150
FINAL_PATIENCE = 3 if SMOKE else 20
STRUCTURE_SCREEN_EPOCHS = 2 if SMOKE else 52
STRUCTURE_SCREEN_PATIENCE = 2 if SMOKE else 12
FREE_RUN_MIN_EPOCHS = 1 if SMOKE else 10
BLOCK_BOOTSTRAP_REPS = 200 if SMOKE else 5000
BLOCK_BOOTSTRAP_LENGTH = 7
WEATHER_SCENARIO = os.getenv("MUFASA_WEATHER_SCENARIO", "strict_history")
if WEATHER_SCENARIO not in {"strict_history", "oracle_weather", "archived_forecast"}:
    raise ValueError("WEATHER_SCENARIO must be strict_history, oracle_weather, or archived_forecast")
RUN_ORACLE = os.getenv("MUFASA_RUN_ORACLE", "1") == "1"
RUN_ARCHIVED = os.getenv("MUFASA_RUN_ARCHIVED", "1") == "1"


BASELINE_BO_TRIALS = FINAL_BO_TRIALS
BASELINE_BO_LOW_EPOCHS = BO_LOW_EPOCHS
BASELINE_BO_PROMOTE = BO_PROMOTE
BENCHMARK_MAX_EPOCHS = FINAL_MAX_EPOCHS
BENCHMARK_PATIENCE = FINAL_PATIENCE
BENCHMARK_SEEDS = ACTIVE_FINAL_SEEDS.copy()
BO_POOL_SIZE = 128 if SMOKE else 2048
BO_LOW_PATIENCE = 2 if SMOKE else max(8, BO_LOW_EPOCHS // 5)
MUFASA_MAX_EPOCHS = FINAL_MAX_EPOCHS
MUFASA_PATIENCE = FINAL_PATIENCE

OUTPUT_DIR = Path(os.getenv("MUFASA_OUTPUT_DIR", "MUFASA_publication_ready_outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_LABEL = "structural_smoke_not_for_paper" if SMOKE else "paper_protocol"
print({
    "run_label": RUN_LABEL, "screen_seeds": ACTIVE_STRUCTURE_SEEDS,
    "final_seeds": ACTIVE_FINAL_SEEDS, "BO_trials": FINAL_BO_TRIALS,
    "strict_primary": True, "oracle_upper_bound": RUN_ORACLE,
})


## 2. Data, chronology, and information boundaries

Load the city datasets, verify the 2016–2020 chronology, construct physically structured features, and separate strict-history, oracle-weather, and archived-forecast scenarios.

**Run note.** Execute the cells in this section in order. Objects created here are consumed by later sections; the notebook intentionally avoids hidden state restoration from unpublished artifacts.


In [ ]:
# Data Loading and Integrity Checks — part 1/2
REQUIRED_COLUMNS = {"Year", "Month", "Day", "Hour", "Temp", "Humi", "WS", "WD", "Solar"}
def candidate_data_roots() -> List[Path]:
    """Return repository-first data roots with an optional environment override."""
    override = os.getenv("MUFASA_DATA_DIR", "").strip()
    roots = [Path(override)] if override else [
        DATA_DIR,
        REPO_ROOT / "data",
        REPO_ROOT,
        Path.cwd(),
    ]
    out, seen = [], set()
    for root in roots:
        root = Path(root).expanduser()
        key = str(root.resolve()) if root.exists() else str(root)
        if key.lower() not in seen:
            seen.add(key.lower())
            out.append(root)
    return out
def infer_site_name(path: Path) -> str:
    name = re.sub(r"\(\d+\)", "", path.stem, flags=re.I)
    name = re.sub(r"_2016_2020_complete.*$", "", name, flags=re.I)
    name = re.sub(r"_imputed_fixed.*$", "", name, flags=re.I)
    return name.strip("_ -")
def discover_site_files() -> List[Path]:
    override = os.getenv("MUFASA_FILE_PATTERN", "").strip()
    patterns = [override] if override else [
        "*_imputed_fixed*.csv",
        "*_2016_2020_complete*.csv",
    ]
    found = []
    for root in candidate_data_roots():
        if not root.exists():
            continue
        for pattern in patterns:
            found.extend(root.glob(pattern))
    by_resolved = {
        str(path.resolve()).lower(): path.resolve()
        for path in found if path.is_file()
    }
    by_site: Dict[str, List[Path]] = {}
    for path in by_resolved.values():
        by_site.setdefault(infer_site_name(path), []).append(path)
    duplicates = {site: paths for site, paths in by_site.items() if len(paths) > 1}
    if duplicates:
        details = {site: [p.name for p in paths] for site, paths in duplicates.items()}
        raise FileExistsError(
            "Multiple candidate CSVs were found for the same city. "
            "Set MUFASA_FILE_PATTERN to select exactly one file family. "
            f"Duplicates={details}"
        )
    paths = sorted((paths[0] for paths in by_site.values()), key=lambda p: p.name.lower())
    if not paths:
        raise FileNotFoundError(
            "No city CSV was found. Place *_imputed_fixed.csv or "
            "*_2016_2020_complete*.csv beside the notebook, or set MUFASA_FILE_PATTERN."
        )
    return paths
@dataclass
class DailySiteData:
    site: str
    path: Path
    latitude: float
    longitude: float
    dates: pd.DatetimeIndex
    solar: np.ndarray
    weather: np.ndarray
    weather_names: Tuple[str, ...]
    forecast_weather: Optional[np.ndarray]
    forecast_names: Tuple[str, ...]
    audit: Dict[str, Any]
def load_site_csv(path: Path) -> DailySiteData:
    site = infer_site_name(path)
    frame = pd.read_csv(path)
    missing = REQUIRED_COLUMNS - set(frame.columns)
    if missing:
        raise ValueError(f"{site}: missing required columns {sorted(missing)}")
    frame = frame.copy()
    frame["date"] = pd.to_datetime(frame[["Year", "Month", "Day"]], errors="raise")
    if frame.duplicated(["date", "Hour"]).any():
        raise ValueError(f"{site}: duplicate date-hour rows exist.")
    hours = np.sort(frame["Hour"].unique())
    if not np.array_equal(hours, TARGET_HOURS):
        raise ValueError(f"{site}: expected hours 08–18, found {hours.tolist()}.")
    counts = frame.groupby("date")["Hour"].count()
    if not (counts == N_HORIZONS).all():
        raise ValueError(f"{site}: incomplete 11-point daily curves exist.")
    dates = pd.DatetimeIndex(sorted(frame["date"].unique()))
    if not dates.equals(EXPECTED_DATES):
        raise ValueError(
            f"{site}: expected daily coverage 2016-01-01–2020-12-31; "
            f"missing={len(EXPECTED_DATES.difference(dates))}, "
            f"extra={len(dates.difference(EXPECTED_DATES))}."
        )
    if frame[list(REQUIRED_COLUMNS)].isna().any().any():
        raise ValueError(
            f"{site}: missing values remain. Impute separately within each chronological "
            "training fold; global two-sided interpolation is forbidden."
        )

    if {"Latitude", "Longitude"}.issubset(frame.columns):
        lat_values = frame["Latitude"].dropna().unique()
        lon_values = frame["Longitude"].dropna().unique()
        if len(lat_values) != 1 or len(lon_values) != 1:
            raise ValueError(f"{site}: Latitude/Longitude must be constant within a file.")
        latitude, longitude = float(lat_values[0]), float(lon_values[0])
    elif site in SITE_COORDINATES:
        latitude, longitude = SITE_COORDINATES[site]
    else:
        raise KeyError(
            f"{site}: coordinates are required for astronomical features. "
            "Add constant Latitude/Longitude columns or extend SITE_COORDINATES."
        )

    wd_rad = np.deg2rad(frame["WD"].to_numpy(float))
    frame["wind_u"] = frame["WS"].to_numpy(float) * np.cos(wd_rad)
    frame["wind_v"] = frame["WS"].to_numpy(float) * np.sin(wd_rad)

    
    temp = frame["Temp"].to_numpy(float)
    rh = np.clip(frame["Humi"].to_numpy(float), 0.1, 100.0)
    magnus_a, magnus_b = 17.625, 243.04
    magnus_gamma = np.log(rh / 100.0) + magnus_a * temp / (magnus_b + temp)
    dew_point = magnus_b * magnus_gamma / np.maximum(magnus_a - magnus_gamma, 1e-6)
    frame["DewPoint"] = dew_point
    frame["VaporPressure"] = 6.1094 * np.exp(
        magnus_a * dew_point / np.maximum(magnus_b + dew_point, 1e-6)
    )
    frame["DewPointDepression"] = temp - dew_point
    weather_names = (
        "Temp", "Humi", "WS", "wind_u", "wind_v",
        "DewPoint", "VaporPressure", "DewPointDepression",
    )

    def pivot(column: str) -> np.ndarray:
        return (
            frame.pivot(index="date", columns="Hour", values=column)
            .reindex(index=dates, columns=TARGET_HOURS)
            .to_numpy(float)
        )

    solar = pivot("Solar")
    weather = np.stack([pivot(name) for name in weather_names], axis=-1)
    forecast, forecast_names = None, tuple()
    if FORECAST_WEATHER_COLUMNS:
        forbidden = {
            logical: physical for logical, physical in FORECAST_WEATHER_COLUMNS.items()
            if physical in {"Temp", "Humi", "WS", "WD", "Solar"}
        }
        if forbidden:
            raise ValueError(
                "Target-day observations cannot masquerade as forecasts: "
                f"{forbidden}"
            )
        absent = [
            physical for physical in FORECAST_WEATHER_COLUMNS.values()
            if physical not in frame.columns
        ]
        if absent:
            raise ValueError(f"{site}: archived forecast columns not found: {absent}")
        forecast_names = tuple(FORECAST_WEATHER_COLUMNS.keys())
        forecast = np.stack(
            [pivot(FORECAST_WEATHER_COLUMNS[name]) for name in forecast_names],
            axis=-1,
        )

    arrays = [solar, weather] + ([] if forecast is None else [forecast])
    if any(not np.isfinite(array).all() for array in arrays):
        raise ValueError(f"{site}: non-finite values remain after loading.")
    if (solar < 0).any():
        raise ValueError(f"{site}: Solar contains negative values.")

    audit = {
        "Site": site,
        "File": path.name,
        "Rows": int(len(frame)),
        "Days": int(len(dates)),
        "Start": str(dates.min().date()),
        "End": str(dates.max().date()),
        "Latitude": latitude,
        "Longitude": longitude,
        "Hours_per_day": int(counts.iloc[0]),
        "Solar_mean": float(solar.mean()),
        "Solar_std": float(solar.std()),
        "Solar_zero_percent": float(100 * np.mean(solar == 0)),
        "Solar_unit_config": SOLAR_UNIT,
        "Solar_min": float(solar.min()),
        "Solar_max": float(solar.max()),
        "Duplicate_date_hour": int(frame.duplicated(["date", "Hour"]).sum()),
        "Missing_values": int(frame[list(REQUIRED_COLUMNS)].isna().sum().sum()),
        "Target_day_weather_mode": (
            "archived forecast" if forecast is not None else "none"
        ),
        "Independent_pipeline": True,
        "SHA256": hashlib.sha256(path.read_bytes()).hexdigest(),
    }
    return DailySiteData(
        site=site,
        path=path,
        latitude=latitude,
        longitude=longitude,
        dates=dates,
        solar=solar.astype(np.float32),
        weather=weather.astype(np.float32),
        weather_names=weather_names,
        forecast_weather=None if forecast is None else forecast.astype(np.float32),
        forecast_names=forecast_names,
        audit=audit,
    )
SITE_PATHS = discover_site_files()
SITE_DATA = {infer_site_name(path): load_site_csv(path) for path in SITE_PATHS}
requested_sites = tuple(
    token.strip() for token in os.getenv("MUFASA_SITE_FILTER", "").split(",")
    if token.strip()
)
if requested_sites:
    unknown_sites = sorted(set(requested_sites) - set(SITE_DATA))
    if unknown_sites:
        raise KeyError(f"MUFASA_SITE_FILTER contains unknown cities: {unknown_sites}")
    SITE_DATA = {site: SITE_DATA[site] for site in requested_sites}
SITES = tuple(sorted(SITE_DATA))


In [ ]:
# Data Loading and Integrity Checks — part 2/2
EXPECTED_SITES = {"Busan", "Daegu", "Daejeon", "Gwangju", "Incheon", "Seoul"}
missing_sites = sorted(EXPECTED_SITES - set(SITES))
extra_sites = sorted(set(SITES) - EXPECTED_SITES)
if RUN_MODE == "paper" and (missing_sites or extra_sites):
    raise RuntimeError(
        "Paper mode requires the exact six-site dataset. "
        f"Missing={missing_sites}; unexpected={extra_sites}"
    )
if missing_sites or extra_sites:
    warnings.warn(
        f"Smoke/debug run uses a noncanonical site set. Missing={missing_sites}; "
        f"unexpected={extra_sites}"
    )
DATA_AUDIT = pd.DataFrame([SITE_DATA[site].audit for site in SITES])
display(DATA_AUDIT.drop(columns="SHA256").round(5))
DATA_AUDIT.to_csv(OUTPUT_DIR / "MUFASA_data_audit.csv", index=False)
print(f"Independent city pipelines={len(SITES)}: {SITES}")


In [ ]:
# Deterministic Solar Geometry and Fitted Reference Potential
def day_fourier(dates: pd.DatetimeIndex, harmonics: int = 4) -> np.ndarray:
    dates = pd.DatetimeIndex(dates)
    denom = np.where(dates.is_leap_year, 366.0, 365.0)
    phase = 2 * np.pi * (dates.dayofyear.to_numpy() - 1) / denom
    columns = []
    for k in range(1, harmonics + 1):
        columns.extend([np.sin(k * phase), np.cos(k * phase)])
    return np.column_stack(columns)


def solar_geometry(
    dates: pd.DatetimeIndex,
    latitude: float,
    longitude: float,
) -> np.ndarray:
    # Spencer/NOAA-style deterministic approximation; Asia/Seoul standard meridian=135°E.
    dates = pd.DatetimeIndex(dates)
    doy = dates.dayofyear.to_numpy(float)[:, None]
    hour = TARGET_HOURS.astype(float)[None, :]
    gamma = 2 * np.pi / 365.0 * (doy - 1 + (hour - 12.0) / 24.0)
    declination = (
        0.006918
        - 0.399912 * np.cos(gamma)
        + 0.070257 * np.sin(gamma)
        - 0.006758 * np.cos(2 * gamma)
        + 0.000907 * np.sin(2 * gamma)
        - 0.002697 * np.cos(3 * gamma)
        + 0.00148 * np.sin(3 * gamma)
    )
    equation_of_time = 229.18 * (
        0.000075
        + 0.001868 * np.cos(gamma)
        - 0.032077 * np.sin(gamma)
        - 0.014615 * np.cos(2 * gamma)
        - 0.040849 * np.sin(2 * gamma)
    )
    solar_minutes = hour * 60.0 + equation_of_time + 4.0 * (longitude - 135.0)
    hour_angle = np.deg2rad(solar_minutes / 4.0 - 180.0)
    latitude_rad = np.deg2rad(latitude)
    sin_elevation = (
        np.sin(latitude_rad) * np.sin(declination)
        + np.cos(latitude_rad) * np.cos(declination) * np.cos(hour_angle)
    )
    sin_elevation = np.clip(sin_elevation, 0.0, 1.0)
    zenith = np.arccos(np.clip(sin_elevation, 0.0, 1.0))
    extraterrestrial = (
        1.0 + 0.033 * np.cos(2 * np.pi * doy / 365.0)
    ) * sin_elevation
    relative_airmass = np.clip(1.0 / np.maximum(sin_elevation, 0.08), 1.0, 12.0)
    return np.stack(
        [
            sin_elevation,
            np.cos(zenith),
            extraterrestrial,
            relative_airmass,
        ],
        axis=-1,
    ).astype(np.float32)


class AstronomyPotential:
    def __init__(self, quantile: float = 0.975, alpha: float = 1.0):
        self.quantile = quantile
        self.alpha = alpha

    @staticmethod
    def design(dates: pd.DatetimeIndex, geometry: np.ndarray) -> np.ndarray:
        fourier = day_fourier(dates, 4)
        pieces = [fourier]
        for channel in range(3):
            g = geometry[:, :, channel]
            pieces.append(g)
            pieces.append(
                np.einsum("nh,nk->nhk", g, fourier).reshape(len(dates), -1)
            )
        return np.concatenate(pieces, axis=1)

    def fit(
        self,
        dates: pd.DatetimeIndex,
        solar: np.ndarray,
        geometry: np.ndarray,
    ):
        x = self.design(dates, geometry)
        self.models_, self.offsets_, self.upper_ = [], [], []
        for h in range(N_HORIZONS):
            model = Ridge(alpha=self.alpha).fit(x, solar[:, h])
            fitted = model.predict(x)
            residual_offset = max(
                float(np.quantile(solar[:, h] - fitted, self.quantile)),
                0.0,
            )
            self.models_.append(model)
            self.offsets_.append(residual_offset)
            self.upper_.append(
                max(float(np.quantile(solar[:, h], 0.997) * 1.15), 0.08)
            )
        return self

    def predict(
        self,
        dates: pd.DatetimeIndex,
        geometry: np.ndarray,
    ) -> np.ndarray:
        x = self.design(dates, geometry)
        prediction = np.column_stack(
            [
                model.predict(x) + offset
                for model, offset in zip(self.models_, self.offsets_)
            ]
        )
        return np.clip(
            prediction,
            0.03,
            np.asarray(self.upper_)[None, :],
        ).astype(np.float32)


In [ ]:
# Feature Engineering and Leakage-Safe Historical Descriptors
SOLAR_LAGS = (1, 2, 3, 7)
STATE_WINDOWS = (3, 7)
WEATHER_SUMMARY_WINDOWS = (3, 7)
YEAR_LAGS = (364, 365, 366)
@dataclass
class SiteBundle:
    site: str
    stage: str
    potential_fit_end: pd.Timestamp
    target_dates: pd.DatetimeIndex
    origin_indices: np.ndarray
    sequence: np.ndarray
    sequence_channels: Tuple[str, ...]
    astronomy: np.ndarray
    astronomy_names: Tuple[str, ...]
    archived_weather: np.ndarray
    archived_weather_names: Tuple[str, ...]
    archived_weather_mask: np.ndarray
    target_weather: np.ndarray
    target_weather_names: Tuple[str, ...]
    context: np.ndarray
    context_names: Tuple[str, ...]
    tabular: np.ndarray
    tabular_families: np.ndarray
    target: np.ndarray
    target_state: np.ndarray
    target_shape: np.ndarray
    target_energy: np.ndarray
    target_dii: np.ndarray
    target_variability: np.ndarray
    potential: np.ndarray
    geometry: np.ndarray
    potential_model: Any
    train: np.ndarray
    val_select: np.ndarray
    val_cal: np.ndarray
    val_all: np.ndarray
    train_val: np.ndarray
    test: np.ndarray
    raw: DailySiteData
def solar_variability_index(solar_curve, potential_curve):
    """Public-release implementation note."""
    solar_curve = np.asarray(solar_curve, float)
    potential_curve = np.asarray(potential_curve, float)
    state = np.clip(solar_curve / np.maximum(potential_curve, 0.05), 0.0, 2.5)
    measured_path = np.sum(np.sqrt(np.diff(solar_curve, axis=1) ** 2 + 1.0), axis=1)
    clear_path = np.sum(np.sqrt(np.diff(potential_curve, axis=1) ** 2 + 1.0), axis=1)
    vi = measured_path / np.maximum(clear_path, 1e-6)
    ramp = np.mean(np.abs(np.diff(state, axis=1)), axis=1)
    curvature = np.mean(np.abs(np.diff(state, n=2, axis=1)), axis=1)
    return (np.log1p(np.maximum(vi - 1.0, 0.0)) + 0.65 * ramp + 0.35 * curvature).astype(np.float32)
def _extend_feature(output, families, values, family):
    flat = np.asarray(values, dtype=float).ravel()
    output.extend(flat.tolist())
    families.extend([family] * len(flat))
def target_astronomy(date, geometry_day, potential_day):
    hour_phase = 2 * np.pi * (
        TARGET_HOURS - TARGET_HOURS.min()
    ) / N_HORIZONS
    day_terms = day_fourier(pd.DatetimeIndex([date]), 3).ravel()
    rows = []
    for h in range(N_HORIZONS):
        rows.append(
            np.concatenate(
                [
                    geometry_day[h],
                    [
                        potential_day[h],
                        np.sin(hour_phase[h]),
                        np.cos(hour_phase[h]),
                    ],
                    day_terms,
                ]
            )
        )
    names = (
        "sin_elevation", "cos_zenith", "extra_horizontal", "airmass",
        "stage_fitted_potential", "hour_sin", "hour_cos",
        "doy_sin_1", "doy_cos_1", "doy_sin_2", "doy_cos_2",
        "doy_sin_3", "doy_cos_3",
    )
    return np.asarray(rows, np.float32), names
def make_issue_sample(i, data, potential, geometry):
    solar = data.solar
    weather = data.weather
    state = np.clip(solar / np.maximum(potential, 0.05), 0.0, 2.5)
    energy = np.maximum(solar.sum(axis=1), 1e-4)
    shape = solar / energy[:, None]
    state_ramp = np.diff(state, axis=1, prepend=state[:, :1])

    feature, families = [], []
    for lag in SOLAR_LAGS:
        family = "lagged_clearness_index_1d" if lag == 1 else f"solar_state_lag_{lag}d"
        _extend_feature(feature, families, state[i - lag], family)
    for lag in (1, 7):
        _extend_feature(feature, families, shape[i - lag], f"solar_shape_lag_{lag}d")
        _extend_feature(feature, families, [energy[i - lag]], f"solar_energy_lag_{lag}d")
    for window in STATE_WINDOWS:
        state_block = state[i - window:i]
        _extend_feature(feature, families, state_block.mean(axis=0), f"solar_state_{window}d_mean")
        _extend_feature(feature, families, state_block.std(axis=0), f"solar_state_{window}d_std")

    
    for lag in (14, 28):
        _extend_feature(feature, families, state[i - lag], f"optional_long_state_lag_{lag}d")
    long_block = state[i - 14:i]
    _extend_feature(feature, families, long_block.mean(axis=0), "optional_long_state_14d_mean")
    _extend_feature(feature, families, long_block.std(axis=0), "optional_long_state_14d_std")

    _extend_feature(feature, families, weather[i - 1], "weather_last_completed_day")
    
    _extend_feature(
        feature, families, np.diff(weather[i - 1, :, 0]), "TempChange_1h"
    )
    _extend_feature(
        feature, families, np.diff(weather[i - 1, :, 1]), "RHChange_1h"
    )
    _extend_feature(
        feature, families, weather[i - 1, :, 0] - weather[i - 2, :, 0],
        "TempChange_1d",
    )
    _extend_feature(
        feature, families, weather[i - 1, :, 1] - weather[i - 2, :, 1],
        "RHChange_1d",
    )
    for window in WEATHER_SUMMARY_WINDOWS:
        block = weather[i - window:i]
        summary = np.concatenate(
            [
                block.mean(axis=(0, 1)),
                np.median(block, axis=(0, 1)),
                block.std(axis=(0, 1)),
            ]
        )
        _extend_feature(
            feature, families, summary,
            f"weather_{window}d_robust_summary",
        )

    astronomy, astronomy_names = target_astronomy(
        data.dates[i], geometry[i], potential[i]
    )
    _extend_feature(
        feature, families,
        np.concatenate([geometry[i, :, :3].ravel(), potential[i]]),
        "target_astronomy",
    )
    _extend_feature(
        feature, families,
        day_fourier(pd.DatetimeIndex([data.dates[i]]), 3).ravel(),
        "target_calendar",
    )

    if data.forecast_weather is None:
        archived = np.zeros((N_HORIZONS, 1), dtype=np.float32)
        archived_names = ("no_archived_forecast",)
        archive_mask = 0.0
    else:
        archived = data.forecast_weather[i].astype(np.float32)
        archived_names = tuple(data.forecast_names)
        archive_mask = 1.0
        _extend_feature(
            feature, families, archived,
            "archived_issue_time_weather_forecast",
        )

    # Sequence branches remain compact. Shape and energy are explicit so a
    # neural network does not have to rediscover their scale separation.
    energy_scale = np.maximum(
        np.median(energy[max(0, i - 28):i]), 0.1
    )
    energy_channel = np.repeat(
        (energy[i - MAX_LOOKBACK_DAYS:i] / energy_scale)[:, None, None],
        N_HORIZONS,
        axis=1,
    )
    sequence = np.concatenate(
        [
            state[i - MAX_LOOKBACK_DAYS:i, :, None],
            shape[i - MAX_LOOKBACK_DAYS:i, :, None],
            energy_channel,
            state_ramp[i - MAX_LOOKBACK_DAYS:i, :, None],
            weather[i - MAX_LOOKBACK_DAYS:i],
        ],
        axis=-1,
    ).astype(np.float32)
    context = np.concatenate(
        [astronomy.ravel(), archived.ravel(), [archive_mask]]
    ).astype(np.float32)
    context_names = (
        tuple(
            f"astro_h{h}_{name}"
            for h in range(1, N_HORIZONS + 1)
            for name in astronomy_names
        )
        + tuple(
            f"archived_h{h}_{name}"
            for h in range(1, N_HORIZONS + 1)
            for name in archived_names
        )
        + ("archived_forecast_available",)
    )
    return {
        "tabular": np.asarray(feature, np.float32),
        "families": np.asarray(families),
        "sequence": sequence,
        "astronomy": astronomy,
        "astronomy_names": astronomy_names,
        "archived": archived,
        "archived_names": archived_names,
        "archive_mask": archive_mask,
        "target_weather": weather[i].astype(np.float32),
        "context": context,
        "context_names": context_names,
    }


In [ ]:
# Site Bundles and Chronological Splits
def build_site_bundle(data, potential_fit_end, stage):
    geometry = solar_geometry(
        data.dates, data.latitude, data.longitude
    )
    fit_days = data.dates <= potential_fit_end
    potential_model = AstronomyPotential().fit(
        data.dates[fit_days],
        data.solar[fit_days],
        geometry[fit_days],
    )
    potential = potential_model.predict(data.dates, geometry)
    state = np.clip(
        data.solar / np.maximum(potential, 0.05), 0.0, 2.5
    )
    energy = np.maximum(data.solar.sum(axis=1), 1e-4)
    shape = data.solar / energy[:, None]

    samples, origins = [], []
    for i in range(MAX_LOOKBACK_DAYS, len(data.dates)):
        samples.append(
            make_issue_sample(i, data, potential, geometry)
        )
        origins.append(i)
    origins = np.asarray(origins)
    target_dates = pd.DatetimeIndex(data.dates[origins])
    train = target_dates <= TRAIN_END
    val_select = (
        (target_dates >= VAL_START)
        & (target_dates <= VAL_SELECT_END)
    )
    val_cal = (
        (target_dates >= VAL_CAL_START)
        & (target_dates <= VAL_END)
    )
    val_all = val_select | val_cal
    test = (
        (target_dates >= TEST_START)
        & (target_dates <= TEST_END)
    )
    first = samples[0]
    return SiteBundle(
        site=data.site,
        stage=stage,
        potential_fit_end=pd.Timestamp(potential_fit_end),
        target_dates=target_dates,
        origin_indices=origins,
        sequence=np.asarray(
            [sample["sequence"] for sample in samples], np.float32
        ),
        sequence_channels=(
            "Clear_sky_state", "Normalized_curve_shape",
            "Relative_daily_energy", "State_ramp",
            *data.weather_names,
        ),
        astronomy=np.asarray(
            [sample["astronomy"] for sample in samples], np.float32
        ),
        astronomy_names=tuple(first["astronomy_names"]),
        archived_weather=np.asarray(
            [sample["archived"] for sample in samples], np.float32
        ),
        archived_weather_names=tuple(first["archived_names"]),
        archived_weather_mask=np.asarray(
            [sample["archive_mask"] for sample in samples], np.float32
        ),
        target_weather=np.asarray(
            [sample["target_weather"] for sample in samples], np.float32
        ),
        target_weather_names=tuple(data.weather_names),
        context=np.asarray(
            [sample["context"] for sample in samples], np.float32
        ),
        context_names=tuple(first["context_names"]),
        tabular=np.asarray(
            [sample["tabular"] for sample in samples], np.float32
        ),
        tabular_families=first["families"],
        target=data.solar[origins].astype(np.float32),
        target_state=state[origins].astype(np.float32),
        target_shape=shape[origins].astype(np.float32),
        target_energy=energy[origins].astype(np.float32),
        target_dii=(
            data.solar[origins].sum(axis=1)
            / np.maximum(potential[origins].sum(axis=1), 0.10)
        ).astype(np.float32),
        target_variability=solar_variability_index(
            data.solar[origins], potential[origins]
        ),
        potential=potential[origins].astype(np.float32),
        geometry=geometry[origins].astype(np.float32),
        potential_model=potential_model,
        train=train,
        val_select=val_select,
        val_cal=val_cal,
        val_all=val_all,
        train_val=train | val_all,
        test=test,
        raw=data,
    )


SELECTION_BUNDLES = {
    site: build_site_bundle(
        SITE_DATA[site], TRAIN_END, "selection_train_only"
    )
    for site in SITES
}
BRIDGE_BUNDLES = {
    site: build_site_bundle(
        SITE_DATA[site], VAL_SELECT_END, "bridge_refit_through_2019_09"
    )
    for site in SITES
}
REFIT_BUNDLES = {
    site: build_site_bundle(
        SITE_DATA[site], VAL_END, "final_train_plus_validation"
    )
    for site in SITES
}
# Backward-compatible alias: every hyperparameter decision uses this train-only view.
SITE_BUNDLES = SELECTION_BUNDLES

FEATURE_AUDIT = pd.DataFrame(
    [
        {
            "Site": site,
            "Stage": bundle.stage,
            "Potential_fit_end": str(bundle.potential_fit_end.date()),
            "Train_days": int(bundle.train.sum()),
            "Validation_days": int(bundle.val_all.sum()),
            "Test_days": int(bundle.test.sum()),
            "Tabular_features": int(bundle.tabular.shape[1]),
            "Predictors_per_train_day": float(
                bundle.tabular.shape[1]
                / max(bundle.train.sum(), 1)
            ),
            "Sequence_shape": str(tuple(bundle.sequence.shape[1:])),
            "Astronomy_shape": str(tuple(bundle.astronomy.shape[1:])),
            "Archived_forecast_available": bool(
                bundle.archived_weather_mask.max() > 0
            ),
        }
        for site in SITES
        for bundle in (
            SELECTION_BUNDLES[site],
            BRIDGE_BUNDLES[site],
            REFIT_BUNDLES[site],
        )
    ]
)
display(FEATURE_AUDIT)
FEATURE_AUDIT.to_csv(
    OUTPUT_DIR / "MUFASA_feature_dimension_audit.csv",
    index=False,
)
if (FEATURE_AUDIT["Predictors_per_train_day"] >= 0.45).any():
    raise AssertionError(
        "The compact engineered table unexpectedly exceeds 0.45 predictors/day."
    )
for site in SITES:
    left, bridge, right = SELECTION_BUNDLES[site], BRIDGE_BUNDLES[site], REFIT_BUNDLES[site]
    if not (np.array_equal(left.target_dates, bridge.target_dates) and np.array_equal(left.target_dates, right.target_dates)):
        raise AssertionError(f"{site}: stage views lost date alignment.")
    if np.allclose(left.potential, right.potential):
        print(
            f"{site}: warning—the 2019 refit did not materially change "
            "the potential envelope."
        )


In [ ]:
# Weather-Information Scenarios
def with_weather_scenario(bundle, scenario):
    """Public-release implementation note."""
    if scenario == "strict_history":
        context = bundle.context.copy()
        astronomy_width = N_HORIZONS * len(bundle.astronomy_names)
        context[:, astronomy_width:] = 0.0
        tabular = bundle.tabular.copy()
        archived_columns = np.where(
            np.asarray(bundle.tabular_families).astype(str) == "archived_issue_time_weather_forecast"
        )[0]
        if len(archived_columns):
            tabular[:, archived_columns] = 0.0
        return replace(
            bundle,
            archived_weather=np.zeros((len(bundle.target_dates), N_HORIZONS, 1), np.float32),
            archived_weather_names=("no_forward_weather",),
            archived_weather_mask=np.zeros(len(bundle.target_dates), np.float32),
            context=context.astype(np.float32),
            tabular=tabular.astype(np.float32),
        )
    if scenario == "archived_forecast":
        if not np.any(bundle.archived_weather_mask > 0):
            return None
        return bundle
    if scenario != "oracle_weather":
        raise KeyError(scenario)

    
    future = np.asarray(bundle.target_weather, np.float32)
    flat = future.reshape(len(future), -1)
    family = np.asarray(["future_weather_covariates"] * flat.shape[1])
    context = np.concatenate([bundle.context, flat, np.ones((len(future), 1), np.float32)], axis=1)
    context_names = tuple(bundle.context_names) + tuple(
        f"oracle_h{h}_{name}"
        for h in range(1, N_HORIZONS + 1)
        for name in bundle.target_weather_names
    ) + ("oracle_weather_flag",)
    return replace(
        bundle,
        archived_weather=future,
        archived_weather_names=tuple(bundle.target_weather_names),
        archived_weather_mask=np.ones(len(future), np.float32),
        context=context.astype(np.float32),
        context_names=context_names,
        tabular=np.concatenate([bundle.tabular, flat], axis=1).astype(np.float32),
        tabular_families=np.concatenate([bundle.tabular_families, family]),
    )


def scenario_bundle_maps(scenario):
    selection, refit = {}, {}
    for site in SITES:
        s = with_weather_scenario(SELECTION_BUNDLES[site], scenario)
        r = with_weather_scenario(REFIT_BUNDLES[site], scenario)
        if s is None or r is None:
            return None, None
        selection[site], refit[site] = s, r
    return selection, refit


STRICT_SELECTION_BUNDLES, STRICT_REFIT_BUNDLES = scenario_bundle_maps("strict_history")
ORACLE_SELECTION_BUNDLES, ORACLE_REFIT_BUNDLES = scenario_bundle_maps("oracle_weather")
ARCHIVED_SELECTION_BUNDLES, ARCHIVED_REFIT_BUNDLES = scenario_bundle_maps("archived_forecast")
ARCHIVED_AVAILABLE = ARCHIVED_SELECTION_BUNDLES is not None
print(f"Archived forecast available={ARCHIVED_AVAILABLE}; oracle is labelled upper-bound only.")


## 3. MUFASA nonlinear expert and training objective

Define leakage-safe scaling, the nonlinear temporal expert, architecture candidates, scheduled TF, and shared fit/predict routines.

**Run note.** Execute the cells in this section in order. Objects created here are consumed by later sections; the notebook intentionally avoids hidden state restoration from unpublished artifacts.


In [ ]:
# Scaling, Target Transforms, and Helper Layers — part 1/2
def _center_scale(values, axes, robust, keepdims=True):
    values = np.asarray(values, dtype=np.float32)
    if robust:
        center = np.median(values, axis=axes, keepdims=keepdims)
        q25 = np.quantile(values, 0.25, axis=axes, keepdims=keepdims)
        q75 = np.quantile(values, 0.75, axis=axes, keepdims=keepdims)
        scale = (q75 - q25) / 1.349
    else:
        center = np.mean(values, axis=axes, keepdims=keepdims)
        scale = np.std(values, axis=axes, keepdims=keepdims)
    scale = np.where(np.isfinite(scale) & (scale > 1e-4), scale, 1.0)
    return center.astype(np.float32), scale.astype(np.float32)
if TORCH_AVAILABLE:
    try:
        from torch.nn.utils.parametrizations import weight_norm
    except Exception:
        from torch.nn.utils import weight_norm

    class SolarFluxResidualBlock(nn.Module):
        """Two causal, weight-normalized dilated convolutions plus a residual path."""
        def __init__(self, width, dilation, dropout, kernel_size=3):
            super().__init__()
            self.left = (kernel_size - 1) * dilation
            self.conv1 = weight_norm(nn.Conv1d(width, width, kernel_size, dilation=dilation))
            self.conv2 = weight_norm(nn.Conv1d(width, width, kernel_size, dilation=dilation))
            self.norm1 = nn.GroupNorm(1, width)
            self.norm2 = nn.GroupNorm(1, width)
            self.dropout = nn.Dropout(dropout)

        def _causal(self, convolution, x):
            return convolution(F.pad(x, (self.left, 0)))

        def forward(self, x):
            residual = x
            y = self.dropout(F.leaky_relu(self.norm1(self._causal(self.conv1, x)), 0.05))
            y = self.dropout(F.leaky_relu(self.norm2(self._causal(self.conv2, y)), 0.05))
            return residual + y


    class HourAlignedTCN(nn.Module):
        """Causal TCN followed by self-attention over completed days, separately by hour."""
        def __init__(self, channels, width, dropout, use_attention=True):
            super().__init__()
            self.projection = nn.Linear(channels, width)
            self.blocks = nn.Sequential(*[
                SolarFluxResidualBlock(width, dilation, dropout) for dilation in (1, 2, 4, 8)
            ])
            self.use_attention = bool(use_attention)
            self.norm = nn.LayerNorm(width)
            self.attention = nn.MultiheadAttention(
                width, 4, dropout=dropout, batch_first=True
            )
            self.mix = nn.Parameter(torch.tensor(0.0))

        def forward(self, sequence):
            batch, days, hours, channels = sequence.shape
            values = sequence.permute(0, 2, 1, 3).reshape(batch * hours, days, channels)
            encoded = self.blocks(self.projection(values).transpose(1, 2)).transpose(1, 2)
            if self.use_attention:
                attended, _ = self.attention(
                    self.norm(encoded), self.norm(encoded), self.norm(encoded), need_weights=False
                )
                encoded = encoded + torch.sigmoid(self.mix) * attended
            return encoded[:, -1].reshape(batch, hours, -1)
from dataclasses import dataclass
@dataclass(frozen=True)
class MUFASAConfig:
    d_model: int = 64
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.12
    learning_rate: float = 7e-4
    weight_decay: float = 2e-4
    batch_size: int = 256
    lookback: int = 28
    scaling_mode: str = "standard_hourwise"
    target_transform: str = "log1p_standard"
    use_weather: bool = True
    use_astronomy: bool = True
    use_engineered: bool = True
    engineered_feature_mode: str = "no_annual"
    use_proxy_weather: bool = False
    use_regime_head: bool = True
    use_multiscale: bool = True
    use_hour_attention: bool = True
    use_cross_alignment: bool = True
    use_autoregressive: bool = True
    use_structured_decoder: bool = True
    use_solar_reference: bool = True
    reference_alpha: float = 1.0
    teacher_forcing_start: float = 0.70
    teacher_forcing_end: float = 0.0
    teacher_forcing_decay: float = 0.0  # retained for backward-compatible tables
    tf_zero_epoch: int = 8
    free_run_min_epochs: int = 10
    teacher_aux_weight: float = 0.25
    free_run_weight: float = 1.0
    consistency_weight: float = 0.04
    ramp_weight: float = 0.025
    energy_weight: float = 0.012
    shape_weight: float = 0.020
    peak_weight: float = 0.018
    weather_aux_weight: float = 0.08
    regime_weight: float = 0.06
    overcast_overprediction_weight: float = 0.0
    uncertainty_weight: float = 0.010
    gate_reference_weight: float = 0.004
    conditional_gate_scale: float = 0.20
    use_revin: bool = True
    use_nlinear: bool = True
    use_channel_independent: bool = True
    max_ar_weight: float = 0.22
    use_bigru: bool = True
    bigru_gate_init: float = 0.10
    use_tail_weighting: bool = True
    tail_weight_strength: float = 0.50
    tail_weight_cap: float = 2.0
REFERENCE_CONFIG = MUFASAConfig(batch_size=GPU_BATCH_SIZE)
@dataclass
class SolarReferenceModel:
    coefficients: np.ndarray
    intercepts: np.ndarray
    monthly_state: np.ndarray
    alpha: float
    candidate_names: Tuple[str, ...]
    enabled: bool = True
@dataclass
class TrainedMUFASA:
    kind: str
    config: MUFASAConfig
    model: Any
    scalers: Dict[str, Any]
    reference_model: SolarReferenceModel
    best_epoch: int
    validation_rmse: float
    seed: int
    teacher_forcing_history: List[float]
_SOLAR_REFERENCE_CACHE: Dict[Tuple[str, str, str], np.ndarray] = {}
def _base_solar_reference_candidates(bundle):
    key = (bundle.site, bundle.stage, str(bundle.potential_fit_end.date()), "compact")
    if key in _SOLAR_REFERENCE_CACHE:
        return _SOLAR_REFERENCE_CACHE[key]
    raw = bundle.raw
    full_geometry = solar_geometry(raw.dates, raw.latitude, raw.longitude)
    full_potential = bundle.potential_model.predict(raw.dates, full_geometry)
    full_state = np.clip(raw.solar / np.maximum(full_potential, 0.05), 0.0, 2.5)
    rows = []
    for i in bundle.origin_indices:
        recent3 = np.median(full_state[max(0, i - 3):i], axis=0)
        recent14 = np.median(full_state[max(0, i - 14):i], axis=0)
        ewma7 = np.average(
            full_state[max(0, i - 7):i], axis=0,
            weights=np.arange(1, min(7, i) + 1, dtype=float),
        )
        rows.append(np.stack([
            full_state[i - 1], recent3, full_state[i - 7], recent14, ewma7,
        ], axis=-1))
    values = np.asarray(rows, np.float32)
    _SOLAR_REFERENCE_CACHE[key] = values
    return values
def fit_solar_reference(bundle, train_idx, alpha=1.0, enabled=True):
    train_idx = np.asarray(train_idx, dtype=int)
    base = _base_solar_reference_candidates(bundle)
    monthly = np.zeros((12, N_HORIZONS), np.float32)
    global_state = np.median(bundle.target_state[train_idx], axis=0)
    for month in range(1, 13):
        mask = bundle.target_dates[train_idx].month == month
        monthly[month - 1] = (
            np.median(bundle.target_state[train_idx[mask]], axis=0)
            if np.any(mask) else global_state
        )
    climatology = monthly[bundle.target_dates.month - 1]
    candidates = np.concatenate([base, climatology[..., None]], axis=-1)
    names = (
        "lag_1d", "median_3d", "lag_7d", "median_14d",
        "ewma_7d", "train_month_hour_climatology",
    )
    coefficients = np.zeros((N_HORIZONS, candidates.shape[-1]), np.float32)
    intercepts = np.zeros(N_HORIZONS, np.float32)
    if enabled:
        for horizon in range(N_HORIZONS):
            regressor = Ridge(alpha=float(alpha), positive=True, fit_intercept=True)
            regressor.fit(candidates[train_idx, horizon], bundle.target_state[train_idx, horizon])
            coefficients[horizon] = regressor.coef_
            intercepts[horizon] = regressor.intercept_
    return SolarReferenceModel(coefficients, intercepts, monthly, float(alpha), names, bool(enabled))
def predict_solar_reference(bundle, reference_model):
    if not reference_model.enabled:
        return np.zeros_like(bundle.target_state, dtype=np.float32)
    base = _base_solar_reference_candidates(bundle)
    climatology = reference_model.monthly_state[bundle.target_dates.month - 1]
    candidates = np.concatenate([base, climatology[..., None]], axis=-1)
    state = np.einsum("nhk,hk->nh", candidates, reference_model.coefficients) + reference_model.intercepts
    return np.clip(state, 0.0, 2.5).astype(np.float32)
def predict_solar_reference_candidates(reference_model, base_candidates, months):
    if not reference_model.enabled:
        return np.zeros(base_candidates.shape[:2], np.float32)
    climatology = reference_model.monthly_state[np.asarray(months, int) - 1]
    candidates = np.concatenate([np.asarray(base_candidates, np.float32), climatology[..., None]], axis=-1)
    state = np.einsum("nhk,hk->nh", candidates, reference_model.coefficients) + reference_model.intercepts
    return np.clip(state, 0.0, 2.5).astype(np.float32)


In [ ]:
# Scaling, Target Transforms, and Helper Layers — part 2/2
def _engineered_feature_mask(bundle, feature_mode):
    families = np.asarray(bundle.tabular_families).astype(str)
    always = (
        "lagged_clearness_index_1d", "solar_state_lag_2d", "solar_state_lag_3d",
        "solar_state_lag_7d", "solar_shape_lag_1d", "solar_shape_lag_7d",
        "solar_energy_lag_1d", "solar_energy_lag_7d", "solar_state_3d_",
        "solar_state_7d_", "weather_last_completed_day", "TempChange_1h",
        "RHChange_1h", "TempChange_1d", "RHChange_1d", "weather_3d_",
        "weather_7d_", "target_astronomy", "target_calendar",
        "archived_issue_time_weather_forecast", "future_weather_covariates",
    )
    minimal = (
        "lagged_clearness_index_1d", "solar_state_lag_2d", "solar_state_lag_3d",
        "solar_state_lag_7d", "solar_shape_lag_1d", "solar_shape_lag_7d",
        "solar_energy_lag_1d", "solar_energy_lag_7d", "solar_state_3d_",
        "solar_state_7d_", "weather_last_completed_day", "target_astronomy",
        "target_calendar",
    )
    if feature_mode in {"literature_compact", "compact_no_annual", "no_annual"}:
        keep = always
    elif feature_mode == "minimal_compact":
        keep = minimal
    elif feature_mode in {"extended_compact", "full"}:
        keep = always + ("optional_long_",)
    else:
        raise ValueError(f"Unknown engineered_feature_mode={feature_mode!r}")
    mask = np.array([any(term in family for term in keep) for family in families])
    if not np.any(mask):
        raise RuntimeError("The engineered feature mask selected no columns.")
    return np.where(mask)[0]
def _fit_regime_thresholds(bundle, indices):
    values = np.asarray(bundle.target_variability[indices], float)
    low, high = np.quantile(values, [0.35, 0.70])
    if not np.isfinite([low, high]).all() or high <= low + 1e-5:
        low, high = float(np.quantile(values, 0.33)), float(np.quantile(values, 0.67))
    return np.asarray([low, high], np.float32)
def _regime_labels(bundle, indices, thresholds):
    return np.digitize(
        np.asarray(bundle.target_variability[indices], float), np.asarray(thresholds, float), right=False
    ).astype(np.int64)
def _rare_ramp_score(bundle, indices):
    """Public-release implementation note."""
    indices = np.asarray(indices, dtype=int)
    state = np.asarray(bundle.target_state[indices], float)
    ramp = np.mean(np.abs(np.diff(state, axis=1)), axis=1)
    variability = np.std(state, axis=1)
    peak = np.max(state, axis=1)
    return np.asarray(ramp + 0.35 * variability + 0.15 * peak, np.float32)
def _fit_lds_profile(scores, bins=41, sigma=2.0):
    """Public-release implementation note."""
    scores = np.asarray(scores, float)
    lo, hi = float(np.min(scores)), float(np.max(scores))
    if not np.isfinite(lo + hi) or hi <= lo + 1e-8:
        return np.array([lo - 1e-4, hi + 1e-4], np.float32), np.ones(1, np.float32)
    edges = np.linspace(lo, hi + 1e-8, int(bins) + 1)
    counts, _ = np.histogram(scores, bins=edges)
    radius = max(2, int(np.ceil(3.0 * sigma)))
    grid = np.arange(-radius, radius + 1, dtype=float)
    kernel = np.exp(-0.5 * (grid / max(float(sigma), 1e-4)) ** 2)
    kernel /= kernel.sum()
    density = np.convolve(counts.astype(float), kernel, mode="same")
    density = np.maximum(density, max(1e-3, density.max() * 1e-3))
    return edges.astype(np.float32), density.astype(np.float32)
def _lds_sample_weights(scores, edges, density, strength=0.5, cap=2.0):
    scores = np.asarray(scores, float)
    density = np.asarray(density, float)
    if len(density) <= 1:
        return np.ones(len(scores), np.float32)
    bin_index = np.clip(np.digitize(scores, np.asarray(edges)[1:-1]), 0, len(density) - 1)
    inverse = np.sqrt(np.max(density) / np.maximum(density[bin_index], 1e-8))
    reference = max(float(np.median(inverse)), 1e-8)
    weight = 1.0 + float(strength) * np.maximum(inverse / reference - 1.0, 0.0)
    return np.clip(weight, 1.0, float(cap)).astype(np.float32)
def fit_scalers(
    bundle, indices, scaling_mode="standard_hourwise",
    target_transform="log1p_standard", solar_ref_state=None, reference_model=None,
    engineered_feature_mode="no_annual", use_tail_weighting=True,
    tail_weight_strength=0.50, tail_weight_cap=2.0,
):
    indices = np.asarray(indices, dtype=int)
    if len(indices) == 0:
        raise ValueError("A scaler cannot be fitted on an empty index set.")
    if np.any(bundle.test[indices]):
        raise AssertionError("Test rows entered a scaler fit.")
    robust = scaling_mode.startswith("robust")
    hourwise = scaling_mode.endswith("hourwise")
    global_mode = scaling_mode.endswith("global")
    if scaling_mode not in {
        "robust_hourwise", "standard_hourwise", "robust_channelwise",
        "standard_channelwise", "standard_global", "robust_global",
    }:
        raise ValueError(f"Unknown scaling_mode={scaling_mode!r}")
    seq_axes = (0, 1, 2, 3) if global_mode else ((0, 1) if hourwise else (0, 1, 2))
    horizon_axes = (0, 1, 2) if global_mode else (0 if hourwise else (0, 1))
    sequence_mean, sequence_std = _center_scale(bundle.sequence[indices], seq_axes, robust)
    astronomy_mean, astronomy_std = _center_scale(bundle.astronomy[indices], horizon_axes, robust)
    archive_mean, archive_std = _center_scale(bundle.archived_weather[indices], horizon_axes, robust)
    context_mean, context_std = _center_scale(bundle.context[indices], 0, robust)
    engineered_columns = _engineered_feature_mask(bundle, engineered_feature_mode)
    engineered_mean, engineered_std = _center_scale(
        bundle.tabular[np.ix_(indices, engineered_columns)], 0, robust
    )
    weather_mean, weather_std = _center_scale(bundle.target_weather[indices], (0, 1), robust)
    regime_thresholds = _fit_regime_thresholds(bundle, indices)
    regime_train = _regime_labels(bundle, indices, regime_thresholds)
    regime_counts = np.bincount(regime_train, minlength=len(REGIME_NAMES)).astype(float)
    regime_weights = regime_counts.sum() / np.maximum(len(REGIME_NAMES) * regime_counts, 1.0)
    regime_weights = np.clip(regime_weights / regime_weights.mean(), 0.5, 2.0).astype(np.float32)
    tail_scores = _rare_ramp_score(bundle, indices)
    tail_edges, tail_density = _fit_lds_profile(tail_scores)
    tail_train_weights = (
        _lds_sample_weights(
            tail_scores, tail_edges, tail_density,
            strength=tail_weight_strength, cap=tail_weight_cap,
        )
        if use_tail_weighting else np.ones(len(indices), np.float32)
    )
    if solar_ref_state is None:
        solar_ref_state = np.zeros_like(bundle.target_state, dtype=np.float32)
    if target_transform == "log1p_standard":
        residual_target = (
            np.log1p(np.clip(bundle.target_state, 0.0, None))
            - np.log1p(np.clip(solar_ref_state, 0.0, None))
        )
        target_space = "log1p_state_residual"
    elif target_transform == "identity_standard":
        residual_target = bundle.target_state - solar_ref_state
        target_space = "state_residual"
    elif target_transform == "asinh_standard":
        residual_target = np.arcsinh(bundle.target_state) - np.arcsinh(solar_ref_state)
        target_space = "asinh_state_residual"
    else:
        raise ValueError(f"Unknown target_transform={target_transform!r}")
    target_center, target_scale = _center_scale(residual_target[indices], 0, False)
    fit_dates = bundle.target_dates[indices]
    reference_arrays = [] if reference_model is None else [reference_model.coefficients, reference_model.intercepts, reference_model.monthly_state]
    arrays_for_digest = [
        sequence_mean, sequence_std, astronomy_mean, astronomy_std, archive_mean, archive_std,
        context_mean, context_std, engineered_mean, engineered_std, weather_mean, weather_std,
        regime_thresholds, regime_weights, target_center, target_scale,
        tail_edges, tail_density, np.asarray([tail_weight_strength, tail_weight_cap]),
        *reference_arrays,
    ]
    digest = hashlib.sha256(b"".join(np.asarray(x, np.float32).tobytes() for x in arrays_for_digest)).hexdigest()
    return {
        "sequence_mean": sequence_mean, "sequence_std": sequence_std,
        "seq_mean": sequence_mean, "seq_std": sequence_std,
        "astronomy_mean": astronomy_mean, "astronomy_std": astronomy_std,
        "archive_mean": archive_mean, "archive_std": archive_std,
        "context_mean": context_mean, "context_std": context_std,
        "engineered_mean": engineered_mean, "engineered_std": engineered_std,
        "engineered_columns": engineered_columns,
        "engineered_feature_mode": engineered_feature_mode,
        "weather_mean": weather_mean, "weather_std": weather_std,
        "regime_thresholds": regime_thresholds, "regime_weights": regime_weights,
        "tail_edges": tail_edges, "tail_density": tail_density,
        "use_tail_weighting": bool(use_tail_weighting),
        "tail_weight_strength": float(tail_weight_strength),
        "tail_weight_cap": float(tail_weight_cap),
        "tail_weight_min": float(tail_train_weights.min()),
        "tail_weight_mean": float(tail_train_weights.mean()),
        "tail_weight_max": float(tail_train_weights.max()),
        "target_center": target_center, "target_scale": target_scale,
        "energy_center": np.asarray(0.0, np.float32), "energy_scale": np.asarray(1.0, np.float32),
        "target_transform": target_transform, "target_space": target_space,
        "scaling_mode": scaling_mode, "reference_model": reference_model,
        "fit_start": str(fit_dates.min().date()), "fit_end": str(fit_dates.max().date()),
        "fit_rows": int(len(indices)), "test_rows_in_fit": int(np.sum(bundle.test[indices])),
        "digest": digest,
    }


In [ ]:
# Nonlinear Temporal Expert — part 1/2
def _amp_context(enabled_override=None):
    amp_enabled = USE_AMP if enabled_override is None else bool(
        enabled_override and DEVICE.type == "cuda"
    )
    return torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16 if DEVICE.type == "cuda" else torch.bfloat16,
        enabled=amp_enabled,
    )
if TORCH_AVAILABLE:
    class CausalPatchBlock(nn.Module):
        """Public-release implementation note."""
        def __init__(self, width, dilation, dropout):
            super().__init__()
            self.left = 2 * dilation
            self.conv = nn.Conv1d(width, width, 3, dilation=dilation, groups=width)
            self.mix = nn.Conv1d(width, width, 1)
            self.norm = nn.GroupNorm(1, width)
            self.drop = nn.Dropout(dropout)

        def forward(self, x):
            update = self.conv(F.pad(x, (self.left, 0)))
            update = self.drop(F.gelu(self.norm(self.mix(update))))
            return x + update


    class ChannelIndependentPatchEncoder(nn.Module):
        """Public-release implementation note."""
        def __init__(self, lookback, width, layers, heads, dropout):
            super().__init__()
            self.lookback = lookback
            self.input_projection = nn.Conv1d(1, width, 1)
            dilations = (1, 2, 4, 8)
            self.tcn = nn.Sequential(*[
                CausalPatchBlock(width, dilations[i % len(dilations)], dropout)
                for i in range(max(1, layers + 1))
            ])
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=width, nhead=heads, dim_feedforward=3 * width,
                dropout=dropout, activation="gelu", batch_first=True,
                norm_first=True,
            )
            self.temporal_attention = nn.TransformerEncoder(
                encoder_layer, num_layers=1, norm=nn.LayerNorm(width)
            )
            self.score = nn.Linear(width, 1)
            self.position = nn.Parameter(torch.zeros(1, lookback, width))
            nn.init.trunc_normal_(self.position, std=0.02)

        def forward(self, values):
            
            b, days, hours, channels = values.shape
            path = values.permute(0, 2, 3, 1).reshape(b * hours * channels, 1, days)
            token = self.tcn(self.input_projection(path)).transpose(1, 2)
            token = self.temporal_attention(token + self.position[:, -days:])
            attention = torch.softmax(self.score(token).squeeze(-1), dim=-1)
            pooled = torch.sum(token * attention.unsqueeze(-1), dim=1)
            return (
                pooled.reshape(b, hours, channels, -1),
                attention.reshape(b, hours, channels, days),
            )


In [ ]:
# Nonlinear Temporal Expert — part 2/2
if TORCH_AVAILABLE:
    class MUFASANet(nn.Module):
        """Public-release implementation note."""
        def __init__(
            self, sequence_shape, astronomy_dim, archive_dim, engineered_dim,
            weather_dim, scalers, config,
        ):
            super().__init__()
            days, hours, channels = sequence_shape
            self.config = config
            self.lookback = min(config.lookback, days)
            self.hours = hours
            self.channels = channels if config.use_weather else 4
            self.weather_dim = weather_dim
            d = config.d_model
            self.encoder = ChannelIndependentPatchEncoder(
                self.lookback, d, config.n_layers, config.n_heads, config.dropout
            )
            self.channel_logits = nn.Parameter(torch.zeros(N_HORIZONS, self.channels))
            with torch.no_grad():
                self.channel_logits[:, :min(4, self.channels)] += 0.55
            self.history_norm = nn.LayerNorm(d)
            
            self.bigru_input = nn.Linear(N_HORIZONS, d)
            self.history_bigru = nn.GRU(
                input_size=d, hidden_size=d // 2, num_layers=1,
                batch_first=True, bidirectional=True,
            )
            self.bigru_norm = nn.LayerNorm(d)
            initial_gate = float(np.clip(config.bigru_gate_init, 1e-4, 1.0 - 1e-4))
            self.bigru_gate_logit = nn.Parameter(torch.tensor(
                math.log(initial_gate / (1.0 - initial_gate)), dtype=torch.float32
            ))
            self.astronomy_projection = nn.Sequential(
                nn.Linear(astronomy_dim, d), nn.LayerNorm(d), nn.GELU()
            )
            self.engineered_projection = nn.Sequential(
                nn.Linear(engineered_dim, 2 * d), nn.GELU(), nn.Dropout(config.dropout),
                nn.Linear(2 * d, d), nn.LayerNorm(d),
            )
            self.prior_projection = nn.Sequential(
                nn.Linear(2, d), nn.LayerNorm(d), nn.GELU()
            )
            self.archive_projection = nn.Sequential(
                nn.Linear(archive_dim, d), nn.LayerNorm(d), nn.GELU()
            )
            self.horizon_embedding = nn.Parameter(torch.zeros(1, N_HORIZONS, d))
            self.scale_logits = nn.Parameter(torch.zeros(N_HORIZONS, 3))
            self.scale_projection = nn.Sequential(nn.Linear(1, d), nn.GELU())
            self.direct_backbone = nn.Sequential(
                nn.Linear(5 * d, 3 * d), nn.GELU(), nn.Dropout(config.dropout),
                nn.Linear(3 * d, d), nn.LayerNorm(d),
            )
            self.direct_head = nn.Sequential(
                nn.Linear(d, d), nn.GELU(), nn.Dropout(config.dropout), nn.Linear(d, 1)
            )
            self.nlinear_weight = nn.Parameter(torch.zeros(N_HORIZONS, self.lookback))
            self.nlinear_bias = nn.Parameter(torch.zeros(N_HORIZONS))
            self.ar_initial = nn.Sequential(nn.Linear(2 * d, d), nn.Tanh())
            self.previous_projection = nn.Sequential(nn.Linear(1, d // 2), nn.GELU())
            self.ar_cell = nn.GRUCell(d + d // 2, d)
            self.ar_head = nn.Linear(d, 1)
            self.expert_logits = nn.Parameter(torch.tensor(
                [[0.25, 0.50, -0.65, 0.15]], dtype=torch.float32
            ).repeat(N_HORIZONS, 1))
            self.regime_head = nn.Sequential(
                nn.Linear(d, d), nn.GELU(), nn.Dropout(config.dropout),
                nn.Linear(d, len(REGIME_NAMES)),
            )
            self.proxy_weather_head = nn.Sequential(
                nn.Linear(2 * d, d), nn.GELU(), nn.Linear(d, weather_dim)
            )
            self.log_scale_head = nn.Linear(d, 1)
            self.quantile_head = nn.Linear(d, 3)
            self.register_buffer(
                "target_center", torch.as_tensor(scalers["target_center"], dtype=torch.float32)
            )
            self.register_buffer(
                "target_scale", torch.as_tensor(scalers["target_scale"], dtype=torch.float32)
            )
            seq_mean = np.asarray(scalers["sequence_mean"], np.float32)
            seq_std = np.asarray(scalers["sequence_std"], np.float32)
            state_mean = np.broadcast_to(seq_mean, (1, 1, N_HORIZONS, sequence_shape[-1]))[0, 0, :, 0]
            state_std = np.broadcast_to(seq_std, (1, 1, N_HORIZONS, sequence_shape[-1]))[0, 0, :, 0]
            self.register_buffer("state_global_mean", torch.as_tensor(state_mean))
            self.register_buffer("state_global_std", torch.as_tensor(state_std))
            self.register_buffer(
                "regime_weights", torch.as_tensor(scalers["regime_weights"], dtype=torch.float32)
            )
            nn.init.trunc_normal_(self.horizon_embedding, std=0.02)
            self._freeze_inactive_candidate_modules()

        def _freeze_inactive_candidate_modules(self):
            """Public-release implementation note."""
            inactive_prefixes = []
            if not self.config.use_bigru:
                inactive_prefixes += ["bigru_input", "history_bigru", "bigru_norm", "bigru_gate_logit"]
            if not self.config.use_nlinear:
                inactive_prefixes += ["nlinear_weight", "nlinear_bias"]
            if not self.config.use_autoregressive:
                inactive_prefixes += ["ar_initial", "previous_projection", "ar_cell", "ar_head"]
            if not self.config.use_astronomy:
                inactive_prefixes += ["astronomy_projection"]
            if not self.config.use_engineered:
                inactive_prefixes += ["engineered_projection"]
            if not self.config.use_solar_reference:
                inactive_prefixes += ["prior_projection"]
            for name, parameter in self.named_parameters():
                if any(name == prefix or name.startswith(prefix + ".") for prefix in inactive_prefixes):
                    parameter.requires_grad_(False)

        def latent_residual(self, raw):
            return self.target_center + self.target_scale * raw

        def latent_residual_at(self, raw, horizon):
            return self.target_center[:, horizon:horizon + 1] + self.target_scale[:, horizon:horizon + 1] * raw

        def state_from_residual(self, residual, solar_ref_state):
            if self.config.target_transform == "log1p_standard":
                state = torch.expm1(torch.log1p(solar_ref_state.clamp_min(0.0)) + residual)
            elif self.config.target_transform == "asinh_standard":
                state = torch.sinh(torch.asinh(solar_ref_state.clamp_min(0.0)) + residual)
            else:
                state = solar_ref_state + residual
            return torch.clamp(state, 0.0, 3.0)

        def raw_state_history(self, sequence):
            state = sequence[:, -self.lookback:, :, 0]
            return state * self.state_global_std[None, None, :] + self.state_global_mean[None, None, :]

        def reversible_normalize(self, values):
            if not self.config.use_revin:
                return values
            center = values.mean(dim=1, keepdim=True)
            scale = values.std(dim=1, keepdim=True, unbiased=False).clamp_min(0.08)
            return (values - center) / scale

        def nlinear_state(self, raw_state):
            last = raw_state[:, -1, :]
            centered = raw_state - last[:, None, :]
            update = torch.einsum("bdh,hd->bh", centered, self.nlinear_weight)
            return torch.clamp(last + update + self.nlinear_bias[None], 0.0, 3.0)

        def encode(self, sequence, astronomy, engineered, solar_ref_state, archived_weather, archive_mask):
            values = sequence[:, -self.lookback:, :, :self.channels]
            normalized = self.reversible_normalize(values)
            encoded, day_attention = self.encoder(normalized)
            channel_weights = torch.softmax(self.channel_logits, dim=-1)
            history = torch.sum(encoded * channel_weights[None, :, :, None], dim=2)
            if self.config.use_bigru:
                
                day_token = self.bigru_input(normalized[..., 0])
                bigru_day, _ = self.history_bigru(day_token)
                bigru_day = self.bigru_norm(bigru_day)
                query = self.horizon_embedding[0]
                score = torch.einsum("bdk,hk->bhd", bigru_day, query) / math.sqrt(query.shape[-1])
                bigru_attention = torch.softmax(score, dim=-1)
                bigru_context = torch.einsum("bhd,bdk->bhk", bigru_attention, bigru_day)
                bigru_gate = torch.sigmoid(self.bigru_gate_logit)
                history = history + bigru_gate * bigru_context
            else:
                bigru_gate = torch.zeros((), device=history.device, dtype=history.dtype)
            history = self.history_norm(history)
            
            astro = self.astronomy_projection(astronomy)
            if not self.config.use_astronomy:
                astro = torch.zeros_like(astro)
            engineered_global = self.engineered_projection(engineered)
            if not self.config.use_engineered:
                engineered_global = torch.zeros_like(engineered_global)
            engineered_token = engineered_global[:, None, :].expand(-1, N_HORIZONS, -1)
            solar_ref_token = self.prior_projection(torch.stack([solar_ref_state, torch.log1p(solar_ref_state)], dim=-1))
            if not self.config.use_solar_reference:
                solar_ref_token = torch.zeros_like(solar_ref_token)
            archive_token = self.archive_projection(archived_weather) * archive_mask[:, None, None]
            raw_state = self.raw_state_history(sequence)
            scale_values = torch.stack([
                raw_state[:, -3:].mean(dim=1), raw_state[:, -7:].mean(dim=1), raw_state.mean(dim=1)
            ], dim=-1)
            scale_weights = torch.softmax(self.scale_logits, dim=-1)
            scale_summary = torch.sum(scale_values * scale_weights[None], dim=-1, keepdim=True)
            scale_token = self.scale_projection(scale_summary)
            direct_token = self.direct_backbone(torch.cat([
                history, astro + self.horizon_embedding, engineered_token,
                solar_ref_token, scale_token + archive_token,
            ], dim=-1))
            state_attention = day_attention[:, :, 0, :]
            return (
                direct_token, engineered_global, channel_weights, state_attention,
                scale_weights, raw_state, bigru_gate,
            )

        def autoregressive(self, token, global_context, solar_ref_state, teacher_residual=None, ratio=0.0):
            b = token.shape[0]
            hidden = self.ar_initial(torch.cat([token.mean(dim=1), global_context], dim=-1))
            previous = torch.zeros((b, 1), device=token.device, dtype=token.dtype)
            states = []
            for horizon in range(N_HORIZONS):
                hidden = self.ar_cell(
                    torch.cat([token[:, horizon], self.previous_projection(previous)], dim=-1), hidden
                )
                residual = self.latent_residual_at(self.ar_head(hidden), horizon)
                current = self.state_from_residual(residual, solar_ref_state[:, horizon:horizon + 1])
                states.append(current)
                if self.training and teacher_residual is not None and ratio > 0 and horizon < N_HORIZONS - 1:
                    use_teacher = torch.rand((b, 1), device=token.device) < float(ratio)
                    previous = torch.where(use_teacher, teacher_residual[:, horizon:horizon + 1], residual.detach())
                else:
                    previous = residual
            return torch.cat(states, dim=1)

        def expert_weights(self):
            
            
            other_logits = self.expert_logits[:, [0, 1, 3]].clone()
            if not self.config.use_nlinear:
                other_logits[:, 0] = -30.0
            if not self.config.use_solar_reference:
                other_logits[:, 2] = -30.0
            other = torch.softmax(other_logits, dim=-1)
            if self.config.use_autoregressive:
                ar = float(self.config.max_ar_weight) * torch.sigmoid(self.expert_logits[:, 2:3])
            else:
                ar = torch.zeros_like(self.expert_logits[:, 2:3])
            other = other * (1.0 - ar)
            return torch.cat([other[:, :2], ar, other[:, 2:]], dim=-1)

        def decode(self, token, global_context, solar_ref_state, raw_state, teacher_residual=None, ratio=0.0):
            direct_residual = self.latent_residual(self.direct_head(token).squeeze(-1))
            direct_state = self.state_from_residual(direct_residual, solar_ref_state)
            nlinear = self.nlinear_state(raw_state) if self.config.use_nlinear else solar_ref_state
            ar_state = (
                self.autoregressive(token, global_context, solar_ref_state, teacher_residual, ratio)
                if self.config.use_autoregressive else solar_ref_state
            )
            weights = self.expert_weights()
            experts = torch.stack([nlinear, direct_state, ar_state, solar_ref_state], dim=-1)
            state = torch.sum(experts * weights[None], dim=-1)
            return state, direct_state, ar_state, nlinear, weights

        def forward(
            self, sequence, astronomy, archived_weather, archive_mask, engineered,
            solar_ref_state=None, teacher_state=None, teacher_forcing_ratio=0.0,
            return_aux=False, teacher_weather=None,
        ):
            b = sequence.shape[0]
            if solar_ref_state is None:
                solar_ref_state = torch.zeros((b, N_HORIZONS), device=sequence.device, dtype=sequence.dtype)
            if not self.config.use_solar_reference:
                solar_ref_state = torch.zeros_like(solar_ref_state)
            (
                token, global_context, channel_weights, day_attention,
                scale_weights, raw_state, bigru_gate,
            ) = self.encode(
                sequence, astronomy, engineered, solar_ref_state, archived_weather, archive_mask
            )
            teacher_decoded = self.decode(
                token, global_context, solar_ref_state, raw_state, teacher_state, teacher_forcing_ratio
            )
            free_decoded = self.decode(token, global_context, solar_ref_state, raw_state, None, 0.0)
            state, direct_state, ar_state, nlinear_state, weights = teacher_decoded
            free_state, free_direct, free_ar, free_nlinear, _ = free_decoded
            regime_logits = self.regime_head(token.mean(dim=1))
            predicted_weather = self.proxy_weather_head(torch.cat([
                token, global_context[:, None, :].expand(-1, N_HORIZONS, -1)
            ], dim=-1))
            log_scale = torch.clamp(self.log_scale_head(token).squeeze(-1), -4.0, 1.5)
            quantiles = self.quantile_head(token)
            
            decoder_weights = weights[:, [1, 2, 3]][None].expand(b, -1, -1)
            solar_mass = channel_weights[:, :min(4, self.channels)].sum(dim=-1)
            weather_mass = channel_weights[:, min(4, self.channels):].sum(dim=-1) if self.channels > 4 else torch.zeros_like(solar_mass)
            source_logits = torch.stack([
                solar_mass, weather_mass,
                torch.full_like(solar_mass, 1.0 if self.config.use_astronomy else -4.0),
                torch.full_like(solar_mass, 1.0 if self.config.use_engineered else -4.0),
                torch.full_like(solar_mass, -4.0),
            ], dim=-1)
            source_weights = torch.softmax(source_logits, dim=-1)[None].expand(b, -1, -1)
            if not return_aux:
                return state
            return {
                "state": state, "free_state": free_state,
                "direct_state": free_direct, "ar_state": free_ar,
                "structured_state": solar_ref_state, "solar_ref_state": solar_ref_state,
                "nlinear_state": free_nlinear,
                "source_weights": source_weights,
                "decoder_weights": decoder_weights,
                "day_attention": day_attention,
                "scale_weights": scale_weights[None].expand(b, -1, -1),
                "channel_weights": channel_weights[None].expand(b, -1, -1).mean(dim=1),
                "log_scale": log_scale, "quantiles": quantiles,
                "predicted_weather": predicted_weather,
                "regime_logits": regime_logits,
                "regime_probabilities": torch.softmax(regime_logits, dim=-1),
                "teacher_forcing_scale": torch.ones((b, 1), device=sequence.device),
                "bigru_gate": bigru_gate.reshape(1, 1, 1).expand(b, N_HORIZONS, 1),
                "expert_weights": weights[None].expand(b, -1, -1),
            }


In [ ]:
# Candidate Definitions
@dataclass(frozen=True)
class CandidateSpec:
    name: str
    use_bigru: bool
    use_tail_weighting: bool
    tail_guard_eligible: bool
    use_revin: bool
    use_nlinear: bool = True
    use_autoregressive: bool = True
    teacher_forcing: bool = True
    use_solar_reference: bool = True
    feature_mode: str = "no_annual"
    diagnostic: bool = False


CANDIDATE_SPECS = {
    "C1-MUFASA-Compact": CandidateSpec("C1-MUFASA-Compact", False, False, False, False),
    "C2-MUFASA-BiGRU": CandidateSpec("C2-MUFASA-BiGRU", True, False, False, False),
    "C3-MUFASA-Tail": CandidateSpec("C3-MUFASA-Tail", False, True, True, False),
    "C4-MUFASA-BiGRU-Tail": CandidateSpec("C4-MUFASA-BiGRU-Tail", True, True, True, False),
    "C5-MUFASA-Full": CandidateSpec("C5-MUFASA-Full", True, True, True, True),
    "D1-No-NLinear": CandidateSpec("D1-No-NLinear", True, False, False, False, use_nlinear=False, diagnostic=True),
    "D2-Direct-Only": CandidateSpec("D2-Direct-Only", True, False, False, False, use_autoregressive=False, diagnostic=True),
    "D3-No-Scheduled-TF": CandidateSpec("D3-No-Scheduled-TF", True, False, False, False, teacher_forcing=False, diagnostic=True),
    "D4-No-Solar-Geometry-Reference": CandidateSpec("D4-No-Solar-Geometry-Reference", True, False, False, False, use_solar_reference=False, diagnostic=True),
    "D5-Minimal-Features": CandidateSpec("D5-Minimal-Features", True, False, False, False, feature_mode="minimal_compact", diagnostic=True),
    "D6-Extended-Features": CandidateSpec("D6-Extended-Features", True, False, False, False, feature_mode="extended_compact", diagnostic=True),
}
PRIMARY_CANDIDATES = tuple(name for name in CANDIDATE_SPECS if name.startswith("C"))
DIAGNOSTIC_CANDIDATES = tuple(name for name in CANDIDATE_SPECS if name.startswith("D"))


def candidate_config(candidate_name, base_config=None):
    spec = CANDIDATE_SPECS[candidate_name]
    base_config = REFERENCE_CONFIG if base_config is None else base_config
    tf_start = base_config.teacher_forcing_start if spec.teacher_forcing else 0.0
    return replace(
        base_config,
        use_bigru=spec.use_bigru,
        use_tail_weighting=spec.use_tail_weighting,
        use_revin=spec.use_revin,
        use_nlinear=spec.use_nlinear,
        use_autoregressive=spec.use_autoregressive,
        teacher_forcing_start=tf_start,
        teacher_forcing_end=0.0,
        use_solar_reference=spec.use_solar_reference,
        engineered_feature_mode=spec.feature_mode,
        use_proxy_weather=False,
        batch_size=GPU_BATCH_SIZE,
    )


def _choice(values, u):
    return values[min(int(float(u) * len(values)), len(values) - 1)]


def model_parameter_profile(trained):
    if getattr(trained, "kind", "") == "torch":
        parameters = list(trained.model.parameters())
        active = int(sum(p.numel() for p in parameters if p.requires_grad))
        total = int(sum(p.numel() for p in parameters))
        return {
            "Active_parameter_count": active,
            "Total_parameter_count": total,
            "Active_parameter_ratio": active / max(total, 1),
            "Model_size_MB_fp32": total * 4 / (1024 ** 2),
        }
    model = getattr(trained, "model", None)
    active = int(sum(
        np.asarray(value).size
        for value in (getattr(model[-1], "coef_", []), getattr(model[-1], "intercept_", []))
    )) if hasattr(model, "__getitem__") else 0
    return {
        "Active_parameter_count": active, "Total_parameter_count": active,
        "Active_parameter_ratio": 1.0, "Model_size_MB_fp32": active * 4 / (1024 ** 2),
    }


def model_parameter_count(trained):
    return model_parameter_profile(trained)["Active_parameter_count"]


RETIRED_CANDIDATES = pd.DataFrame([{
    "Candidate": "C0-MUFASA-CurrentFull", "Status": "retired_duplicate",
    "Duplicate_of": "C5-MUFASA-Full",
    "Reason": "The previous notebook instantiated identical flags for C0 and C5.",
    "Selection_uses_test": False,
}])
RETIRED_CANDIDATES.to_csv(OUTPUT_DIR / "MUFASA_retired_duplicate_candidates.csv", index=False)

assert CANONICAL_CORE_CANDIDATE == "C2-MUFASA-BiGRU"
assert set(PRIMARY_CANDIDATES) == {
    "C1-MUFASA-Compact", "C2-MUFASA-BiGRU", "C3-MUFASA-Tail",
    "C4-MUFASA-BiGRU-Tail", "C5-MUFASA-Full",
}


In [ ]:
# Training Losses and Scheduled Teacher Forcing
def teacher_forcing_probability(epoch, max_epochs, config):
    if not config.use_autoregressive or config.teacher_forcing_start <= 0:
        return 0.0
    zero_epoch = max(2, min(int(config.tf_zero_epoch), int(max_epochs)))
    if epoch >= zero_epoch:
        return 0.0
    progress = (epoch - 1) / max(zero_epoch - 1, 1)
    return float(config.teacher_forcing_start * 0.5 * (1.0 + math.cos(math.pi * progress)))


def mufasa_loss(
    output, state_target, potential, weather_target, regime_target,
    tail_weight, config, regime_weights,
):
    
    sample_weight = tail_weight.reshape(-1).clamp(1.0, float(config.tail_weight_cap))
    sample_weight = sample_weight / sample_weight.mean().clamp_min(1e-6)

    def weighted_mean(per_sample):
        return torch.mean(per_sample * sample_weight)

    def primary(state_prediction, weighted):
        error = (state_prediction - state_target) * potential
        mse = torch.mean(error ** 2, dim=1)
        huber = F.smooth_l1_loss(
            error, torch.zeros_like(error), beta=0.15, reduction="none"
        ).mean(dim=1)
        per_sample = 0.80 * mse + 0.20 * huber
        return weighted_mean(per_sample) if weighted else torch.mean(per_sample)

    
    teacher_primary = primary(output["state"], weighted=False)
    free_primary = primary(output["free_state"], weighted=config.use_tail_weighting)
    physical_prediction = output["free_state"] * potential
    physical_target = state_target * potential
    ramp_per_sample = torch.mean(
        (torch.diff(physical_prediction, dim=1) - torch.diff(physical_target, dim=1)) ** 2,
        dim=1,
    )
    ramp_loss = weighted_mean(ramp_per_sample) if config.use_tail_weighting else ramp_per_sample.mean()
    energy_loss = torch.mean(
        (physical_prediction.sum(dim=1) - physical_target.sum(dim=1)) ** 2
    ) / N_HORIZONS
    predicted_shape = physical_prediction / physical_prediction.sum(dim=1, keepdim=True).clamp_min(1e-4)
    target_shape = physical_target / physical_target.sum(dim=1, keepdim=True).clamp_min(1e-4)
    shape_loss = torch.mean((predicted_shape - target_shape) ** 2)
    peak_per_sample = (
        physical_prediction.max(dim=1).values - physical_target.max(dim=1).values
    ) ** 2
    peak_loss = weighted_mean(peak_per_sample) if config.use_tail_weighting else peak_per_sample.mean()
    normalized_squared = (output["free_state"] - state_target) ** 2
    nll = torch.mean(
        0.5 * torch.exp(-2.0 * output["log_scale"]) * normalized_squared
        + output["log_scale"]
    )
    consistency = torch.mean((output["state"] - output["free_state"]) ** 2)
    zero = torch.zeros((), device=state_target.device)
    weather_loss = (
        F.smooth_l1_loss(output["predicted_weather"], weather_target, beta=0.30)
        if config.use_proxy_weather else zero
    )
    regime_loss = (
        F.cross_entropy(output["regime_logits"], regime_target, weight=regime_weights)
        if config.use_regime_head else zero
    )
    expert_curve = output["expert_weights"].mean(dim=0)
    gate_regularization = torch.mean(torch.diff(expert_curve, dim=0) ** 2)
    return (
        config.free_run_weight * free_primary
        + config.teacher_aux_weight * teacher_primary
        + config.consistency_weight * consistency
        + config.ramp_weight * ramp_loss + config.energy_weight * energy_loss
        + config.shape_weight * shape_loss + config.peak_weight * peak_loss
        + config.weather_aux_weight * weather_loss + config.regime_weight * regime_loss
        + config.uncertainty_weight * nll + config.gate_reference_weight * gate_regularization
    )


In [ ]:
# Shared Training and Prediction Routines — part 1/2
def _residual_target(bundle, solar_ref_state, target_transform):
    if target_transform == "log1p_standard":
        return (
            np.log1p(np.clip(bundle.target_state, 0.0, None))
            - np.log1p(np.clip(solar_ref_state, 0.0, None))
        ).astype(np.float32)
    if target_transform == "asinh_standard":
        return (np.arcsinh(bundle.target_state) - np.arcsinh(solar_ref_state)).astype(np.float32)
    return (bundle.target_state - solar_ref_state).astype(np.float32)
def _transform_mufasa(bundle, indices, scalers, solar_ref_state=None, training_weights=False):
    indices = np.asarray(indices, dtype=int)
    if solar_ref_state is None:
        solar_ref_state = predict_solar_reference(bundle, scalers["reference_model"])
    residual_target = _residual_target(bundle, solar_ref_state, scalers["target_transform"])
    if training_weights and scalers.get("use_tail_weighting", False):
        if np.any(bundle.test[indices]):
            raise AssertionError("Test rows cannot be used to construct LDS training weights.")
        tail_weight = _lds_sample_weights(
            _rare_ramp_score(bundle, indices), scalers["tail_edges"], scalers["tail_density"],
            scalers["tail_weight_strength"], scalers["tail_weight_cap"],
        )
    else:
        
        tail_weight = np.ones(len(indices), np.float32)
    return (
        ((bundle.sequence[indices] - scalers["sequence_mean"]) / scalers["sequence_std"]).astype(np.float32),
        ((bundle.astronomy[indices] - scalers["astronomy_mean"]) / scalers["astronomy_std"]).astype(np.float32),
        ((bundle.archived_weather[indices] - scalers["archive_mean"]) / scalers["archive_std"]).astype(np.float32),
        bundle.archived_weather_mask[indices].astype(np.float32),
        ((
            bundle.tabular[np.ix_(indices, scalers["engineered_columns"])]
            - scalers["engineered_mean"]
        ) / scalers["engineered_std"]).astype(np.float32),
        solar_ref_state[indices].astype(np.float32), residual_target[indices].astype(np.float32),
        bundle.target_state[indices].astype(np.float32), bundle.potential[indices].astype(np.float32),
        ((bundle.target_weather[indices] - scalers["weather_mean"]) / scalers["weather_std"]).astype(np.float32),
        _regime_labels(bundle, indices, scalers["regime_thresholds"]),
        tail_weight.astype(np.float32),
    )
def _torch_predict_arrays(model, sequence, astronomy, archive, archive_mask, engineered, solar_ref_state, mc_passes=1, mc_seed=SEED):
    all_states, all_aux = [], []
    passes = max(1, int(mc_passes))
    for draw in range(passes):
        set_seed(mc_seed + draw)
        model.train() if passes > 1 else model.eval()
        state_parts = []
        aux_parts = {key: [] for key in [
            "source_weights", "decoder_weights", "day_attention", "scale_weights",
            "channel_weights", "log_scale", "direct_state", "ar_state",
            "structured_state", "solar_ref_state", "predicted_weather",
            "regime_probabilities", "teacher_forcing_scale", "bigru_gate",
        ]}
        with torch.inference_mode():
            inference_batch = max(
                16, min(256, int(getattr(getattr(model, "config", None), "batch_size", GPU_BATCH_SIZE)))
            )
            for start in range(0, len(sequence), inference_batch):
                stop = start + inference_batch
                with _amp_context():
                    output = model(
                        torch.from_numpy(sequence[start:stop]).to(DEVICE, non_blocking=True),
                        torch.from_numpy(astronomy[start:stop]).to(DEVICE, non_blocking=True),
                        torch.from_numpy(archive[start:stop]).to(DEVICE, non_blocking=True),
                        torch.from_numpy(archive_mask[start:stop]).to(DEVICE, non_blocking=True),
                        torch.from_numpy(engineered[start:stop]).to(DEVICE, non_blocking=True),
                        torch.from_numpy(solar_ref_state[start:stop]).to(DEVICE, non_blocking=True),
                        teacher_state=None, teacher_forcing_ratio=0.0, return_aux=True,
                    )
                state_parts.append(output["state"].float().cpu().numpy())
                for key in aux_parts:
                    aux_parts[key].append(output[key].float().cpu().numpy())
        all_states.append(np.concatenate(state_parts))
        all_aux.append({key: np.concatenate(parts) for key, parts in aux_parts.items()})
    model.eval()
    state_draws = np.stack(all_states)
    mean_state = state_draws.mean(axis=0)
    if mean_state.ndim != 2 or mean_state.shape[1] != N_HORIZONS:
        raise ValueError(f"Neural prediction shape={mean_state.shape}; expected [N, {N_HORIZONS}].")
    if not np.isfinite(mean_state).all():
        raise FloatingPointError("Neural prediction contains NaN or infinity.")
    epistemic_std = state_draws.std(axis=0, ddof=1) if passes > 1 else np.zeros_like(mean_state)
    mean_aux = {key: np.mean(np.stack([item[key] for item in all_aux]), axis=0) for key in all_aux[0]}
    aleatoric_std = np.exp(mean_aux["log_scale"])
    mean_aux["epistemic_std_state"] = epistemic_std
    mean_aux["aleatoric_std_state"] = aleatoric_std
    mean_aux["total_std_state"] = np.sqrt(epistemic_std ** 2 + aleatoric_std ** 2)
    mean_aux["mc_passes"] = passes
    mean_aux["teacher_forcing_ratio"] = 0.0
    return mean_state, mean_aux
def validate_mufasa_config(config):
    if int(config.d_model) % int(config.n_heads) != 0:
        raise ValueError(
            f"d_model={config.d_model} must be divisible by n_heads={config.n_heads}."
        )
    if not (2 <= int(config.lookback) <= MAX_LOOKBACK_DAYS):
        raise ValueError(
            f"lookback must be in [2, {MAX_LOOKBACK_DAYS}], found {config.lookback}."
        )
    if int(config.n_layers) < 1:
        raise ValueError(f"n_layers must be positive, found {config.n_layers}.")
    if not (0.0 <= float(config.dropout) < 0.8):
        raise ValueError(f"dropout must be in [0, 0.8), found {config.dropout}.")
    if int(config.batch_size) < 1:
        raise ValueError(f"batch_size must be positive, found {config.batch_size}.")
    if not (0.0 <= float(config.teacher_forcing_start) <= 1.0):
        raise ValueError("teacher_forcing_start must be in [0, 1].")
    if int(config.tf_zero_epoch) < 2 or int(config.free_run_min_epochs) < 1:
        raise ValueError("The zero-TF epoch and free-run phase are invalid.")
    if config.use_bigru and int(config.d_model) % 2 != 0:
        raise ValueError("Historical BiGRU requires an even d_model.")
    if not (0.0 <= float(config.bigru_gate_init) <= 1.0):
        raise ValueError("bigru_gate_init must be in [0, 1].")
    if not (1.0 <= float(config.tail_weight_cap) <= 3.0):
        raise ValueError("tail_weight_cap must be in [1, 3].")
    return config
def _validate_mufasa_arrays(bundle, indices, arrays, stage):
    indices = np.asarray(indices, int)
    names = (
        "sequence", "astronomy", "archived_weather", "archive_mask", "engineered",
        "solar_ref_state", "teacher_residual", "target_state", "potential",
        "target_weather_label", "target_regime_label", "tail_weight",
    )
    if len(arrays) != len(names):
        raise RuntimeError(f"{bundle.site} [{stage}]: expected 12 arrays, found {len(arrays)}.")
    for name, values in zip(names, arrays):
        values = np.asarray(values)
        if values.shape[0] != len(indices):
            raise ValueError(
                f"{bundle.site} [{stage}] {name}: rows={values.shape[0]} != {len(indices)}."
            )
        if not np.isfinite(values).all():
            first = np.argwhere(~np.isfinite(values))[:5].tolist()
            raise FloatingPointError(
                f"{bundle.site} [{stage}] {name} contains NaN/inf; first={first}."
            )
    (sequence, astronomy, archive, archive_mask, engineered, prior, residual, target,
     potential, weather_label, regime_label, tail_weight) = arrays
    if sequence.ndim != 4 or sequence.shape[2] != N_HORIZONS:
        raise ValueError(
            f"{bundle.site} [{stage}] sequence must be [N, days, {N_HORIZONS}, C], "
            f"found {sequence.shape}."
        )
    expected_curve = (len(indices), N_HORIZONS)
    for name, values in (("prior", prior), ("residual", residual), ("target", target), ("potential", potential)):
        if tuple(values.shape) != expected_curve:
            raise ValueError(
                f"{bundle.site} [{stage}] {name} shape={values.shape}; expected={expected_curve}."
            )
    if astronomy.shape[:2] != expected_curve or archive.shape[:2] != expected_curve:
        raise ValueError(
            f"{bundle.site} [{stage}] future-known tensor horizon mismatch: "
            f"astronomy={astronomy.shape}, archive={archive.shape}."
        )
    if tail_weight.shape != (len(indices),) or np.any(tail_weight < 1.0 - 1e-6):
        raise ValueError(f"{bundle.site} [{stage}] invalid LDS weight shape/range={tail_weight.shape}.")
    if np.any(bundle.test[indices]) and stage != "prediction":
        raise AssertionError(f"{bundle.site} [{stage}]: test rows entered a fit transform.")
FIT_RETRY_ROWS = []
def _release_torch_memory():
    gc.collect()
    if TORCH_AVAILABLE and torch.cuda.is_available():
        torch.cuda.empty_cache()
def fit_fast_linear_surrogate(bundle, train_idx, valid_idx, config, seed, **kwargs):
    reference_model = fit_solar_reference(bundle, train_idx, config.reference_alpha, config.use_solar_reference)
    solar_ref_state = predict_solar_reference(bundle, reference_model)
    scalers = fit_scalers(
        bundle, train_idx, config.scaling_mode, config.target_transform, solar_ref_state,
        reference_model, config.engineered_feature_mode,
        config.use_tail_weighting, config.tail_weight_strength, config.tail_weight_cap,
    )
    columns = scalers["engineered_columns"]
    model = make_pipeline(StandardScaler(), Ridge(alpha=3000.0)).fit(
        bundle.tabular[np.ix_(train_idx, columns)],
        bundle.target_state[train_idx] - solar_ref_state[train_idx],
    )
    if len(valid_idx):
        state = np.clip(
            solar_ref_state[valid_idx]
            + model.predict(bundle.tabular[np.ix_(valid_idx, columns)]), 0.0, 3.0
        )
        validation_rmse = metric_set(bundle.target[valid_idx], state * bundle.potential[valid_idx])["RMSE"]
    else:
        validation_rmse = float("nan")
    return TrainedMUFASA(
        "fast_linear_surrogate", config, model, scalers, reference_model, 1,
        validation_rmse, seed, [0.0],
    )


In [ ]:
# Shared Training and Prediction Routines — part 2/2
def fit_torch_mufasa(
    bundle, train_idx, valid_idx, config, seed, fixed_epochs=None,
    max_epochs_override=None, patience_override=None, amp_enabled_override=None,
):
    set_seed(seed)
    train_idx = np.asarray(train_idx, dtype=int)
    
    if SMOKE_TRAIN_LIMIT > 0 and len(train_idx) > SMOKE_TRAIN_LIMIT:
        train_idx = train_idx[-SMOKE_TRAIN_LIMIT:]
    reference_model = fit_solar_reference(bundle, train_idx, config.reference_alpha, config.use_solar_reference)
    solar_ref_state = predict_solar_reference(bundle, reference_model)
    scalers = fit_scalers(
        bundle, train_idx, config.scaling_mode, config.target_transform, solar_ref_state,
        reference_model, config.engineered_feature_mode,
        config.use_tail_weighting, config.tail_weight_strength, config.tail_weight_cap,
    )
    tr = _transform_mufasa(bundle, train_idx, scalers, solar_ref_state, training_weights=True)
    va = _transform_mufasa(bundle, valid_idx, scalers, solar_ref_state) if len(valid_idx) else None
    _validate_mufasa_arrays(bundle, train_idx, tr, "training")
    if va is not None:
        _validate_mufasa_arrays(bundle, valid_idx, va, "validation_transform")
    model = MUFASANet(
        tuple(tr[0].shape[1:]), tr[1].shape[-1], tr[2].shape[-1], tr[4].shape[-1],
        tr[9].shape[-1], scalers, config
    ).to(DEVICE)
    active_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not active_parameters:
        raise RuntimeError("No active MUFASA parameters remain after candidate masking.")
    optimizer = torch.optim.AdamW(active_parameters, lr=config.learning_rate, weight_decay=config.weight_decay)
    requested_epochs = int(fixed_epochs) if fixed_epochs is not None else int(max_epochs_override or MUFASA_MAX_EPOCHS)
    tf_phase = int(config.tf_zero_epoch) if config.teacher_forcing_start > 0 else 0
    minimum_eligible = min(
        requested_epochs, max(1, int(tf_phase + config.free_run_min_epochs))
    )
    max_epochs = max(requested_epochs, minimum_eligible)
    patience = max_epochs + 1 if fixed_epochs is not None else int(patience_override or MUFASA_PATIENCE)
    loader = DataLoader(
        TensorDataset(*[torch.from_numpy(array) for array in tr]),
        batch_size=config.batch_size, shuffle=True, drop_last=False,
        pin_memory=DEVICE.type == "cuda", num_workers=0,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, max_epochs), eta_min=config.learning_rate / 100.0
    )
    amp_is_enabled = USE_AMP if amp_enabled_override is None else bool(
        amp_enabled_override and DEVICE.type == "cuda"
    )
    try:
        amp_scaler = torch.amp.GradScaler("cuda", enabled=amp_is_enabled)
    except Exception:
        amp_scaler = torch.cuda.amp.GradScaler(enabled=amp_is_enabled)
    best_rmse, best_state, best_epoch, stale = np.inf, None, 0, 0
    tf_history = []
    for epoch in range(1, max_epochs + 1):
        ratio = teacher_forcing_probability(epoch, max_epochs, config)
        tf_history.append(ratio)
        model.train()
        for batch in loader:
            (sequence, astronomy, archive, archive_mask, engineered, solar_ref_batch,
             teacher_residual, state_target, potential, weather_target, regime_target,
             tail_weight) = [
                value.to(DEVICE, dtype=torch.float32, non_blocking=True) for value in batch
            ]
            optimizer.zero_grad(set_to_none=True)
            with _amp_context(amp_enabled_override):
                output = model(
                    sequence, astronomy, archive, archive_mask, engineered, solar_ref_batch,
                    teacher_state=teacher_residual, teacher_forcing_ratio=ratio,
                    return_aux=True, teacher_weather=weather_target,
                )
                loss = mufasa_loss(
                    output, state_target, potential, weather_target, regime_target.long(),
                    tail_weight, config, model.regime_weights,
                )
            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"{bundle.site}: non-finite training loss at epoch={epoch}; "
                    f"AMP={USE_AMP if amp_enabled_override is None else amp_enabled_override}."
                )
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if not torch.isfinite(grad_norm):
                raise FloatingPointError(
                    f"{bundle.site}: non-finite gradient norm at epoch={epoch}."
                )
            amp_scaler.step(optimizer)
            amp_scaler.update()
        scheduler.step()
        if fixed_epochs is not None:
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            continue
        state, _ = _torch_predict_arrays(model, *va[:6], mc_passes=1, mc_seed=seed)
        physical = state * bundle.potential[valid_idx]
        validation_rmse = metric_set(bundle.target[valid_idx], physical)["RMSE"]
        if epoch < minimum_eligible:
            continue
        if validation_rmse < best_rmse - 1e-5:
            best_rmse, best_epoch, stale = validation_rmse, epoch, 0
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        else:
            stale += 1
            if stale >= patience:
                break
    if best_state is None:
        raise RuntimeError(f"{bundle.site}: MUFASA CI core did not retain a free-run-eligible state.")
    model.load_state_dict(best_state)
    model.eval()
    return TrainedMUFASA(
        "torch", config, model, scalers, reference_model, int(best_epoch),
        float(best_rmse), seed, tf_history[:int(best_epoch)] if fixed_epochs is None else tf_history,
    )
def fit_mufasa(bundle, train_idx, valid_idx, config, seed, fixed_epochs=None, **kwargs):
    validate_mufasa_config(config)
    if not TORCH_AVAILABLE:
        return fit_fast_linear_surrogate(
            bundle, train_idx, valid_idx, config, seed, fixed_epochs=fixed_epochs, **kwargs
        )

    current = config
    amp_override = None
    failures = []
    for attempt in range(4):
        try:
            trained = fit_torch_mufasa(
                bundle, train_idx, valid_idx, current, seed, fixed_epochs,
                amp_enabled_override=amp_override, **kwargs,
            )
            FIT_RETRY_ROWS.append({
                "Site": bundle.site, "Attempt": attempt + 1, "Status": "ok",
                "Batch_size": current.batch_size,
                "AMP": USE_AMP if amp_override is None else bool(amp_override),
                "Error": "",
            })
            return trained
        except Exception as exc:
            message = f"{type(exc).__name__}: {exc}"
            lower = message.lower()
            numerical = isinstance(exc, FloatingPointError) or "nan" in lower or "non-finite" in lower
            memory = "out of memory" in lower or "cublas_status_alloc_failed" in lower
            cudnn = "cudnn_status" in lower or "unable to find a valid cudnn" in lower
            retryable = numerical or memory or cudnn
            failures.append(message)
            FIT_RETRY_ROWS.append({
                "Site": bundle.site, "Attempt": attempt + 1, "Status": "failed",
                "Batch_size": current.batch_size,
                "AMP": USE_AMP if amp_override is None else bool(amp_override),
                "Error": message,
            })
            _release_torch_memory()
            if not retryable:
                raise
            if numerical:
                if amp_override is False:
                    current = replace(current, batch_size=max(16, current.batch_size // 2))
                amp_override = False
            else:
                next_batch = max(16, current.batch_size // 2)
                if next_batch == current.batch_size:
                    raise RuntimeError(
                        f"{bundle.site}: CUDA retry floor reached. Failures={failures}"
                    ) from exc
                current = replace(current, batch_size=next_batch)
    raise RuntimeError(
        f"{bundle.site}: robust GPU fit exhausted all retries. Failures={failures}"
    )
def predict_mufasa(trained, bundle, indices, return_aux=False, mc_passes=None):
    indices = np.asarray(indices, dtype=int)
    solar_ref_state = predict_solar_reference(bundle, trained.reference_model)
    if trained.kind != "torch":
        columns = trained.scalers["engineered_columns"]
        state = np.clip(
            solar_ref_state[indices]
            + trained.model.predict(bundle.tabular[np.ix_(indices, columns)]), 0.0, 3.0
        )
        n = len(indices)
        auxiliary = {
            "source_weights": np.tile(np.array([0.2] * 5)[None, None, :], (n, N_HORIZONS, 1)),
            "decoder_weights": np.tile(np.array([0.0, 0.0, 1.0])[None, None, :], (n, N_HORIZONS, 1)),
            "day_attention": np.full((n, N_HORIZONS, trained.config.lookback), 1.0 / trained.config.lookback),
            "log_scale": np.zeros((n, N_HORIZONS)), "direct_state": state, "ar_state": state,
            "structured_state": solar_ref_state[indices], "solar_ref_state": solar_ref_state[indices],
            "total_std_state": np.zeros_like(state), "teacher_forcing_ratio": 0.0, "mc_passes": 1,
            "predicted_weather": np.zeros((n, N_HORIZONS, bundle.target_weather.shape[-1])),
            "regime_probabilities": np.full((n, len(REGIME_NAMES)), 1.0 / len(REGIME_NAMES)),
            "teacher_forcing_scale": np.ones((n, 1)),
            "bigru_gate": np.zeros((n, N_HORIZONS, 1)),
            "scale_weights": np.full((n, N_HORIZONS, 3), 1.0 / 3.0),
            "channel_weights": np.ones((
                n, 4 if not trained.config.use_weather else bundle.sequence.shape[-1]
            )),
        }
    else:
        transformed = _transform_mufasa(bundle, indices, trained.scalers, solar_ref_state)
        state, auxiliary = _torch_predict_arrays(
            trained.model, *transformed[:6], mc_passes=MC_PASSES if mc_passes is None else mc_passes,
            mc_seed=trained.seed + 10000,
        )
    prediction = np.clip(state * bundle.potential[indices], 0.0, None)
    expected_shape = (len(indices), N_HORIZONS)
    if prediction.shape != expected_shape or not np.isfinite(prediction).all():
        raise FloatingPointError(
            f"{bundle.site}: prediction shape/finite audit failed; "
            f"shape={prediction.shape}, expected={expected_shape}."
        )
    auxiliary["predictive_std_physical"] = auxiliary["total_std_state"] * bundle.potential[indices]
    return (prediction, auxiliary) if return_aux else prediction
print(
    "MUFASA core active: literature features + CI Patch-TCN + historical BiGRU residual + "
    "LDS rare-ramp weighting + causal zero-ended scheduled teacher forcing."
)


## 4. Development selection and aggregation

Use 2019 only for block evaluation, architecture screening, Bayesian optimization, seed consensus, the Ridge expert, and development-only aggregation calibration.

**Run note.** Execute the cells in this section in order. Objects created here are consumed by later sections; the notebook intentionally avoids hidden state restoration from unpublished artifacts.


In [ ]:
# Chronological Development-Block Evaluation
VALIDATION_BLOCKS = {
    "Q1": (pd.Timestamp("2019-01-01"), pd.Timestamp("2019-03-31")),
    "Q2": (pd.Timestamp("2019-04-01"), pd.Timestamp("2019-06-30")),
    "Q3": (pd.Timestamp("2019-07-01"), pd.Timestamp("2019-09-30")),
    "Q4": (pd.Timestamp("2019-10-01"), pd.Timestamp("2019-12-31")),
}


def temporal_block_metrics(truth, prediction, dates):
    truth, prediction = np.asarray(truth), np.asarray(prediction)
    dates = pd.DatetimeIndex(dates)
    overall = metric_set(truth, prediction)
    row = {
        "Overall_RMSE": overall["RMSE"], "Overall_MAE": overall["MAE"],
        "Overall_R2": overall["R2"],
    }
    block_rmse, block_mae = [], []
    for name, (start, end) in VALIDATION_BLOCKS.items():
        mask = (dates >= start) & (dates <= end)
        if not np.any(mask):
            raise AssertionError(f"Validation block {name} is empty.")
        values = metric_set(truth[mask], prediction[mask])
        row[f"{name}_RMSE"] = values["RMSE"]
        row[f"{name}_MAE"] = values["MAE"]
        block_rmse.append(values["RMSE"]); block_mae.append(values["MAE"])
    row.update({
        "Mean_block_RMSE": float(np.mean(block_rmse)),
        "SD_block_RMSE": float(np.std(block_rmse, ddof=1)),
        "Worst_block_RMSE": float(np.max(block_rmse)),
        "Mean_block_MAE": float(np.mean(block_mae)),
        "SD_block_MAE": float(np.std(block_mae, ddof=1)),
    })
    return row


def stable_near_best(frame, name_column, tolerance=STRUCTURE_NEAR_BEST_TOL):
    if frame.empty:
        raise ValueError("Stable selection received an empty table.")
    best_mean = float(frame["Mean_block_RMSE"].min())
    near = frame[frame["Mean_block_RMSE"] <= best_mean * (1.0 + tolerance)].copy()
    near = near.sort_values(
        ["SD_block_RMSE", "Parameter_count", "Worst_block_RMSE", "Mean_block_RMSE", name_column],
        kind="mergesort",
    )
    return near.iloc[0], near


def seven_day_block_ci(difference, dates, repetitions=BLOCK_BOOTSTRAP_REPS, seed=SEED):
    values = np.asarray(difference, float)
    dates = pd.DatetimeIndex(dates)
    order = np.argsort(dates.values)
    values = values[order]
    if len(values) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    block = min(BLOCK_BOOTSTRAP_LENGTH, len(values))
    starts = np.arange(max(1, len(values) - block + 1))
    draws = []
    for _ in range(int(repetitions)):
        sampled = []
        while len(sampled) < len(values):
            start = int(rng.choice(starts))
            sampled.extend(values[start:start + block].tolist())
        draws.append(np.mean(sampled[:len(values)]))
    return tuple(np.quantile(draws, [0.025, 0.975]).astype(float))


def daily_squared_loss(truth, prediction):
    return np.mean((np.asarray(truth) - np.asarray(prediction)) ** 2, axis=1)


In [ ]:
# Stage A — Architecture Screening
def run_candidate_screen(selection_bundles, scenario, candidates=PRIMARY_CANDIDATES):
    rows, reasons, artifacts, top_two = [], [], {}, {}
    for site, bundle in selection_bundles.items():
        train_idx, val_idx = positions(bundle.train), positions(bundle.val_all)
        artifacts[site] = {}
        site_rows = []
        for candidate_name in candidates:
            config = candidate_config(candidate_name)
            seed_predictions, seed_models = [], []
            started = time.time()
            for seed in ACTIVE_STRUCTURE_SEEDS:
                trained = fit_mufasa(
                    bundle, train_idx, val_idx, config, seed,
                    max_epochs_override=STRUCTURE_SCREEN_EPOCHS,
                    patience_override=STRUCTURE_SCREEN_PATIENCE,
                )
                prediction = predict_mufasa(trained, bundle, val_idx, mc_passes=1)
                seed_predictions.append(prediction); seed_models.append(trained)
                if prediction.shape != bundle.target[val_idx].shape:
                    raise AssertionError(f"{site}/{candidate_name}: validation shape mismatch")
            mean_prediction = np.mean(np.stack(seed_predictions, axis=-1), axis=-1)
            metrics = temporal_block_metrics(
                bundle.target[val_idx], mean_prediction, bundle.target_dates[val_idx]
            )
            profiles = [model_parameter_profile(model) for model in seed_models]
            row = {
                "Scenario": scenario, "Site": site, "Candidate": candidate_name,
                "Seeds": "|".join(map(str, ACTIVE_STRUCTURE_SEEDS)),
                "Expected_paper_seeds": "11|29|47",
                "Active_parameter_count": max(p["Active_parameter_count"] for p in profiles),
                "Total_parameter_count": max(p["Total_parameter_count"] for p in profiles),
                "Active_parameter_ratio": min(p["Active_parameter_ratio"] for p in profiles),
                "Model_size_MB_fp32": max(p["Model_size_MB_fp32"] for p in profiles),
                "Parameter_count": max(p["Active_parameter_count"] for p in profiles),
                "Seconds": time.time() - started,
                "Selection_uses_test": False,
                **metrics,
            }
            site_rows.append(row)
            artifacts[site][candidate_name] = {
                "config": config, "models": seed_models,
                "seed_predictions": np.stack(seed_predictions, axis=-1),
                "prediction": mean_prediction,
            }
        site_frame = pd.DataFrame(site_rows)
        for quarter in VALIDATION_BLOCKS:
            winner = site_frame[f"{quarter}_RMSE"].idxmin()
            site_frame.loc[winner, "Block_win_count"] = site_frame.loc[winner].get("Block_win_count", 0) + 1
        site_frame["Block_win_count"] = site_frame.get("Block_win_count", 0).fillna(0).astype(int)
        selected, near = stable_near_best(site_frame, "Candidate")
        site_frame["Selected"] = site_frame["Candidate"].eq(selected["Candidate"])
        ranked = site_frame.sort_values(
            ["Mean_block_RMSE", "SD_block_RMSE", "Parameter_count", "Candidate"],
            kind="mergesort",
        )
        
        challengers = ranked[~ranked["Candidate"].eq(CANONICAL_CORE_CANDIDATE)]
        top_two[site] = [CANONICAL_CORE_CANDIDATE]
        if len(challengers) and HPO_TOP_K > 1:
            top_two[site].append(str(challengers.iloc[0]["Candidate"]))
        sensitivity = {}
        for tolerance in STRUCTURE_TOL_SENSITIVITY:
            picked, _ = stable_near_best(site_frame, "Candidate", tolerance)
            sensitivity[f"tol_{tolerance:.4f}"] = picked["Candidate"]
        second = ranked.iloc[1] if len(ranked) > 1 else selected
        reasons.append({
            "Scenario": scenario, "Site": site,
            "Selected_candidate": selected["Candidate"],
            "Near_best_candidates": "|".join(near["Candidate"].tolist()),
            "Mean_RMSE_difference_percent": 100 * (
                float(second["Mean_block_RMSE"]) / max(float(selected["Mean_block_RMSE"]), 1e-12) - 1
            ),
            "Stability_reason": "0.5% near-best set -> lower Q1-Q4 RMSE SD",
            "Complexity_reason": "remaining tie -> fewer active trainable parameters (inactive modules frozen)",
            "Tolerance_sensitivity": json.dumps(sensitivity, ensure_ascii=False),
            "Selection_uses_test": False,
        })
        rows.extend(site_frame.to_dict("records"))
    return pd.DataFrame(rows), pd.DataFrame(reasons), artifacts, top_two


STRICT_CANDIDATE_SCREEN, STRICT_SELECTION_REASON, STRICT_SCREEN_ARTIFACTS, STRICT_TOP_TWO = (
    run_candidate_screen(STRICT_SELECTION_BUNDLES, "strict_history")
)
STRICT_CANDIDATE_SCREEN.to_csv(OUTPUT_DIR / "MUFASA_candidate_screen_2019.csv", index=False)
STRICT_SELECTION_REASON.to_csv(OUTPUT_DIR / "MUFASA_candidate_selection_reason.csv", index=False)
display(STRICT_CANDIDATE_SCREEN.sort_values(["Site", "Mean_block_RMSE"]).round(5))


In [ ]:
# Stage B — Gaussian-Process Bayesian Optimization
REDUCED_HPO_DIMENSION = 12
REDUCED_DMODEL = (48, 64, 80, 96)
REDUCED_LAYERS = (1, 2, 3)
REDUCED_LOOKBACK = (14, 21, 28)
REDUCED_SCALING = (
    "standard_hourwise", "robust_hourwise",
    "standard_channelwise", "robust_channelwise",
)
REDUCED_TARGET = ("identity_standard", "asinh_standard", "log1p_standard")


def decode_reduced_vector(vector, candidate_name):
    u = np.clip(np.asarray(vector, float), 0.0, 1.0 - 1e-12)
    spec = CANDIDATE_SPECS[candidate_name]
    zero_choices = (2,) if SMOKE else (4, 6, 8, 10)
    config = candidate_config(candidate_name)
    return replace(
        config,
        d_model=_choice(REDUCED_DMODEL, u[0]),
        n_layers=_choice(REDUCED_LAYERS, u[1]),
        dropout=float(0.04 + 0.18 * u[2]),
        learning_rate=float(np.exp(np.log(2e-4) + u[3] * (np.log(1.8e-3) - np.log(2e-4)))),
        weight_decay=float(np.exp(np.log(1e-6) + u[4] * (np.log(1e-3) - np.log(1e-6)))),
        lookback=_choice(REDUCED_LOOKBACK, u[5]),
        scaling_mode=_choice(REDUCED_SCALING, u[6]),
        target_transform=_choice(REDUCED_TARGET, u[7]),
        teacher_forcing_start=(float(0.10 + 0.60 * u[8]) if spec.teacher_forcing else 0.0),
        teacher_forcing_end=0.0,
        tf_zero_epoch=_choice(zero_choices, u[9]),
        free_run_min_epochs=FREE_RUN_MIN_EPOCHS,
        max_ar_weight=float(0.08 + 0.22 * u[10]),
        reference_alpha=float(np.exp(np.log(0.10) + u[11] * (np.log(20.0) - np.log(0.10)))),
        batch_size=GPU_BATCH_SIZE,
    )


class ReducedBayesEI:
    def __init__(self, seed):
        self.rng = np.random.default_rng(seed)
        self.x, self.y, self.keys = [], [], set()

    def suggest(self, candidate_name):
        if len(self.x) < BO_INITIAL_RANDOM:
            return self.rng.random(REDUCED_HPO_DIMENSION), "random_warmup"
        kernel = ConstantKernel(1.0, (1e-2, 1e2)) * Matern(
            length_scale=np.ones(REDUCED_HPO_DIMENSION), nu=2.5
        ) + WhiteKernel(1e-5, (1e-8, 1e-1))
        gp = GaussianProcessRegressor(
            kernel=kernel, normalize_y=True, alpha=1e-6,
            n_restarts_optimizer=1, random_state=HPO_SEED,
        ).fit(np.asarray(self.x), np.asarray(self.y))
        pool = self.rng.random((BO_POOL_SIZE, REDUCED_HPO_DIMENSION))
        mean, std = gp.predict(pool, return_std=True)
        improvement = np.min(self.y) - mean - 0.001
        z = improvement / np.maximum(std, 1e-9)
        acquisition = improvement * norm.cdf(z) + std * norm.pdf(z)
        for index in np.argsort(acquisition)[::-1]:
            key = json.dumps(asdict(decode_reduced_vector(pool[index], candidate_name)), sort_keys=True)
            if key not in self.keys:
                return pool[index], "GP_Matern_expected_improvement"
        return self.rng.random(REDUCED_HPO_DIMENSION), "random_fallback"

    def observe(self, vector, score, config):
        self.x.append(np.asarray(vector, float)); self.y.append(float(score))
        self.keys.add(json.dumps(asdict(config), sort_keys=True))


def run_reduced_hpo(selection_bundles, top_two, scenario):
    trial_rows, promoted_rows, promoted_artifacts = [], [], {}
    for site, bundle in selection_bundles.items():
        train_idx, val_idx = positions(bundle.train), positions(bundle.val_all)
        promoted_artifacts[site] = {}
        for candidate_index, candidate_name in enumerate(top_two[site]):
            optimizer = ReducedBayesEI(HPO_SEED + 1009 * candidate_index + sum(map(ord, site)))
            successful = []
            for trial in range(FINAL_BO_TRIALS):
                vector, acquisition = optimizer.suggest(candidate_name)
                config = decode_reduced_vector(vector, candidate_name)
                started = time.time()
                try:
                    trained = fit_mufasa(
                        bundle, train_idx, val_idx, config, HPO_SEED,
                        max_epochs_override=BO_LOW_EPOCHS,
                        patience_override=BO_LOW_PATIENCE,
                    )
                    prediction = predict_mufasa(trained, bundle, val_idx, mc_passes=1)
                    metrics = temporal_block_metrics(
                        bundle.target[val_idx], prediction, bundle.target_dates[val_idx]
                    )
                    score = metrics["Mean_block_RMSE"]
                    optimizer.observe(vector, score, config)
                    record = {
                        "Scenario": scenario, "Site": site, "Candidate": candidate_name,
                        "Trial": trial, "Acquisition": acquisition,
                        "Parameters": json.dumps(asdict(config), sort_keys=True),
                        "Parameter_count": model_parameter_count(trained),
                        **model_parameter_profile(trained),
                        "Validation_RMSE": metrics["Overall_RMSE"],
                        "Validation_MAE": metrics["Overall_MAE"],
                        "Seconds": time.time() - started, "Status": "ok",
                        "Promoted": False, "Selection_reason": "pending stability promotion",
                        **{key: value for key, value in metrics.items() if key not in {"Overall_RMSE", "Overall_MAE"}},
                    }
                    successful.append((record, config, trained, prediction))
                    trial_rows.append(record)
                except Exception as exc:
                    optimizer.observe(vector, 1e3, config)
                    trial_rows.append({
                        "Scenario": scenario, "Site": site, "Candidate": candidate_name,
                        "Trial": trial, "Acquisition": acquisition,
                        "Parameters": json.dumps(asdict(config), sort_keys=True),
                        "Status": "failed", "Error": f"{type(exc).__name__}: {exc}",
                        "Promoted": False, "Selection_reason": "training failure",
                    })
            if not successful:
                raise RuntimeError(f"{site}/{candidate_name}: every reduced HPO trial failed")
            success_frame = pd.DataFrame([item[0] for item in successful])
            best_mean = success_frame["Mean_block_RMSE"].min()
            near_ids = success_frame[
                success_frame["Mean_block_RMSE"] <= best_mean * (1 + STRUCTURE_NEAR_BEST_TOL)
            ].sort_values(
                ["SD_block_RMSE", "Worst_block_RMSE", "Parameter_count", "Mean_block_RMSE", "Trial"]
            )["Trial"].tolist()
            remaining = success_frame.sort_values(
                ["Mean_block_RMSE", "SD_block_RMSE", "Worst_block_RMSE", "Trial"]
            )["Trial"].tolist()
            promotion_ids =  []
            for trial_id in near_ids + remaining:
                if trial_id not in promotion_ids:
                    promotion_ids.append(int(trial_id))
                if len(promotion_ids) >= BO_PROMOTE:
                    break
            for row in trial_rows:
                if row.get("Scenario") == scenario and row.get("Site") == site and row.get("Candidate") == candidate_name and row.get("Trial") in promotion_ids:
                    row["Promoted"] = True
                    row["Selection_reason"] = "near-best mean -> SD -> worst block -> complexity"
            candidate_promotions = []
            lookup = {int(item[0]["Trial"]): item for item in successful}
            for rank, trial_id in enumerate(promotion_ids, 1):
                _, config, _, _ = lookup[trial_id]
                trained = fit_mufasa(
                    bundle, train_idx, val_idx, config, HPO_SEED + rank,
                    max_epochs_override=FINAL_MAX_EPOCHS,
                    patience_override=FINAL_PATIENCE,
                )
                prediction = predict_mufasa(trained, bundle, val_idx, mc_passes=1)
                metrics = temporal_block_metrics(bundle.target[val_idx], prediction, bundle.target_dates[val_idx])
                row = {
                    "Scenario": scenario, "Site": site, "Candidate": candidate_name,
                    "Promotion_rank": rank, "Source_trial": trial_id,
                    "Parameter_count": model_parameter_count(trained),
                    **model_parameter_profile(trained),
                    "Best_epoch": trained.best_epoch,
                    "Parameters": json.dumps(asdict(config), sort_keys=True), **metrics,
                }
                promoted_rows.append(row)
                candidate_promotions.append((row, config, trained, prediction))
            promoted_artifacts[site][candidate_name] = candidate_promotions
    return pd.DataFrame(trial_rows), pd.DataFrame(promoted_rows), promoted_artifacts


STRICT_HPO_TRIALS, STRICT_HPO_PROMOTIONS, STRICT_HPO_ARTIFACTS = run_reduced_hpo(
    STRICT_SELECTION_BUNDLES, STRICT_TOP_TWO, "strict_history"
)
STRICT_HPO_TRIALS.to_csv(OUTPUT_DIR / "MUFASA_reduced_HPO_trials.csv", index=False)


In [ ]:
# Promotion and Final Candidate Locking
def promote_final_candidate(promoted_table, promoted_artifacts, scenario):
    selections, configs, epochs = [], {}, {}
    for site in promoted_artifacts:
        all_rows = promoted_table[
            promoted_table["Scenario"].eq(scenario) & promoted_table["Site"].eq(site)
        ].copy()
        frame = all_rows[all_rows["Candidate"].eq(CANONICAL_CORE_CANDIDATE)].copy()
        if frame.empty:
            raise RuntimeError(f"{site}/{scenario}: canonical C2 was not promoted.")
        selected, near = stable_near_best(frame, "Promotion_rank")
        matching = [
            item for item in promoted_artifacts[site][CANONICAL_CORE_CANDIDATE]
            if int(item[0]["Promotion_rank"]) == int(selected["Promotion_rank"])
        ]
        if len(matching) != 1:
            raise AssertionError(f"{site}: canonical promoted C2 lookup is ambiguous")
        row, config, trained, prediction = matching[0]
        configs[site] = config
        epochs[site] = max(
            int(trained.best_epoch),
            int(config.tf_zero_epoch + config.free_run_min_epochs)
            if config.teacher_forcing_start > 0 else 1,
        )
        challenger = all_rows[~all_rows["Candidate"].eq(CANONICAL_CORE_CANDIDATE)]
        challenger_best = float(challenger["Mean_block_RMSE"].min()) if len(challenger) else np.nan
        selections.append({
            "Scenario": scenario, "Site": site,
            "Selected_candidate": CANONICAL_CORE_CANDIDATE,
            "Core_identity_policy": CORE_SELECTION_POLICY,
            "Near_best_C2_promotions": "|".join(map(str, near["Promotion_rank"].astype(int))),
            "Mean_block_RMSE": selected["Mean_block_RMSE"],
            "SD_block_RMSE": selected["SD_block_RMSE"],
            "Worst_block_RMSE": selected["Worst_block_RMSE"],
            "Best_challenger_mean_block_RMSE": challenger_best,
            "C2_gap_vs_best_challenger_percent": 100 * (
                float(selected["Mean_block_RMSE"]) / max(challenger_best, 1e-12) - 1
            ) if np.isfinite(challenger_best) else np.nan,
            "Selected_epoch": epochs[site],
            "Parameter_count": selected["Parameter_count"],
            "Active_parameter_count": selected.get("Active_parameter_count", selected["Parameter_count"]),
            "Total_parameter_count": selected.get("Total_parameter_count", selected["Parameter_count"]),
            "Selection_uses_test": False,
            "Canonical_choice_retrospective": CANONICAL_CHOICE_RETROSPECTIVE,
        })
    result = pd.DataFrame(selections)
    assert result["Selected_candidate"].eq(CANONICAL_CORE_CANDIDATE).all()
    return result, configs, epochs


STRICT_PROMOTION_SELECTION, STRICT_SELECTED_CONFIGS, STRICT_FIXED_EPOCHS = promote_final_candidate(
    STRICT_HPO_PROMOTIONS, STRICT_HPO_ARTIFACTS, "strict_history"
)
display(STRICT_PROMOTION_SELECTION)


In [ ]:
# Three-Seed Training
def train_selected_core_seeds(selection_bundles, selected_table, configs, epochs, scenario):
    models, cubes, aux_by_site, seed_rows = {}, {}, {}, []
    for site, bundle in selection_bundles.items():
        train_idx, val_idx = positions(bundle.train), positions(bundle.val_all)
        predictions, auxiliaries, trained_models = [], [], []
        for seed in ACTIVE_FINAL_SEEDS:
            started = time.time()
            trained = fit_mufasa(
                bundle, train_idx, val_idx, configs[site], seed,
                fixed_epochs=epochs[site],
            )
            prediction, auxiliary = predict_mufasa(
                trained, bundle, val_idx, return_aux=True, mc_passes=1
            )
            predictions.append(prediction); auxiliaries.append(auxiliary); trained_models.append(trained)
            seed_rows.append({
                "Scenario": scenario, "Site": site, "Seed": seed,
                "Candidate": selected_table.loc[selected_table["Site"].eq(site), "Selected_candidate"].iloc[0],
                "Fit_end": trained.scalers["fit_end"],
                "Test_rows_in_scaler_fit": trained.scalers["test_rows_in_fit"],
                "Teacher_forcing_validation": 0.0,
                "Seconds": time.time() - started,
                **metric_set(bundle.target[val_idx], prediction),
            })
        models[site] = trained_models
        cubes[site] = np.stack(predictions, axis=-1)
        aux_by_site[site] = auxiliaries
    return models, cubes, aux_by_site, pd.DataFrame(seed_rows)


STRICT_SELECTION_MODELS, STRICT_VAL_SEED_CUBES, STRICT_VAL_AUX, STRICT_CORE_SEEDS = (
    train_selected_core_seeds(
        STRICT_SELECTION_BUNDLES, STRICT_PROMOTION_SELECTION,
        STRICT_SELECTED_CONFIGS, STRICT_FIXED_EPOCHS, "strict_history"
    )
)
STRICT_CORE_SEEDS.to_csv(OUTPUT_DIR / "MUFASA_core_validation_seeds.csv", index=False)


In [ ]:
# Validation-Safe Seed Consensus
def simplex_projection(vector):
    values = np.asarray(vector, float)
    order = np.sort(values)[::-1]
    cumulative = np.cumsum(order)
    rho = np.where(order * np.arange(1, len(values) + 1) > cumulative - 1)[0]
    if len(rho) == 0:
        return np.full(len(values), 1.0 / len(values))
    threshold = (cumulative[rho[-1]] - 1) / (rho[-1] + 1)
    projected = np.maximum(values - threshold, 0.0)
    return projected / max(projected.sum(), 1e-12)


def fit_seed_simplex(x, y, alpha=1.0):
    x, y = np.asarray(x, float), np.asarray(y, float)
    prior = np.full(x.shape[1], 1.0 / x.shape[1])
    gram = x.T @ x + float(alpha) * np.eye(x.shape[1])
    rhs = x.T @ y + float(alpha) * prior
    return simplex_projection(np.linalg.solve(gram, rhs))


def fit_seed_policy(seed_cube, truth, dates, site, scenario):
    n_seed = seed_cube.shape[-1]
    equal_weights = np.full((N_HORIZONS, n_seed), 1.0 / n_seed)
    final_weights = equal_weights.copy(); audits = []
    if n_seed < 2:
        for horizon, hour in enumerate(TARGET_HOURS):
            audits.append({
                "Scenario": scenario, "Site": site, "Horizon": horizon + 1, "Hour": int(hour),
                "Equal_RMSE": metric_set(truth[:, horizon], seed_cube[:, horizon, 0])["RMSE"],
                "Weighted_RMSE": metric_set(truth[:, horizon], seed_cube[:, horizon, 0])["RMSE"],
                "Bootstrap_CI_low": 0.0, "Bootstrap_CI_high": 0.0,
                "Weighted_active": False, "Weights": json.dumps(equal_weights[horizon].tolist()),
                "Decision": "smoke/single-seed exact equal fallback", "Selection_uses_test": False,
            })
        return final_weights, pd.DataFrame(audits)
    dates = pd.DatetimeIndex(dates)
    for horizon, hour in enumerate(TARGET_HOURS):
        oof_equal, oof_weighted, oof_truth, oof_dates = [], [], [], []
        for train_quarters, eval_quarter in [("Q1", "Q2"), ("Q1Q2", "Q3"), ("Q1Q2Q3", "Q4")]:
            eval_start, eval_end = VALIDATION_BLOCKS[eval_quarter]
            eval_mask = (dates >= eval_start) & (dates <= eval_end)
            train_mask = dates < eval_start
            x_train = seed_cube[train_mask, horizon, :]
            y_train = truth[train_mask, horizon]
            weights = fit_seed_simplex(x_train, y_train, alpha=max(1.0, len(y_train) * 0.01))
            oof_equal.append(seed_cube[eval_mask, horizon, :].mean(axis=1))
            oof_weighted.append(seed_cube[eval_mask, horizon, :] @ weights)
            oof_truth.append(truth[eval_mask, horizon]); oof_dates.extend(dates[eval_mask])
        equal = np.concatenate(oof_equal); weighted = np.concatenate(oof_weighted)
        observed = np.concatenate(oof_truth); oof_dates = pd.DatetimeIndex(oof_dates)
        difference = (weighted - observed) ** 2 - (equal - observed) ** 2
        low, high = seven_day_block_ci(difference, oof_dates, seed=SEED + horizon)
        active = bool(high < 0)
        if active:
            final_weights[horizon] = fit_seed_simplex(
                seed_cube[:, horizon, :], truth[:, horizon], alpha=max(1.0, len(truth) * 0.01)
            )
        audits.append({
            "Scenario": scenario, "Site": site, "Horizon": horizon + 1, "Hour": int(hour),
            "Equal_RMSE": metric_set(observed, equal)["RMSE"],
            "Weighted_RMSE": metric_set(observed, weighted)["RMSE"],
            "Bootstrap_CI_low": low, "Bootstrap_CI_high": high,
            "Weighted_active": active,
            "Weights": json.dumps(final_weights[horizon].tolist()),
            "Decision": "activate only if rolling-OOF block CI upper < 0",
            "Selection_uses_test": False,
        })
    return final_weights, pd.DataFrame(audits)


def apply_seed_policy(seed_cube, weights):
    return np.einsum("nhs,hs->nh", np.asarray(seed_cube), np.asarray(weights))


STRICT_SEED_WEIGHTS, seed_audits = {}, []
for site, bundle in STRICT_SELECTION_BUNDLES.items():
    val_idx = positions(bundle.val_all)
    weights, audit = fit_seed_policy(
        STRICT_VAL_SEED_CUBES[site], bundle.target[val_idx], bundle.target_dates[val_idx],
        site, "strict_history",
    )
    STRICT_SEED_WEIGHTS[site] = weights; seed_audits.append(audit)
STRICT_SEED_POLICY_AUDIT = pd.concat(seed_audits, ignore_index=True)
STRICT_SEED_POLICY_AUDIT.to_csv(OUTPUT_DIR / "MUFASA_seed_policy_audit.csv", index=False)


In [ ]:
# Uncertainty Tail Guard
TAIL_GRID_A = (-3.0, -2.0, -1.0)
TAIL_GRID_B = (0.0, 1.0, 2.0, 3.0)
TAIL_GRID_C = (0.0, 1.0, 2.0)
TAIL_GRID_GMAX = (0.15, 0.25, 0.35)


def _robust_z(values, center=None, scale=None):
    values = np.asarray(values, float)
    center = float(np.median(values)) if center is None else float(center)
    scale = float(np.quantile(values, 0.75) - np.quantile(values, 0.25)) if scale is None else float(scale)
    scale = max(scale, 1e-6)
    return (values - center) / scale, center, scale


def _fit_tail_grid(core, robust, uncertainty, disagreement, truth):
    zu, u_center, u_scale = _robust_z(uncertainty)
    zd, d_center, d_scale = _robust_z(disagreement)
    best = None
    for a in TAIL_GRID_A:
        for b in TAIL_GRID_B:
            for c in TAIL_GRID_C:
                for gmax in TAIL_GRID_GMAX:
                    gate = gmax / (1.0 + np.exp(-(a + b * zu + c * zd)))
                    prediction = (1 - gate) * core + gate * robust
                    mse = float(np.mean((prediction - truth) ** 2))
                    candidate = (mse, a, b, c, gmax)
                    if best is None or candidate < best:
                        best = candidate
    return {
        "a": best[1], "b": best[2], "c": best[3], "gmax": best[4],
        "u_center": u_center, "u_scale": u_scale,
        "d_center": d_center, "d_scale": d_scale,
    }


def _apply_tail_horizon(policy, core, robust, uncertainty):
    disagreement = np.abs(core - robust)
    zu = (uncertainty - policy["u_center"]) / policy["u_scale"]
    zd = (disagreement - policy["d_center"]) / policy["d_scale"]
    gate = policy["gmax"] / (1.0 + np.exp(-(
        policy["a"] + policy["b"] * zu + policy["c"] * zd
    )))
    if not policy.get("active", False):
        gate = np.zeros_like(gate)
    return (1 - gate) * core + gate * robust, gate


def fit_tail_guard_policy(core, seed_std, ridge, prior, truth, dates, site, scenario, eligible):
    dates = pd.DatetimeIndex(dates); policies, audits = [], []
    for horizon, hour in enumerate(TARGET_HOURS):
        if not eligible:
            policy = {"active": False, "robust_source": "Ridge-Residual", "a": 0.0, "b": 0.0, "c": 0.0, "gmax": 0.0,
                      "u_center": 0.0, "u_scale": 1.0, "d_center": 0.0, "d_scale": 1.0}
            policies.append(policy)
            audits.append({"Scenario": scenario, "Site": site, "Horizon": horizon + 1, "Hour": int(hour),
                           "Robust_source": policy["robust_source"], "Active": False, "Gate_mean": 0.0, "Gate_max": 0.0,
                           "Core_OOF_RMSE": np.nan, "Guard_OOF_RMSE": np.nan, "Loss_diff_mean": 0.0,
                           "Block_CI_lower": 0.0, "Block_CI_upper": 0.0, "Decision": "candidate does not include TailGuard",
                           "Selection_uses_test": False})
            continue
        oof_core, oof_guard, oof_truth, oof_dates, oof_gate, fold_wins = [], [], [], [], [], 0
        for eval_quarter in ("Q2", "Q3", "Q4"):
            start, end = VALIDATION_BLOCKS[eval_quarter]
            train_mask, eval_mask = dates < start, (dates >= start) & (dates <= end)
            ridge_train = np.mean((ridge[train_mask, horizon] - truth[train_mask, horizon]) ** 2)
            solar_ref_train = np.mean((prior[train_mask, horizon] - truth[train_mask, horizon]) ** 2)
            robust_name = "Ridge-Residual" if ridge_train <= solar_ref_train else "Solar-Geometry-Reference"
            robust = ridge[:, horizon] if robust_name == "Ridge-Residual" else prior[:, horizon]
            fitted = _fit_tail_grid(
                core[train_mask, horizon], robust[train_mask], seed_std[train_mask, horizon],
                np.abs(core[train_mask, horizon] - robust[train_mask]), truth[train_mask, horizon],
            )
            fitted["active"] = True
            guarded, gate = _apply_tail_horizon(
                fitted, core[eval_mask, horizon], robust[eval_mask], seed_std[eval_mask, horizon]
            )
            core_fold = core[eval_mask, horizon]
            fold_wins += int(np.sqrt(np.mean((guarded - truth[eval_mask, horizon]) ** 2)) <= np.sqrt(np.mean((core_fold - truth[eval_mask, horizon]) ** 2)))
            oof_core.append(core_fold); oof_guard.append(guarded); oof_truth.append(truth[eval_mask, horizon])
            oof_dates.extend(dates[eval_mask]); oof_gate.append(gate)
        oof_core = np.concatenate(oof_core); oof_guard = np.concatenate(oof_guard); observed = np.concatenate(oof_truth)
        difference = (oof_guard - observed) ** 2 - (oof_core - observed) ** 2
        low, high = seven_day_block_ci(difference, pd.DatetimeIndex(oof_dates), seed=SEED + 100 + horizon)
        active = bool(high < 0 and fold_wins >= 2)
        ridge_all = np.mean((ridge[:, horizon] - truth[:, horizon]) ** 2)
        solar_ref_all = np.mean((prior[:, horizon] - truth[:, horizon]) ** 2)
        robust_name = "Ridge-Residual" if ridge_all <= solar_ref_all else "Solar-Geometry-Reference"
        robust_all = ridge[:, horizon] if robust_name == "Ridge-Residual" else prior[:, horizon]
        policy = _fit_tail_grid(core[:, horizon], robust_all, seed_std[:, horizon], np.abs(core[:, horizon] - robust_all), truth[:, horizon])
        policy.update({"active": active, "robust_source": robust_name})
        if not active:
            policy["gmax"] = 0.0
        policies.append(policy)
        gates = np.concatenate(oof_gate)
        audits.append({
            "Scenario": scenario, "Site": site, "Horizon": horizon + 1, "Hour": int(hour),
            "Robust_source": robust_name, "Active": active,
            "Gate_mean": float(gates.mean()), "Gate_max": float(gates.max()),
            "Core_OOF_RMSE": metric_set(observed, oof_core)["RMSE"],
            "Guard_OOF_RMSE": metric_set(observed, oof_guard)["RMSE"],
            "Loss_diff_mean": float(np.mean(difference)),
            "Block_CI_lower": low, "Block_CI_upper": high,
            "Decision": "active iff CI upper<0 and >=2/3 rolling folds win",
            "Selection_uses_test": False,
        })
    return policies, pd.DataFrame(audits)


def apply_tail_guard(policy, core, seed_std, ridge, prior):
    output, gates = np.asarray(core).copy(), np.zeros_like(core, float)
    for horizon, item in enumerate(policy):
        robust = ridge[:, horizon] if item["robust_source"] == "Ridge-Residual" else prior[:, horizon]
        output[:, horizon], gates[:, horizon] = _apply_tail_horizon(
            item, core[:, horizon], robust, seed_std[:, horizon]
        )
    return output, gates


In [ ]:
# Regularized Ridge Expert and Solar-Geometry Reference
@dataclass(frozen=True)
class RidgeResidualConfig:
    alpha: float = 10.0
    reference_alpha: float = 1.0
    feature_mode: str = "balanced_no_annual"
    target_transform: str = "identity"


@dataclass
class RidgeResidualExpert:
    config: RidgeResidualConfig
    scaler: Any
    model: Any
    reference_model: SolarReferenceModel
    feature_columns: np.ndarray
    feature_families: np.ndarray
    fit_start: str
    fit_end: str
    test_rows_in_fit: int
    scaler_digest: str


def _ridge_feature_mask(bundle, mode):
    families = np.asarray(bundle.tabular_families).astype(str)
    if mode == "compact_no_annual":
        columns = _engineered_feature_mask(bundle, "compact_no_annual")
        mask = np.zeros(len(families), dtype=bool)
        mask[columns] = True
    elif mode == "balanced_no_annual":
        columns = _engineered_feature_mask(bundle, "no_annual")
        mask = np.zeros(len(families), dtype=bool)
        mask[columns] = True
    elif mode in {"balanced_with_annual", "full"}:
        mask = np.ones(len(families), dtype=bool)
    else:
        raise ValueError(f"Unknown Ridge feature_mode={mode!r}")
    if not np.any(mask):
        raise RuntimeError("Ridge residual expert selected no columns.")
    return mask

def decode_ridge_vector(vector):
    u = np.clip(np.asarray(vector, float), 0.0, 1.0 - 1e-12)
    return RidgeResidualConfig(
        alpha=float(np.exp(np.log(0.05) + u[0] * (np.log(10000.0) - np.log(0.05)))),
        reference_alpha=float(np.exp(np.log(0.03) + u[1] * (np.log(50.0) - np.log(0.03)))),
        feature_mode=_choice(
            ("compact_no_annual", "balanced_no_annual", "balanced_with_annual", "full"), u[2]
        ),
        target_transform=_choice(("identity", "log1p", "direct_state", "direct_solar"), u[3]),
    )


class RidgeBayesEI:
    def __init__(self, seed, trials):
        self.dimension = 4
        self.trials = int(trials)
        self.rng = np.random.default_rng(seed)
        self.x, self.y, self.keys = [], [], set()
        # [alpha, prior_alpha, feature_mode, target_mode] unit vectors
        
        p1 = 0.472672326862543
        self.initial = [
            np.array([0.901362802001115, p1, 0.10, 0.62]),  # compact, direct state, alpha=3000
            np.array([0.811357415291376, p1, 0.10, 0.62]),  # compact, direct state, alpha=1000
            np.array([0.811357415291376, p1, 0.90, 0.62]),  # full, direct state, alpha=1000
            np.array([1.000000000000000, p1, 0.90, 0.10]),  # full, prior residual, alpha=10000
            np.array([1.000000000000000, p1, 0.10, 0.10]),  # compact, prior residual, alpha=10000
            np.array([0.901362802001115, p1, 0.10, 0.90]),  # compact, direct solar
            np.array([0.811357415291376, p1, 0.90, 0.90]),  # full, direct solar
            np.array([0.901362802001115, p1, 0.35, 0.62]),  # no-annual, direct state
            np.array([0.622714830582752, p1, 0.10, 0.62]),  # compact, alpha=100
            np.array([0.620000000000000, 0.60, 0.55, 0.75]),
        ]

    def suggest(self):
        if len(self.x) < min(len(self.initial), self.trials):
            return self.initial[len(self.x)].copy(), "initial"
        if len(self.x) < min(5, self.trials):
            return self.rng.random(self.dimension), "random_warmup"
        gp = GaussianProcessRegressor(
            kernel=ConstantKernel(1.0) * Matern(length_scale=np.ones(self.dimension), nu=2.5)
            + WhiteKernel(1e-5),
            normalize_y=True, alpha=1e-6, random_state=SEED,
        ).fit(np.asarray(self.x), np.asarray(self.y))
        pool = self.rng.random((BO_POOL_SIZE, self.dimension))
        mean, std = gp.predict(pool, return_std=True)
        improvement = np.min(self.y) - mean - 0.001
        z = improvement / np.maximum(std, 1e-9)
        acquisition = improvement * norm.cdf(z) + std * norm.pdf(z)
        for index in np.argsort(acquisition)[::-1]:
            config = decode_ridge_vector(pool[index])
            key = json.dumps(asdict(config), sort_keys=True)
            if key not in self.keys:
                return pool[index], "GP_Matern_expected_improvement"
        return self.rng.random(self.dimension), "random_fallback"

    def observe(self, vector, score, config):
        self.x.append(np.asarray(vector, float))
        self.y.append(float(score))
        self.keys.add(json.dumps(asdict(config), sort_keys=True))


def fit_ridge_residual(bundle, fit_idx, config):
    fit_idx = np.asarray(fit_idx, int)
    if len(fit_idx) == 0 or np.any(bundle.test[fit_idx]):
        raise AssertionError("Ridge residual fit boundary is empty or contains test rows.")
    mask = _ridge_feature_mask(bundle, config.feature_mode)
    columns = np.where(mask)[0]
    reference_model = fit_solar_reference(bundle, fit_idx, config.reference_alpha, enabled=True)
    solar_ref_state = predict_solar_reference(bundle, reference_model)
    if config.target_transform == "log1p":
        target = np.log1p(np.clip(bundle.target_state, 0.0, None)) - np.log1p(solar_ref_state)
    elif config.target_transform == "direct_state":
        target = bundle.target_state
    elif config.target_transform == "direct_solar":
        target = bundle.target
    else:
        target = bundle.target_state - solar_ref_state
    scaler = StandardScaler().fit(bundle.tabular[np.ix_(fit_idx, columns)])
    design = scaler.transform(bundle.tabular[np.ix_(fit_idx, columns)])
    model = Ridge(
        alpha=float(config.alpha), solver="lsqr", tol=1e-5, max_iter=10000,
    ).fit(design, target[fit_idx])
    fit_dates = bundle.target_dates[fit_idx]
    digest_arrays = [
        scaler.mean_, scaler.scale_, reference_model.coefficients,
        reference_model.intercepts, reference_model.monthly_state,
    ]
    digest = hashlib.sha256(
        b"".join(np.asarray(value, np.float32).tobytes() for value in digest_arrays)
    ).hexdigest()
    return RidgeResidualExpert(
        config, scaler, model, reference_model, columns,
        np.asarray(bundle.tabular_families)[columns],
        str(fit_dates.min().date()), str(fit_dates.max().date()),
        int(np.sum(bundle.test[fit_idx])), digest,
    )


def predict_ridge_residual(expert, bundle, query_idx):
    query_idx = np.asarray(query_idx, int)
    design = expert.scaler.transform(
        bundle.tabular[np.ix_(query_idx, expert.feature_columns)]
    )
    residual = expert.model.predict(design)
    prior = predict_solar_reference(bundle, expert.reference_model)[query_idx]
    if expert.config.target_transform == "direct_solar":
        return np.clip(residual, 0.0, None).astype(np.float32)
    if expert.config.target_transform == "direct_state":
        state = residual
    elif expert.config.target_transform == "log1p":
        state = np.expm1(np.log1p(np.clip(prior, 0.0, None)) + residual)
    else:
        state = prior + residual
    return np.clip(state, 0.0, 3.0).astype(np.float32) * bundle.potential[query_idx]


def predict_anchor(expert, bundle, query_idx):
    query_idx = np.asarray(query_idx, int)
    state = predict_solar_reference(bundle, expert.reference_model)[query_idx]
    return np.clip(state, 0.0, 3.0) * bundle.potential[query_idx]


def run_ridge_experts(selection_bundles, scenario):
    experts, configs, ridge_predictions, solar_ref_predictions, rows = {}, {}, {}, {}, []
    trials = FINAL_BO_TRIALS
    for site, bundle in selection_bundles.items():
        train_idx, val_idx = positions(bundle.train), positions(bundle.val_all)
        optimizer = RidgeBayesEI(SEED + 31000 + sum(map(ord, site)), trials)
        candidates = []
        for trial in range(trials):
            vector, acquisition = optimizer.suggest()
            config = decode_ridge_vector(vector)
            expert = fit_ridge_residual(bundle, train_idx, config)
            prediction = predict_ridge_residual(expert, bundle, val_idx)
            metrics = temporal_block_metrics(bundle.target[val_idx], prediction, bundle.target_dates[val_idx])
            optimizer.observe(vector, metrics["Mean_block_RMSE"], config)
            row = {"Scenario": scenario, "Site": site, "Trial": trial, "Acquisition": acquisition,
                   "Parameter_count": int(np.asarray(expert.model.coef_).size), **asdict(config), **metrics}
            rows.append(row); candidates.append((row, config, expert, prediction))
        frame = pd.DataFrame([item[0] for item in candidates])
        selected, _ = stable_near_best(frame, "Trial")
        match = [item for item in candidates if int(item[0]["Trial"]) == int(selected["Trial"])][0]
        _, config, expert, prediction = match
        experts[site], configs[site], ridge_predictions[site] = expert, config, prediction
        solar_ref_predictions[site] = predict_anchor(expert, bundle, val_idx)
    return experts, configs, ridge_predictions, solar_ref_predictions, pd.DataFrame(rows)


STRICT_RIDGE_EXPERTS, STRICT_RIDGE_CONFIGS, STRICT_RIDGE_VAL, STRICT_SOLAR_REFERENCE_VAL, STRICT_RIDGE_HPO = (
    run_ridge_experts(STRICT_SELECTION_BUNDLES, "strict_history")
)
STRICT_RIDGE_HPO.to_csv(OUTPUT_DIR / "MUFASA_ridge_HPO_2019.csv", index=False)


In [ ]:
# Development-Only Aggregation Policy
def source_stability_table(source_predictions, truth, dates, site, scenario):
    rows = []
    for source, prediction in source_predictions.items():
        rows.append({"Scenario": scenario, "Site": site, "Source": source,
                     "Parameter_count": 0, **temporal_block_metrics(truth, prediction, dates)})
    return pd.DataFrame(rows)


def subset_source_stability_table(source_predictions, truth, dates, site, scenario):
    """Public-release implementation note."""
    dates = pd.DatetimeIndex(dates); rows = []
    for source, prediction in source_predictions.items():
        monthly = []
        for month in sorted(dates.month.unique()):
            mask = dates.month == month
            monthly.append(metric_set(truth[mask], prediction[mask])["RMSE"])
        overall = metric_set(truth, prediction)
        rows.append({
            "Scenario": scenario, "Site": site, "Source": source, "Parameter_count": 0,
            "Mean_block_RMSE": overall["RMSE"],
            "SD_block_RMSE": float(np.std(monthly, ddof=1)) if len(monthly) > 1 else 0.0,
            "Worst_block_RMSE": float(np.max(monthly)),
        })
    return pd.DataFrame(rows)


def fit_simplex_fusion(source_predictions, truth, source_names, winner_name, alpha=20.0):
    x = np.stack([source_predictions[name] for name in source_names], axis=-1).reshape(-1, len(source_names))
    y = np.asarray(truth).reshape(-1)
    prior = np.zeros(len(source_names)); prior[source_names.index(winner_name)] = 1.0
    gram = x.T @ x + float(alpha) * np.eye(len(source_names))
    rhs = x.T @ y + float(alpha) * prior
    return simplex_projection(np.linalg.solve(gram, rhs))


def apply_source_weights(source_predictions, source_names, weights):
    cube = np.stack([source_predictions[name] for name in source_names], axis=-1)
    return np.einsum("nhs,s->nh", cube, np.asarray(weights))


def fit_winner_or_fusion(source_predictions, truth, dates, site, scenario):
    source_names = list(source_predictions)
    dates = pd.DatetimeIndex(dates)
    full_table = source_stability_table(source_predictions, truth, dates, site, scenario)
    full_winner, near = stable_near_best(full_table, "Source")
    stable_winner = full_winner["Source"]
    fold_rows, oof_winner, oof_fusion, oof_truth, oof_dates = [], [], [], [], []
    fold_wins = 0
    for fold, eval_quarter in enumerate(("Q2", "Q3", "Q4"), 1):
        start, end = VALIDATION_BLOCKS[eval_quarter]
        train_mask, eval_mask = dates < start, (dates >= start) & (dates <= end)
        train_sources = {name: values[train_mask] for name, values in source_predictions.items()}
        train_table = subset_source_stability_table(
            train_sources, truth[train_mask], dates[train_mask], site, scenario
        )
        fold_winner, _ = stable_near_best(train_table, "Source")
        winner_name = fold_winner["Source"]
        weights = fit_simplex_fusion(train_sources, truth[train_mask], source_names, winner_name)
        eval_sources = {name: values[eval_mask] for name, values in source_predictions.items()}
        winner_prediction = eval_sources[winner_name]
        fusion_prediction = apply_source_weights(eval_sources, source_names, weights)
        winner_rmse = metric_set(truth[eval_mask], winner_prediction)["RMSE"]
        fusion_rmse = metric_set(truth[eval_mask], fusion_prediction)["RMSE"]
        fold_wins += int(fusion_rmse <= winner_rmse)
        fold_rows.append({
            "Scenario": scenario, "Site": site, "Fold": f"F{fold}:{eval_quarter}",
            "Winner_source": winner_name, "Winner_RMSE": winner_rmse,
            "Fusion_RMSE": fusion_rmse,
            "Loss_diff": float(np.mean(daily_squared_loss(truth[eval_mask], fusion_prediction) - daily_squared_loss(truth[eval_mask], winner_prediction))),
            "Weights": json.dumps(dict(zip(source_names, weights))),
            "Selection_uses_test": False,
        })
        oof_winner.append(winner_prediction); oof_fusion.append(fusion_prediction)
        oof_truth.append(truth[eval_mask]); oof_dates.extend(dates[eval_mask])
    winner_oof, fusion_oof, truth_oof = map(np.concatenate, (oof_winner, oof_fusion, oof_truth))
    difference = daily_squared_loss(truth_oof, fusion_oof) - daily_squared_loss(truth_oof, winner_oof)
    low, high = seven_day_block_ci(difference, pd.DatetimeIndex(oof_dates), seed=SEED + 700 + sum(map(ord, site)))
    active = bool(high < 0 and fold_wins >= 2)
    if active:
        weights = fit_simplex_fusion(source_predictions, truth, source_names, stable_winner)
        mode = "MUFASA"
    else:
        weights = np.zeros(len(source_names)); weights[source_names.index(stable_winner)] = 1.0
        mode = "StableWinner"
    for row in fold_rows:
        row.update({"Bootstrap_CI_low": low, "Bootstrap_CI_high": high,
                    "Fold_wins": fold_wins, "Fusion_active_final": active,
                    "Final_source_mode": mode})
    policy = {"mode": mode, "source_names": source_names, "weights": weights,
              "winner": stable_winner, "bootstrap_ci": (low, high), "fold_wins": fold_wins}
    return policy, full_table, pd.DataFrame(fold_rows)


def fit_postprocessing(selection_bundles, selected_table, seed_cubes, seed_weights,
                       ridge_predictions, solar_ref_predictions, scenario):
    tail_policies, source_policies, final_val_predictions = {}, {}, {}
    tail_audits, source_tables, fusion_audits = [], [], []
    for site, bundle in selection_bundles.items():
        val_idx = positions(bundle.val_all); dates = bundle.target_dates[val_idx]; truth = bundle.target[val_idx]
        cube = seed_cubes[site]
        core = apply_seed_policy(cube, seed_weights[site])
        seed_std = cube.std(axis=-1, ddof=1) if cube.shape[-1] > 1 else np.zeros_like(core)
        candidate_name = selected_table.loc[selected_table["Site"].eq(site), "Selected_candidate"].iloc[0]
        eligible = CANDIDATE_SPECS[candidate_name].tail_guard_eligible
        tail_policy, tail_audit = fit_tail_guard_policy(
            core, seed_std, ridge_predictions[site], solar_ref_predictions[site], truth, dates,
            site, scenario, eligible,
        )
        guarded_core, gates = apply_tail_guard(
            tail_policy, core, seed_std, ridge_predictions[site], solar_ref_predictions[site]
        )
        sources = {"MUFASA-Core": guarded_core, "Ridge-Residual": ridge_predictions[site],
                   "Solar-Geometry-Reference": solar_ref_predictions[site]}
        source_policy, source_table, fusion_audit = fit_winner_or_fusion(
            sources, truth, dates, site, scenario
        )
        final_val_predictions[site] = apply_source_weights(
            sources, source_policy["source_names"], source_policy["weights"]
        )
        tail_policies[site], source_policies[site] = tail_policy, source_policy
        tail_audits.append(tail_audit); source_tables.append(source_table); fusion_audits.append(fusion_audit)
    return (
        tail_policies, source_policies, final_val_predictions,
        pd.concat(tail_audits, ignore_index=True), pd.concat(source_tables, ignore_index=True),
        pd.concat(fusion_audits, ignore_index=True),
    )


(STRICT_TAIL_POLICIES, STRICT_SOURCE_POLICIES, STRICT_FINAL_VAL,
 STRICT_TAIL_AUDIT, STRICT_SOURCE_STABILITY, STRICT_FUSION_AUDIT) = fit_postprocessing(
    STRICT_SELECTION_BUNDLES, STRICT_PROMOTION_SELECTION, STRICT_VAL_SEED_CUBES,
    STRICT_SEED_WEIGHTS, STRICT_RIDGE_VAL, STRICT_SOLAR_REFERENCE_VAL, "strict_history"
)
STRICT_TAIL_AUDIT.to_csv(OUTPUT_DIR / "MUFASA_tail_guard_stability_audit.csv", index=False)
STRICT_SOURCE_STABILITY.to_csv(OUTPUT_DIR / "MUFASA_source_2019_stability.csv", index=False)
STRICT_FUSION_AUDIT.to_csv(OUTPUT_DIR / "MUFASA_winner_or_fusion_audit.csv", index=False)


## 5. Locked refit and 2020 evaluation

Refit after all development choices are frozen and evaluate the locked 2020 test year once. Strict-history and oracle-weather results remain separate.

**Run note.** Execute the cells in this section in order. Objects created here are consumed by later sections; the notebook intentionally avoids hidden state restoration from unpublished artifacts.


In [ ]:
# Final Refit and Locked 2020 Inference
def final_refit_scenario(refit_bundles, selected_table, configs, epochs, seed_weights,
                         tail_policies, source_policies, ridge_configs, scenario):
    results, final_models, final_ridge, config_rows = {}, {}, {}, []
    for site, bundle in refit_bundles.items():
        fit_idx, test_idx = positions(bundle.train_val), positions(bundle.test)
        if np.any(bundle.test[fit_idx]):
            raise AssertionError("Test rows entered final refit")
        seed_predictions, seed_auxiliaries, models = [], [], []
        started = time.time()
        for seed in ACTIVE_FINAL_SEEDS:
            trained = fit_mufasa(bundle, fit_idx, np.array([], int), configs[site], seed,
                                  fixed_epochs=epochs[site])
            prediction, auxiliary = predict_mufasa(
                trained, bundle, test_idx, return_aux=True, mc_passes=1
            )
            seed_predictions.append(prediction); seed_auxiliaries.append(auxiliary)
            models.append(trained)
            assert trained.scalers["test_rows_in_fit"] == 0
        cube = np.stack(seed_predictions, axis=-1)
        core = apply_seed_policy(cube, seed_weights[site])
        seed_std = cube.std(axis=-1, ddof=1) if cube.shape[-1] > 1 else np.zeros_like(core)
        ridge = fit_ridge_residual(bundle, fit_idx, ridge_configs[site])
        ridge_prediction = predict_ridge_residual(ridge, bundle, test_idx)
        solar_ref_prediction = predict_anchor(ridge, bundle, test_idx)
        guarded_core, tail_gate = apply_tail_guard(
            tail_policies[site], core, seed_std, ridge_prediction, solar_ref_prediction
        )
        sources = {"MUFASA-Core": guarded_core, "Ridge-Residual": ridge_prediction,
                   "Solar-Geometry-Reference": solar_ref_prediction}
        policy = source_policies[site]
        final_prediction = apply_source_weights(sources, policy["source_names"], policy["weights"])
        selected_candidate = selected_table.loc[selected_table["Site"].eq(site), "Selected_candidate"].iloc[0]
        results[site] = {
            "prediction": final_prediction, "core": guarded_core, "core_before_tail": core,
            "ridge": ridge_prediction, "prior": solar_ref_prediction, "seed_cube": cube,
            "seed_std": seed_std, "tail_gate": tail_gate, "truth": bundle.target[test_idx],
            "dates": bundle.target_dates[test_idx], "test_idx": test_idx,
            "regime_probabilities": np.mean(np.stack([
                item.get("regime_probabilities", np.full((len(test_idx), len(REGIME_NAMES)), 1 / len(REGIME_NAMES)))
                for item in seed_auxiliaries
            ]), axis=0),
            "bigru_gate": np.mean(np.stack([
                item.get("bigru_gate", np.zeros((len(test_idx), N_HORIZONS, 1)))
                for item in seed_auxiliaries
            ]), axis=0),
            "source_weights": np.mean(np.stack([
                item.get("source_weights", np.zeros((len(test_idx), N_HORIZONS, 5)))
                for item in seed_auxiliaries
            ]), axis=0),
        }
        final_models[site], final_ridge[site] = models, ridge
        spec, config = CANDIDATE_SPECS[selected_candidate], configs[site]
        config_rows.append({
            "Scenario": scenario, "Site": site, "Selected_candidate": selected_candidate,
            "use_bigru": spec.use_bigru, "use_tail_weighting": spec.use_tail_weighting,
            "use_revin": spec.use_revin, "use_nlinear": spec.use_nlinear,
            "use_autoregressive": spec.use_autoregressive, "lookback": config.lookback,
            "d_model": config.d_model, "n_layers": config.n_layers,
            "scaling_mode": config.scaling_mode, "target_transform": config.target_transform,
            "tf_start": config.teacher_forcing_start, "tf_zero_epoch": config.tf_zero_epoch,
            "prior_alpha": config.reference_alpha,
            "seed_policy": "bootstrap-weighted" if np.any(np.abs(seed_weights[site] - 1 / seed_weights[site].shape[1]) > 1e-8) else "equal",
            "tail_guard_active": bool(any(item["active"] for item in tail_policies[site])),
            "source_mode": policy["mode"], "winner_or_fusion": policy["mode"],
            "selected_sources": "|".join(name for name, weight in zip(policy["source_names"], policy["weights"]) if weight > 1e-8),
            "parameter_count": max(model_parameter_count(model) for model in models),
            "Selection_uses_test": False, "Configuration_locked_before_test": True,
            "Training_seconds": time.time() - started,
        })
    return results, final_models, final_ridge, pd.DataFrame(config_rows)


STRICT_RESULTS, STRICT_FINAL_MODELS, STRICT_FINAL_RIDGE, STRICT_FINAL_CONFIG = final_refit_scenario(
    STRICT_REFIT_BUNDLES, STRICT_PROMOTION_SELECTION, STRICT_SELECTED_CONFIGS,
    STRICT_FIXED_EPOCHS, STRICT_SEED_WEIGHTS, STRICT_TAIL_POLICIES,
    STRICT_SOURCE_POLICIES, STRICT_RIDGE_CONFIGS, "strict_history"
)
STRICT_FINAL_CONFIG.to_csv(OUTPUT_DIR / "MUFASA_final_selected_configuration.csv", index=False)
display(STRICT_FINAL_CONFIG)


In [ ]:
# Strict-History Evaluation
def persistence_prediction(
    bundle: SiteBundle,
    indices: np.ndarray,
    lag_days: int,
) -> np.ndarray:
    origins = bundle.origin_indices[np.asarray(indices)]
    return bundle.raw.solar[origins - lag_days]


def monthly_climatology_prediction(
    bundle: SiteBundle,
    fit_idx: np.ndarray,
    predict_idx: np.ndarray,
) -> np.ndarray:
    fit_dates = bundle.target_dates[np.asarray(fit_idx)]
    fit_targets = bundle.target[np.asarray(fit_idx)]
    lookup = {
        month: fit_targets[fit_dates.month == month].mean(axis=0)
        for month in range(1, 13)
    }
    return np.vstack(
        [
            lookup[date.month]
            for date in bundle.target_dates[np.asarray(predict_idx)]
        ]
    )


def initial_model_predictions(results, refit_bundles):
    all_predictions = {}
    for site, result in results.items():
        bundle = refit_bundles[site]; fit_idx = positions(bundle.train_val); test_idx = positions(bundle.test)
        all_predictions[site] = {
            "MUFASA": result["prediction"],
            "MUFASA-Core": result["core"],
            "Ridge-Residual": result["ridge"],
            "Solar-Geometry-Reference": result["prior"],
            "Persistence-1d": persistence_prediction(bundle, test_idx, 1),
            "Persistence-7d": persistence_prediction(bundle, test_idx, 7),
            "Seasonal climatology": monthly_climatology_prediction(bundle, fit_idx, test_idx),
        }
    return all_predictions


STRICT_MODEL_PREDICTIONS = initial_model_predictions(STRICT_RESULTS, STRICT_REFIT_BUNDLES)
STRICT_LOCKED_METRICS = pd.DataFrame([
    {"Scenario": "strict_history", "Site": site, "Model": model, **metric_set(STRICT_RESULTS[site]["truth"], prediction)}
    for site, models in STRICT_MODEL_PREDICTIONS.items() for model, prediction in models.items()
])
display(STRICT_LOCKED_METRICS[STRICT_LOCKED_METRICS["Model"].isin(["MUFASA", "MUFASA-Core", "Ridge-Residual", "Solar-Geometry-Reference"])].round(5))


In [ ]:
# Oracle-Weather Evaluation
def run_complete_scenario(selection_bundles, refit_bundles, scenario):
    screen, reason, screen_artifacts, top_two = run_candidate_screen(selection_bundles, scenario)
    hpo_trials, hpo_promotions, hpo_artifacts = run_reduced_hpo(selection_bundles, top_two, scenario)
    selected, configs, epochs = promote_final_candidate(hpo_promotions, hpo_artifacts, scenario)
    selection_models, val_cubes, val_aux, core_seed_rows = train_selected_core_seeds(
        selection_bundles, selected, configs, epochs, scenario
    )
    seed_weights, seed_frames = {}, []
    for site, bundle in selection_bundles.items():
        val_idx = positions(bundle.val_all)
        weights, audit = fit_seed_policy(
            val_cubes[site], bundle.target[val_idx], bundle.target_dates[val_idx], site, scenario
        )
        seed_weights[site] = weights; seed_frames.append(audit)
    ridge_experts, ridge_configs, ridge_val, solar_ref_val, ridge_hpo = run_ridge_experts(
        selection_bundles, scenario
    )
    tail, source, final_val, tail_audit, source_stability, fusion_audit = fit_postprocessing(
        selection_bundles, selected, val_cubes, seed_weights, ridge_val, solar_ref_val, scenario
    )
    results, final_models, final_ridge, final_config = final_refit_scenario(
        refit_bundles, selected, configs, epochs, seed_weights, tail, source,
        ridge_configs, scenario,
    )
    return {
        "screen": screen, "reason": reason, "top_two": top_two,
        "hpo_trials": hpo_trials, "hpo_promotions": hpo_promotions,
        "selected": selected, "configs": configs, "epochs": epochs,
        "selection_models": selection_models, "val_cubes": val_cubes,
        "val_aux": val_aux, "core_seed_rows": core_seed_rows,
        "seed_weights": seed_weights, "seed_audit": pd.concat(seed_frames, ignore_index=True),
        "ridge_experts": ridge_experts, "ridge_configs": ridge_configs,
        "ridge_val": ridge_val, "prior_val": solar_ref_val, "ridge_hpo": ridge_hpo,
        "tail": tail, "source": source, "final_val": final_val,
        "tail_audit": tail_audit, "source_stability": source_stability,
        "fusion_audit": fusion_audit, "results": results,
        "final_models": final_models, "final_ridge": final_ridge,
        "final_config": final_config,
    }


if RUN_ORACLE:
    ORACLE_RUN = run_complete_scenario(
        ORACLE_SELECTION_BUNDLES, ORACLE_REFIT_BUNDLES, "oracle_weather"
    )
    ORACLE_MODEL_PREDICTIONS = initial_model_predictions(
        ORACLE_RUN["results"], ORACLE_REFIT_BUNDLES
    )
    ORACLE_RUN["screen"].to_csv(OUTPUT_DIR / "MUFASA_oracle_candidate_screen_2019.csv", index=False)
    ORACLE_RUN["seed_audit"].to_csv(OUTPUT_DIR / "MUFASA_oracle_seed_policy_audit.csv", index=False)
    ORACLE_RUN["tail_audit"].to_csv(OUTPUT_DIR / "MUFASA_oracle_tail_guard_stability_audit.csv", index=False)
    ORACLE_RUN["fusion_audit"].to_csv(OUTPUT_DIR / "MUFASA_oracle_winner_or_fusion_audit.csv", index=False)
else:
    ORACLE_RUN, ORACLE_MODEL_PREDICTIONS = None, {}

ARCHIVED_STATUS = pd.DataFrame([{
    "Scenario": "archived_forecast", "Available": ARCHIVED_AVAILABLE,
    "Executed": bool(RUN_ARCHIVED and ARCHIVED_AVAILABLE),
    "Reason": "executed only when issue-time archived NWP columns physically exist" if ARCHIVED_AVAILABLE else "no archived forecast columns; skipped without renaming observations",
}])
ARCHIVED_STATUS.to_csv(OUTPUT_DIR / "MUFASA_archived_forecast_status.csv", index=False)


## 6. Matched-budget benchmark suite

Retrain the 18 comparator architectures on the same data boundary and matched computational budget, then audit implementation provenance.

**Run note.** Execute the cells in this section in order. Objects created here are consumed by later sections; the notebook intentionally avoids hidden state restoration from unpublished artifacts.


In [ ]:
# Benchmark Suite and Matched-Budget Optimization — part 1/5
DEEP_BENCHMARKS = [
    "GRU", "Legacy-GRU", "Legacy-Attention-BiLSTM", "TCN", "DLinear", "PatchTST",
    "iTransformer", "TimeMixer", "ModernTCN", "TiDE", "TimeXer", "TimeMixer++",
    "SolarFlux-TF", "NHiTS", "TSMixer", "WPMixer", "XLinear", "Cross-Unet-11",
]
CANONICAL_REPRODUCTIONS = {"DLinear", "PatchTST", "iTransformer", "TSMixer"}
ARCHITECTURE_ALIGNED_REPRODUCTIONS = {"WPMixer", "Cross-Unet-11"}
BENCHMARK_PROVENANCE = {
    "GRU": ("unified_compact_adaptation", "PyTorch GRU"),
    "Legacy-GRU": ("approximate_legacy_reproduction", "current split; original hyperparameters incomplete"),
    "Legacy-Attention-BiLSTM": ("approximate_legacy_reproduction", "current split; original hyperparameters incomplete"),
    "TCN": ("unified_compact_adaptation", "causal dilated TCN"),
    "DLinear": ("canonical_self_contained_reproduction", "trend/seasonal decomposition + separate linear heads"),
    "PatchTST": ("canonical_self_contained_reproduction", "channel-independent shared patch Transformer"),
    "iTransformer": ("canonical_self_contained_reproduction", "variates-as-tokens Transformer"),
    "TimeMixer": ("unified_compact_adaptation", "multi-scale temporal mixing"),
    "ModernTCN": ("unified_compact_adaptation", "multi-kernel temporal mixing"),
    "TiDE": ("unified_compact_adaptation", "dense encoder-decoder"),
    "TimeXer": ("unified_compact_adaptation", "endogenous/exogenous cross attention"),
    "TimeMixer++": ("unified_compact_adaptation", "multi-scale season/trend mixing"),
    "SolarFlux-TF": ("architecture_aligned_reproduction", "causal TCN-attention scheduled-TF decoder"),
    "NHiTS": ("architecture_aligned_reproduction", "hierarchical interpolation-style blocks"),
    "TSMixer": ("canonical_self_contained_reproduction", "time and feature MLP mixing"),
    "WPMixer": ("architecture_aligned_reproduction", "Haar low/high multi-resolution mixer"),
    "XLinear": ("unified_compact_adaptation", "endogenous/exogenous gated linear interaction"),
    "Cross-Unet-11": ("architecture_aligned_reproduction", "11-step multi-scale cross-attention; not native-paper score"),
}
BENCHMARK_REPOSITORIES = {
    "DLinear": "https://github.com/honeywell21/DLinear",
    "PatchTST": "https://github.com/yuqinie98/PatchTST",
    "iTransformer": "https://github.com/thuml/iTransformer",
    "TSMixer": "https://github.com/google-research/google-research/tree/master/tsmixer",
    "WPMixer": "https://github.com/Secure-and-Intelligent-Systems-Lab/WPMixer",
    "Cross-Unet-11": "https://github.com/ZjuMachine/PV-power",
}
BENCHMARK_PROVENANCE_TABLE = pd.DataFrame([
    {
        "Model": name, "Implementation_tier": BENCHMARK_PROVENANCE[name][0],
        "Architecture_note": BENCHMARK_PROVENANCE[name][1],
        "Repository": BENCHMARK_REPOSITORIES.get(name, ""),
        "Current_dataset_retraining": True, "Same_split_and_information_boundary": True,
        "Native_paper_score_reused": False,
    }
    for name in DEEP_BENCHMARKS
])
BENCHMARK_PROVENANCE_TABLE.to_csv(OUTPUT_DIR / "MUFASA_benchmark_provenance.csv", index=False)
@dataclass(frozen=True)
class BenchmarkConfig:
    width: int = 64
    dropout: float = 0.10
    learning_rate: float = 8e-4
    weight_decay: float = 3e-4
    lookback: int = 28
    scaling_mode: str = "standard_hourwise"
    target_transform: str = "identity_standard"
    n_layers: int = 2
    patch_length: int = 4
    kernel_size: int = 3
    tf_zero_epoch: int = 8
    free_run_min_epochs: int = 8
BENCHMARK_DIMENSION = 12
BENCHMARK_WIDTHS = (32, 48, 64, 80, 96)
BENCHMARK_LOOKBACKS = (14, 21, 28)
BENCHMARK_SCALING = (
    "standard_hourwise", "robust_hourwise", "standard_channelwise",
    "robust_channelwise", "standard_global",
)
BENCHMARK_TARGETS = ("identity_standard", "log1p_standard")
def decode_benchmark_vector(vector):
    u = np.clip(np.asarray(vector, float), 0.0, 1.0 - 1e-12)
    tf_zero = _choice((1, 1, 2) if SMOKE else (6, 8, 10, 12), u[10])
    free_min = _choice((1, 1, 2) if SMOKE else (6, 8, 10, 12), u[11])
    return BenchmarkConfig(
        width=_choice(BENCHMARK_WIDTHS, u[0]),
        dropout=float(0.04 + 0.20 * u[1]),
        learning_rate=float(np.exp(np.log(1.5e-4) + u[2] * (np.log(2.0e-3) - np.log(1.5e-4)))),
        weight_decay=float(np.exp(np.log(1e-6) + u[3] * (np.log(1e-3) - np.log(1e-6)))),
        lookback=_choice(BENCHMARK_LOOKBACKS, u[4]),
        scaling_mode=_choice(BENCHMARK_SCALING, u[5]),
        target_transform=_choice(BENCHMARK_TARGETS, u[6]),
        n_layers=_choice((1, 2, 3), u[7]),
        patch_length=_choice((2, 4, 7), u[8]),
        kernel_size=_choice((3, 5, 7), u[9]),
        tf_zero_epoch=tf_zero,
        free_run_min_epochs=free_min,
    )
def benchmark_config_key(config):
    return json.dumps(asdict(config), sort_keys=True)
class BenchmarkBayesEI:
    def __init__(self, dimension, seed):
        self.dimension = dimension
        self.rng = np.random.default_rng(seed)
        self.x, self.y, self.keys = [], [], set()

    def suggest(self):
        if len(self.x) < BO_INITIAL_RANDOM:
            return self.rng.random(self.dimension), "random_warmup"
        kernel = ConstantKernel(1.0, (1e-2, 1e2)) * Matern(
            length_scale=np.ones(self.dimension), nu=2.5
        ) + WhiteKernel(1e-5, (1e-8, 1e-1))
        gp = GaussianProcessRegressor(
            kernel=kernel, normalize_y=True, alpha=1e-6,
            n_restarts_optimizer=1, random_state=SEED,
        )
        gp.fit(np.asarray(self.x), np.asarray(self.y))
        pool = self.rng.random((BO_POOL_SIZE, self.dimension))
        mean, std = gp.predict(pool, return_std=True)
        improvement = np.min(self.y) - mean - 0.002
        z = improvement / np.maximum(std, 1e-9)
        acquisition = improvement * norm.cdf(z) + std * norm.pdf(z)
        for index in np.argsort(acquisition)[::-1]:
            candidate = pool[index]
            if benchmark_config_key(decode_benchmark_vector(candidate)) not in self.keys:
                return candidate, "GP_Matern_expected_improvement"
        return self.rng.random(self.dimension), "random_fallback"

    def observe(self, vector, score, config):
        self.x.append(np.asarray(vector, float))
        self.y.append(float(score))
        self.keys.add(benchmark_config_key(config))
DEEP_SELECTION_ROWS: List[Dict[str, Any]] = []
DEEP_PROMOTION_ROWS: List[Dict[str, Any]] = []
DEEP_SEED_ROWS: List[Dict[str, Any]] = []
DEEP_SCALER_ROWS: List[Dict[str, Any]] = []
DEEP_VALIDATION_PREDICTIONS = {site: {} for site in SITES}
DEEP_CALIBRATION_PREDICTIONS = {site: {} for site in SITES}
DEEP_LEGACY_CALIBRATION_PREDICTIONS = {site: {} for site in SITES}
DEEP_FINAL_MODELS = {site: {} for site in SITES}
DEEP_BRIDGE_MODELS = {site: {} for site in SITES}
DEEP_TEST_SEED_PREDICTIONS = {site: {} for site in SITES}
DEEP_HPO_CONFIGS = {site: {} for site in SITES}
if TORCH_AVAILABLE and RUN_DEEP_BENCHMARKS:
    class CausalBenchmarkBlock(nn.Module):
        def __init__(self, width, dilation, dropout, kernel_size=3):
            super().__init__()
            self.left = (kernel_size - 1) * dilation
            self.conv1 = weight_norm(nn.Conv1d(width, width, kernel_size, dilation=dilation))
            self.conv2 = weight_norm(nn.Conv1d(width, width, kernel_size, dilation=dilation))
            self.norm1 = nn.GroupNorm(1, width)
            self.norm2 = nn.GroupNorm(1, width)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x):
            y = self.conv1(F.pad(x, (self.left, 0)))
            y = self.dropout(F.leaky_relu(self.norm1(y), 0.05))
            y = self.conv2(F.pad(y, (self.left, 0)))
            y = self.dropout(F.leaky_relu(self.norm2(y), 0.05))
            return x + y


    class MultiKernelMixer(nn.Module):
        def __init__(self, width, dropout):
            super().__init__()
            self.norm = nn.LayerNorm(width)
            self.short = nn.Conv1d(width, width, 3, padding=1, groups=width)
            self.long = nn.Conv1d(width, width, 7, padding=3, groups=width)
            self.mix = nn.Conv1d(2 * width, width, 1)
            self.dropout = nn.Dropout(dropout)

        def forward(self, x):
            residual = x
            y = self.norm(x).transpose(1, 2)
            y = self.mix(torch.cat([self.short(y), self.long(y)], dim=1)).transpose(1, 2)
            return residual + self.dropout(F.gelu(y))


    class TSMixerBlock(nn.Module):
        def __init__(self, days, width, dropout):
            super().__init__()
            hidden_time = max(16, 2 * days)
            self.time_norm = nn.LayerNorm(width)
            self.time_mlp = nn.Sequential(
                nn.Linear(days, hidden_time), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(hidden_time, days),
            )
            self.feature_norm = nn.LayerNorm(width)
            self.feature_mlp = nn.Sequential(
                nn.Linear(width, 2 * width), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(2 * width, width),
            )

        def forward(self, x):
            x = x + self.time_mlp(self.time_norm(x).transpose(1, 2)).transpose(1, 2)
            return x + self.feature_mlp(self.feature_norm(x))


In [ ]:
# Benchmark Suite and Matched-Budget Optimization — part 2/5
if TORCH_AVAILABLE and RUN_DEEP_BENCHMARKS:
    class DeepBenchmarkNet(nn.Module):
        def __init__(self, name, sequence_shape, context_dim, config):
            super().__init__()
            self.name, self.config = name, config
            days, hours, channels = sequence_shape
            self.days, self.hours, self.channels = days, hours, channels
            self.input_dim = hours * channels
            width, dropout = config.width, config.dropout

            if name in {"GRU", "Legacy-GRU"}:
                self.recurrent = nn.GRU(
                    self.input_dim, width, num_layers=config.n_layers,
                    dropout=dropout if config.n_layers > 1 else 0.0, batch_first=True,
                )
                self.head = self._head(width + context_dim, width, dropout)
            elif name == "Legacy-Attention-BiLSTM":
                self.recurrent = nn.LSTM(
                    self.input_dim, width // 2, num_layers=config.n_layers,
                    dropout=dropout if config.n_layers > 1 else 0.0,
                    batch_first=True, bidirectional=True,
                )
                self.legacy_attention = nn.Linear(width, 1)
                self.head = self._head(width + context_dim, width, dropout)
            elif name in {"TCN", "ModernTCN", "SolarFlux-TF"}:
                self.projection = nn.Linear(self.input_dim, width)
                if name == "ModernTCN":
                    self.temporal = nn.Sequential(*[
                        MultiKernelMixer(width, dropout) for _ in range(config.n_layers + 1)
                    ])
                else:
                    dilations = (1, 2, 4, 8)[:config.n_layers + 1]
                    self.temporal = nn.Sequential(*[
                        CausalBenchmarkBlock(width, dilation, dropout, config.kernel_size)
                        for dilation in dilations
                    ])
                if name == "TCN":
                    self.head = self._head(width + context_dim, width, dropout)
                elif name == "ModernTCN":
                    self.head = self._head(2 * width + context_dim, width, dropout)
                else:
                    layer = nn.TransformerEncoderLayer(
                        width, 4, 3 * width, dropout=dropout, activation="gelu",
                        batch_first=True, norm_first=True,
                    )
                    self.sf_attention = nn.TransformerEncoder(
                        layer, num_layers=config.n_layers, norm=nn.LayerNorm(width)
                    )
                    self.sf_context = nn.Linear(context_dim, width)
                    self.sf_horizon = nn.Parameter(torch.zeros(1, N_HORIZONS, width))
                    self.sf_initial = nn.Linear(2 * width, width)
                    self.sf_cell = nn.GRUCell(width + 1, width)
                    self.sf_out = nn.Linear(width, 1)
                    self.sf_bos = nn.Parameter(torch.zeros(1, 1))
            elif name == "TiDE":
                total = days * self.input_dim + context_dim
                self.tide_encoder = nn.Sequential(
                    nn.Linear(total, 4 * width), nn.LayerNorm(4 * width), nn.GELU(),
                    nn.Dropout(dropout), nn.Linear(4 * width, 2 * width), nn.GELU(),
                    nn.Dropout(dropout), nn.Linear(2 * width, N_HORIZONS),
                )
            elif name == "TimeXer":
                self.xer_past = nn.Linear(self.input_dim, width)
                self.xer_exogenous = nn.Linear(context_dim, N_HORIZONS * width)
                self.xer_horizon = nn.Parameter(torch.zeros(1, N_HORIZONS, width))
                layer = nn.TransformerEncoderLayer(
                    width, 4, 3 * width, dropout=dropout, activation="gelu",
                    batch_first=True, norm_first=True,
                )
                self.xer_encoder = nn.TransformerEncoder(
                    layer, num_layers=config.n_layers, norm=nn.LayerNorm(width)
                )
                self.xer_cross = nn.MultiheadAttention(width, 4, dropout=dropout, batch_first=True)
                self.xer_head = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 1))
            elif name == "TimeMixer++":
                self.tmpp_projection = nn.Linear(self.input_dim, width)
                self.tmpp_scales = (1, 2, 4, 7)
                self.tmpp_season = nn.ModuleList([
                    nn.Sequential(*[MultiKernelMixer(width, dropout) for _ in range(config.n_layers)])
                    for _ in self.tmpp_scales
                ])
                self.tmpp_trend = nn.ModuleList([
                    nn.Sequential(*[MultiKernelMixer(width, dropout) for _ in range(config.n_layers)])
                    for _ in self.tmpp_scales
                ])
                self.tmpp_head = self._head(
                    2 * len(self.tmpp_scales) * width + context_dim, width, dropout
                )
            elif name == "DLinear":
                self.dlinear_seasonal = nn.Linear(days, N_HORIZONS)
                self.dlinear_trend = nn.Linear(days, N_HORIZONS)
                self.dlinear_feature = nn.Linear(self.input_dim, 1)
                self.context_head = nn.Linear(context_dim, N_HORIZONS)
            elif name in {"PatchTST", "iTransformer"}:
                if name == "PatchTST":
                    self.patch_length = min(config.patch_length, days)
                    self.patch_stride = max(1, self.patch_length // 2)
                    token_dim = self.patch_length
                    tokens = 1 + (days - self.patch_length) // self.patch_stride
                    self.patch_variable_score = nn.Linear(width, 1)
                else:
                    token_dim, tokens = days, self.input_dim
                self.token_projection = nn.Linear(token_dim, width)
                self.position = nn.Parameter(torch.zeros(1, tokens, width))
                layer = nn.TransformerEncoderLayer(
                    width, 4, 3 * width, dropout=dropout, activation="gelu",
                    batch_first=True, norm_first=True,
                )
                self.transformer = nn.TransformerEncoder(
                    layer, num_layers=config.n_layers, norm=nn.LayerNorm(width)
                )
                self.head = self._head(width + context_dim, width, dropout)
            elif name == "TimeMixer":
                self.projection = nn.Linear(self.input_dim, width)
                self.scales = (1, 2, 4, 7)
                self.mixers = nn.ModuleList([
                    nn.Sequential(*[MultiKernelMixer(width, dropout) for _ in range(config.n_layers)])
                    for _ in self.scales
                ])
                self.head = self._head(len(self.scales) * width + context_dim, width, dropout)
            elif name == "NHiTS":
                self.nhits_scales = (1, 2, 4)
                self.nhits_blocks = nn.ModuleList()
                for scale in self.nhits_scales:
                    pooled_days = int(math.ceil(days / scale))
                    self.nhits_blocks.append(nn.Sequential(
                        nn.Linear(pooled_days * self.input_dim + context_dim, 2 * width),
                        nn.LayerNorm(2 * width), nn.GELU(), nn.Dropout(dropout),
                        nn.Linear(2 * width, width), nn.GELU(),
                        nn.Linear(width, N_HORIZONS),
                    ))
                self.nhits_weights = nn.Parameter(torch.zeros(len(self.nhits_scales)))
            elif name == "TSMixer":
                self.tsmix_projection = nn.Linear(self.input_dim, width)
                self.tsmix_blocks = nn.Sequential(*[
                    TSMixerBlock(days, width, dropout) for _ in range(config.n_layers + 1)
                ])
                self.tsmix_head = self._head(2 * width + context_dim, width, dropout)
            elif name == "WPMixer":
                self.wp_projection = nn.Linear(self.input_dim, width)
                self.wp_local = nn.Sequential(*[
                    MultiKernelMixer(width, dropout) for _ in range(config.n_layers)
                ])
                self.wp_low = nn.Sequential(*[
                    MultiKernelMixer(width, dropout) for _ in range(config.n_layers)
                ])
                self.wp_high = nn.Sequential(*[
                    MultiKernelMixer(width, dropout) for _ in range(config.n_layers)
                ])
                self.wp_head = self._head(3 * width + context_dim, width, dropout)
            elif name == "Cross-Unet-11":
                
                self.cu_projection = nn.Linear(self.input_dim, width)
                self.cu_context = nn.Linear(context_dim, N_HORIZONS * width)
                self.cu_horizon = nn.Parameter(torch.zeros(1, N_HORIZONS, width))
                self.cu_blocks = nn.ModuleList([
                    MultiKernelMixer(width, dropout) for _ in range(3)
                ])
                self.cu_cross = nn.ModuleList([
                    nn.MultiheadAttention(width, 4, dropout=dropout, batch_first=True)
                    for _ in range(3)
                ])
                self.cu_norm = nn.ModuleList([nn.LayerNorm(width) for _ in range(3)])
                self.cu_head = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 1))
            elif name == "XLinear":
                self.xlin_endogenous = nn.Sequential(
                    nn.Linear(days * self.input_dim, 2 * width), nn.GELU(),
                    nn.Dropout(dropout), nn.Linear(2 * width, width),
                )
                self.xlin_exogenous = nn.Sequential(
                    nn.Linear(context_dim, 2 * width), nn.GELU(), nn.Linear(2 * width, width),
                )
                self.xlin_gate = nn.Sequential(nn.Linear(2 * width, width), nn.Sigmoid())
                self.xlin_head = self._head(2 * width, width, dropout)
            else:
                raise KeyError(name)

        @staticmethod
        def _head(input_dim, width, dropout):
            return nn.Sequential(
                nn.Linear(input_dim, width), nn.LayerNorm(width), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(width, N_HORIZONS),
            )

        def forward(self, sequence, context, teacher=None, teacher_ratio=0.0):
            b, days, hours, channels = sequence.shape
            flat = sequence.reshape(b, days, -1)
            if self.name in {"GRU", "Legacy-GRU"}:
                output, _ = self.recurrent(flat)
                return self.head(torch.cat([output[:, -1], context], dim=1))
            if self.name == "Legacy-Attention-BiLSTM":
                output, _ = self.recurrent(flat)
                weight = torch.softmax(self.legacy_attention(output).squeeze(-1), dim=1)
                summary = torch.sum(output * weight.unsqueeze(-1), dim=1)
                return self.head(torch.cat([summary, context], dim=1))
            if self.name == "TCN":
                temporal = self.temporal(self.projection(flat).transpose(1, 2))
                return self.head(torch.cat([temporal[:, :, -1], context], dim=1))
            if self.name == "ModernTCN":
                temporal = self.temporal(self.projection(flat))
                return self.head(torch.cat([temporal[:, -1], temporal.mean(dim=1), context], dim=1))
            if self.name == "SolarFlux-TF":
                temporal = self.temporal(self.projection(flat).transpose(1, 2)).transpose(1, 2)
                attended = self.sf_attention(temporal)
                context_token = self.sf_context(context)
                hidden = torch.tanh(self.sf_initial(torch.cat([attended[:, -1], context_token], dim=1)))
                previous = self.sf_bos.expand(b, 1)
                outputs = []
                for horizon in range(N_HORIZONS):
                    hidden = self.sf_cell(
                        torch.cat([self.sf_horizon[:, horizon].expand(b, -1), previous], dim=1), hidden
                    )
                    current = self.sf_out(hidden)
                    outputs.append(current)
                    if teacher is not None and horizon < N_HORIZONS - 1 and teacher_ratio > 0:
                        mask = torch.rand((b, 1), device=current.device) < teacher_ratio
                        previous = torch.where(mask, teacher[:, horizon:horizon + 1], current.detach())
                    else:
                        previous = current
                return torch.cat(outputs, dim=1)
            if self.name == "TiDE":
                return self.tide_encoder(torch.cat([flat.reshape(b, -1), context], dim=1))
            if self.name == "TimeXer":
                memory = self.xer_encoder(self.xer_past(flat))
                query = self.xer_exogenous(context).reshape(b, N_HORIZONS, -1) + self.xer_horizon
                cross, _ = self.xer_cross(query, memory, memory, need_weights=False)
                return self.xer_head(cross + query).squeeze(-1)
            if self.name == "TimeMixer++":
                tokens = self.tmpp_projection(flat)
                summaries = []
                for scale, season_mixer, trend_mixer in zip(
                    self.tmpp_scales, self.tmpp_season, self.tmpp_trend
                ):
                    scaled = (
                        F.avg_pool1d(tokens.transpose(1, 2), scale, scale, ceil_mode=True).transpose(1, 2)
                        if scale > 1 else tokens
                    )
                    trend = F.avg_pool1d(scaled.transpose(1, 2), 3, 1, 1).transpose(1, 2)
                    season = scaled - trend
                    summaries.extend([season_mixer(season).mean(dim=1), trend_mixer(trend).mean(dim=1)])
                return self.tmpp_head(torch.cat([*summaries, context], dim=1))
            if self.name == "DLinear":
                series = flat.transpose(1, 2)
                padded = F.pad(series, (1, 1), mode="replicate")
                trend = F.avg_pool1d(padded, kernel_size=3, stride=1)
                seasonal = series - trend
                seasonal_out = self.dlinear_seasonal(seasonal).transpose(1, 2)
                trend_out = self.dlinear_trend(trend).transpose(1, 2)
                return self.dlinear_feature(seasonal_out + trend_out).squeeze(-1) + self.context_head(context)
            if self.name == "PatchTST":
                # [B, variable, patch, patch_length]; Transformer weights are shared by variable.
                patches = flat.transpose(1, 2).unfold(2, self.patch_length, self.patch_stride)
                variable_count, patch_count = patches.shape[1], patches.shape[2]
                tokens = self.token_projection(
                    patches.reshape(b * variable_count, patch_count, self.patch_length)
                )
                encoded = self.transformer(tokens + self.position[:, :patch_count])
                variable_repr = encoded.mean(dim=1).reshape(b, variable_count, -1)
                variable_weight = torch.softmax(
                    self.patch_variable_score(variable_repr).squeeze(-1), dim=1
                )
                summary = torch.sum(variable_repr * variable_weight.unsqueeze(-1), dim=1)
                return self.head(torch.cat([summary, context], dim=1))
            if self.name == "iTransformer":
                tokens = self.token_projection(flat.transpose(1, 2))
                encoded = self.transformer(tokens + self.position[:, :tokens.shape[1]])
                return self.head(torch.cat([encoded.mean(dim=1), context], dim=1))
            if self.name == "TimeMixer":
                tokens = self.projection(flat)
                summaries = []
                for scale, mixer in zip(self.scales, self.mixers):
                    scaled = (
                        F.avg_pool1d(tokens.transpose(1, 2), scale, scale, ceil_mode=True).transpose(1, 2)
                        if scale > 1 else tokens
                    )
                    summaries.append(mixer(scaled).mean(dim=1))
                return self.head(torch.cat([*summaries, context], dim=1))
            if self.name == "NHiTS":
                outputs = []
                for scale, block in zip(self.nhits_scales, self.nhits_blocks):
                    scaled = (
                        F.avg_pool1d(flat.transpose(1, 2), scale, scale, ceil_mode=True).transpose(1, 2)
                        if scale > 1 else flat
                    )
                    outputs.append(block(torch.cat([scaled.reshape(b, -1), context], dim=1)))
                weights = torch.softmax(self.nhits_weights, dim=0)
                return sum(weight * output for weight, output in zip(weights, outputs))
            if self.name == "TSMixer":
                tokens = self.tsmix_blocks(self.tsmix_projection(flat))
                return self.tsmix_head(torch.cat([tokens[:, -1], tokens.mean(dim=1), context], dim=1))
            if self.name == "WPMixer":
                tokens = self.wp_local(self.wp_projection(flat))
                usable = tokens[:, -2 * (tokens.shape[1] // 2):]
                even, odd = usable[:, 0::2], usable[:, 1::2]
                low = self.wp_low((even + odd) / math.sqrt(2.0))
                high = self.wp_high((even - odd) / math.sqrt(2.0))
                return self.wp_head(torch.cat([
                    tokens.mean(dim=1), low.mean(dim=1), high.mean(dim=1), context
                ], dim=1))
            if self.name == "Cross-Unet-11":
                token = self.cu_projection(flat)
                levels = []
                for index, block in enumerate(self.cu_blocks):
                    token = block(token); levels.append(token)
                    if index < 2:
                        token = F.avg_pool1d(
                            token.transpose(1, 2), 2, 2, ceil_mode=True
                        ).transpose(1, 2)
                query = self.cu_context(context).reshape(b, N_HORIZONS, -1) + self.cu_horizon
                for level, cross, norm_layer in zip(reversed(levels), self.cu_cross, self.cu_norm):
                    update, _ = cross(query, level, level, need_weights=False)
                    query = norm_layer(query + update)
                return self.cu_head(query).squeeze(-1)
            if self.name == "XLinear":
                endogenous = self.xlin_endogenous(flat.reshape(b, -1))
                exogenous = self.xlin_exogenous(context)
                gate = self.xlin_gate(torch.cat([endogenous, exogenous], dim=1))
                interacted = gate * exogenous + (1.0 - gate) * endogenous
                return self.xlin_head(torch.cat([endogenous, interacted], dim=1))
            raise KeyError(self.name)


In [ ]:
# Benchmark Suite and Matched-Budget Optimization — part 3/5
if TORCH_AVAILABLE and RUN_DEEP_BENCHMARKS:
    @dataclass
    class TrainedBenchmark:
        model: Any
        config: BenchmarkConfig
        seq_mean: np.ndarray
        seq_std: np.ndarray
        context_mean: np.ndarray
        context_std: np.ndarray
        target_mean: np.ndarray
        target_std: np.ndarray
        best_epoch: int
        validation_rmse: float
        name: str
        width: int
        scaler_digest: str
        scaler_fit_start: str
        scaler_fit_end: str
        test_rows_in_scaler_fit: int


    def _benchmark_scalers(bundle, train_idx, config):
        train_idx = np.asarray(train_idx, int)
        if np.any(bundle.test[train_idx]):
            raise AssertionError("Test rows entered benchmark preprocessing.")
        sequence = bundle.sequence[:, -config.lookback:]
        robust = config.scaling_mode.startswith("robust")
        hourwise = config.scaling_mode.endswith("hourwise")
        global_mode = config.scaling_mode == "standard_global"
        seq_axes = (0, 1, 2, 3) if global_mode else ((0, 1) if hourwise else (0, 1, 2))
        seq_mean, seq_std = _center_scale(sequence[train_idx], seq_axes, robust)
        context = np.concatenate([bundle.context, bundle.tabular], axis=1)
        context_mean, context_std = _center_scale(context[train_idx], 0, robust)
        transformed_target = (
            np.log1p(np.clip(bundle.target, 0.0, None))
            if config.target_transform == "log1p_standard" else bundle.target
        )
        target_axes = 0 if hourwise else (0, 1)
        target_mean, target_std = _center_scale(transformed_target[train_idx], target_axes, False)
        dates = bundle.target_dates[train_idx]
        digest = hashlib.sha256(b"".join(
            np.asarray(x, np.float32).tobytes()
            for x in (seq_mean, seq_std, context_mean, context_std, target_mean, target_std)
        )).hexdigest()
        return {
            "sequence": sequence, "context": context,
            "seq_mean": seq_mean, "seq_std": seq_std,
            "context_mean": context_mean, "context_std": context_std,
            "target_mean": target_mean, "target_std": target_std,
            "digest": digest, "fit_start": str(dates.min().date()),
            "fit_end": str(dates.max().date()), "test_rows": int(np.sum(bundle.test[train_idx])),
        }


    def fit_benchmark_network(
        bundle, train_idx, valid_idx, name, config, seed, fixed_epochs=None,
        max_epochs_override=None, patience_override=None,
        batch_size_override=None, amp_enabled_override=None,
    ):
        set_seed(seed)
        scalers = _benchmark_scalers(bundle, train_idx, config)

        def transform(indices):
            indices = np.asarray(indices, int)
            sequence = ((scalers["sequence"][indices] - scalers["seq_mean"]) / scalers["seq_std"]).astype(np.float32)
            context = ((scalers["context"][indices] - scalers["context_mean"]) / scalers["context_std"]).astype(np.float32)
            physical = bundle.target[indices]
            target_latent = np.log1p(np.clip(physical, 0.0, None)) if config.target_transform == "log1p_standard" else physical
            target = ((target_latent - scalers["target_mean"]) / scalers["target_std"]).astype(np.float32)
            return sequence, context, target

        tr_seq, tr_context, tr_target = transform(train_idx)
        va_seq, va_context, _ = transform(valid_idx) if len(valid_idx) else (None, None, None)
        model = DeepBenchmarkNet(name, tuple(tr_seq.shape[1:]), tr_context.shape[1], config).to(DEVICE)
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
        )
        loader = DataLoader(
            TensorDataset(torch.from_numpy(tr_seq), torch.from_numpy(tr_context), torch.from_numpy(tr_target)),
            batch_size=int(batch_size_override or GPU_BATCH_SIZE), shuffle=True, pin_memory=DEVICE.type == "cuda", num_workers=0,
        )
        max_epochs = int(fixed_epochs) if fixed_epochs is not None else int(
            BENCHMARK_MAX_EPOCHS if max_epochs_override is None else max_epochs_override
        )
        patience = max_epochs + 1 if fixed_epochs is not None else int(
            BENCHMARK_PATIENCE if patience_override is None else patience_override
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(1, max_epochs), eta_min=max(1e-6, config.learning_rate / 100.0)
        )
        amp_enabled = USE_AMP if amp_enabled_override is None else bool(
            amp_enabled_override and DEVICE.type == "cuda"
        )
        try:
            amp_scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
        except Exception:
            amp_scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
        best_rmse, best_state, best_epoch, stale = np.inf, None, 0, 0
        minimum_eligible = (
            min(max_epochs, config.tf_zero_epoch + config.free_run_min_epochs)
            if name == "SolarFlux-TF" else 1
        )
        for epoch in range(1, max_epochs + 1):
            model.train()
            if name == "SolarFlux-TF":
                sf_ratio = max(0.0, 0.80 * (1.0 - (epoch - 1) / max(config.tf_zero_epoch - 1, 1)))
                if epoch >= config.tf_zero_epoch:
                    sf_ratio = 0.0
            else:
                sf_ratio = 0.0
            for sequence_batch, context_batch, target_batch in loader:
                sequence_batch, context_batch, target_batch = [
                    value.to(DEVICE, dtype=torch.float32, non_blocking=True)
                    for value in (sequence_batch, context_batch, target_batch)
                ]
                optimizer.zero_grad(set_to_none=True)
                with _amp_context(amp_enabled_override):
                    prediction = model(sequence_batch, context_batch, target_batch, sf_ratio)
                    if name == "SolarFlux-TF" and sf_ratio > 0:
                        free_prediction = model(sequence_batch, context_batch, teacher=None, teacher_ratio=0.0)
                        loss = F.smooth_l1_loss(free_prediction, target_batch) + 0.20 * F.smooth_l1_loss(prediction, target_batch)
                    else:
                        loss = F.smooth_l1_loss(prediction, target_batch)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"{name}: non-finite loss")
                amp_scaler.scale(loss).backward()
                amp_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                amp_scaler.step(optimizer)
                amp_scaler.update()
            scheduler.step()

            if fixed_epochs is not None:
                best_epoch = epoch
                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
                continue
            if epoch < minimum_eligible:
                continue
            model.eval()
            parts = []
            with torch.inference_mode():
                for start in range(0, len(va_seq), 512):
                    with _amp_context(amp_enabled_override):
                        batch = model(
                            torch.from_numpy(va_seq[start:start + 512]).to(DEVICE),
                            torch.from_numpy(va_context[start:start + 512]).to(DEVICE),
                            teacher=None, teacher_ratio=0.0,
                        )
                    parts.append(batch.float().cpu().numpy())
            latent = np.concatenate(parts) * scalers["target_std"] + scalers["target_mean"]
            physical = np.expm1(latent) if config.target_transform == "log1p_standard" else latent
            physical = np.clip(physical, 0.0, None)
            rmse = metric_set(bundle.target[valid_idx], physical)["RMSE"]
            if rmse < best_rmse - 1e-5:
                best_rmse, best_epoch, stale = rmse, epoch, 0
                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            else:
                stale += 1
                if stale >= patience:
                    break
        if best_state is None:
            raise RuntimeError(f"{name} failed to retain an eligible free-run state.")
        model.load_state_dict(best_state)
        model.eval()
        return TrainedBenchmark(
            model, config, scalers["seq_mean"], scalers["seq_std"],
            scalers["context_mean"], scalers["context_std"],
            scalers["target_mean"], scalers["target_std"], int(best_epoch),
            float(best_rmse), name, config.width, scalers["digest"],
            scalers["fit_start"], scalers["fit_end"], scalers["test_rows"],
        )


    def predict_benchmark_network(trained, bundle, indices):
        """Public-release implementation note."""
        indices = np.asarray(indices, int)
        expected_shape = (len(indices), N_HORIZONS)
        if len(indices) == 0:
            return np.empty(expected_shape, dtype=np.float32)
        sequence = bundle.sequence[indices, -trained.config.lookback:]
        sequence = ((sequence - trained.seq_mean) / trained.seq_std).astype(np.float32)
        combined_context = np.concatenate([bundle.context[indices], bundle.tabular[indices]], axis=1)
        context = ((combined_context - trained.context_mean) / trained.context_std).astype(np.float32)
        parts = []
        trained.model.eval()
        with torch.inference_mode():
            for start in range(0, len(indices), 512):
                
                with _amp_context(False):
                    batch = trained.model(
                        torch.from_numpy(sequence[start:start + 512]).to(DEVICE),
                        torch.from_numpy(context[start:start + 512]).to(DEVICE),
                        teacher=None, teacher_ratio=0.0,
                    )
                if tuple(batch.shape) != (len(sequence[start:start + 512]), N_HORIZONS):
                    raise RuntimeError(
                        f"{trained.name}: latent batch shape={tuple(batch.shape)}; "
                        f"expected={(len(sequence[start:start + 512]), N_HORIZONS)}"
                    )
                parts.append(batch.float().cpu().numpy())
        latent = np.concatenate(parts, axis=0) * trained.target_std + trained.target_mean
        physical = np.expm1(latent) if trained.config.target_transform == "log1p_standard" else latent
        physical = np.clip(np.asarray(physical, dtype=np.float32), 0.0, None)
        if physical.shape != expected_shape:
            raise RuntimeError(
                f"{trained.name}: benchmark prediction shape mismatch: "
                f"{physical.shape} != {expected_shape}"
            )
        if not np.isfinite(physical).all():
            bad = np.argwhere(~np.isfinite(physical))[:5].tolist()
            raise FloatingPointError(f"{trained.name}: benchmark prediction contains NaN/inf at {bad}")
        return physical.astype(np.float32, copy=False)
BENCHMARK_RETRY_ROWS = []


In [ ]:
# Benchmark Suite and Matched-Budget Optimization — part 4/5
def fit_benchmark_network_safe(bundle, train_idx, valid_idx, name, config, seed, **kwargs):
    """Public-release implementation note."""
    attempts = [
        (GPU_BATCH_SIZE, None, "original"),
        (max(8, GPU_BATCH_SIZE // 2), False, "half_batch_amp_off"),
        (max(8, GPU_BATCH_SIZE // 4), False, "quarter_batch_amp_off"),
    ]
    errors = []
    for batch_size, amp_override, label in attempts:
        try:
            trained = fit_benchmark_network(
                bundle, train_idx, valid_idx, name, config, seed,
                batch_size_override=batch_size, amp_enabled_override=amp_override, **kwargs,
            )
            BENCHMARK_RETRY_ROWS.append({
                "Site": bundle.site, "Model": name, "Seed": seed, "Attempt": label,
                "Batch_size": batch_size, "AMP_override": amp_override,
                "Status": "ok", "Error": "",
            })
            return trained
        except Exception as exc:
            message = f"{type(exc).__name__}: {exc}"
            errors.append(message)
            BENCHMARK_RETRY_ROWS.append({
                "Site": bundle.site, "Model": name, "Seed": seed, "Attempt": label,
                "Batch_size": batch_size, "AMP_override": amp_override,
                "Status": "failed", "Error": message,
            })
            lower = message.lower()
            retryable = any(token in lower for token in (
                "out of memory", "cublas", "cudnn", "non-finite", "nan", "inf"
            ))
            _release_torch_memory()
            if not retryable:
                break
    raise RuntimeError(f"{bundle.site}/{name}: benchmark training failed: {' | '.join(errors)}")
def run_benchmark_contract_preflight(bundle, scenario):
    rows = []
    if not (TORCH_AVAILABLE and RUN_DEEP_BENCHMARKS):
        return pd.DataFrame([{
            "Scenario": scenario, "Model": "ALL", "Status": "skipped",
            "Reason": "PyTorch/deep benchmarks disabled; paper eligibility remains false",
        }])
    train_idx = positions(bundle.train)
    probe_idx = train_idx[-min(3, len(train_idx)):]
    config = BenchmarkConfig(width=32, n_layers=1, lookback=min(14, MAX_LOOKBACK_DAYS))
    scalers = _benchmark_scalers(bundle, train_idx, config)
    seq_shape = tuple(scalers["sequence"][probe_idx].shape[1:])
    context_dim = int(scalers["context"].shape[1])
    for name in DEEP_BENCHMARKS:
        started = time.perf_counter()
        try:
            model = DeepBenchmarkNet(name, seq_shape, context_dim, config).to(DEVICE).eval()
            trained = TrainedBenchmark(
                model, config, scalers["seq_mean"], scalers["seq_std"],
                scalers["context_mean"], scalers["context_std"],
                scalers["target_mean"], scalers["target_std"], 0, np.nan,
                name, config.width, scalers["digest"], scalers["fit_start"],
                scalers["fit_end"], scalers["test_rows"],
            )
            prediction = predict_benchmark_network(trained, bundle, probe_idx)
            rows.append({
                "Scenario": scenario, "Model": name, "Status": "passed",
                "Prediction_shape": str(tuple(prediction.shape)),
                "Finite": bool(np.isfinite(prediction).all()),
                "Seconds": time.perf_counter() - started, "Reason": "",
            })
            del model, trained
            _release_torch_memory()
        except Exception as exc:
            rows.append({
                "Scenario": scenario, "Model": name, "Status": "failed",
                "Prediction_shape": "", "Finite": False,
                "Seconds": time.perf_counter() - started,
                "Reason": f"{type(exc).__name__}: {exc}",
            })
    frame = pd.DataFrame(rows)
    if REQUIRE_COMPLETE_BENCHMARK and not frame["Status"].eq("passed").all():
        failures = frame.loc[~frame["Status"].eq("passed"), ["Model", "Reason"]]
        raise RuntimeError(f"Benchmark contract preflight failed:\n{failures.to_string(index=False)}")
    return frame
def run_fair_deep_benchmarks(selection_bundles, refit_bundles, scenario):
    predictions = {site: {} for site in selection_bundles}
    hpo_rows, promotion_rows, seed_rows, policy_rows = [], [], [], []
    if not (TORCH_AVAILABLE and RUN_DEEP_BENCHMARKS):
        return predictions, pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame([{
            "Scenario": scenario, "Status": "skipped",
            "Reason": "PyTorch/GPU benchmark disabled; paper claim audit must remain false",
        }])
    for site, bundle in selection_bundles.items():
        refit_bundle = refit_bundles[site]
        train_idx, val_idx = positions(bundle.train), positions(bundle.val_all)
        fit_idx, test_idx = positions(refit_bundle.train_val), positions(refit_bundle.test)
        for model_index, name in enumerate(DEEP_BENCHMARKS):
            optimizer = BenchmarkBayesEI(
                BENCHMARK_DIMENSION, HPO_SEED + 1009 * (model_index + 1) + sum(map(ord, site))
            )
            successful = []
            for trial in range(BASELINE_BO_TRIALS):
                vector, acquisition = optimizer.suggest(); config = decode_benchmark_vector(vector)
                try:
                    trained = fit_benchmark_network_safe(
                        bundle, train_idx, val_idx, name, config, HPO_SEED,
                        max_epochs_override=BASELINE_BO_LOW_EPOCHS,
                        patience_override=BO_LOW_PATIENCE,
                    )
                    prediction = predict_benchmark_network(trained, bundle, val_idx)
                    metrics = temporal_block_metrics(bundle.target[val_idx], prediction, bundle.target_dates[val_idx])
                    optimizer.observe(vector, metrics["Mean_block_RMSE"], config)
                    row = {"Scenario": scenario, "Site": site, "Model": name, "Trial": trial,
                           "Acquisition": acquisition, "Parameters": json.dumps(asdict(config), sort_keys=True),
                           "Parameter_count": int(sum(p.numel() for p in trained.model.parameters())),
                           "Status": "ok", **metrics}
                    hpo_rows.append(row); successful.append((row, config, trained))
                except Exception as exc:
                    optimizer.observe(vector, 1e3, config)
                    hpo_rows.append({"Scenario": scenario, "Site": site, "Model": name,
                                     "Trial": trial, "Status": "failed", "Error": f"{type(exc).__name__}: {exc}"})
            if not successful:
                continue
            frame = pd.DataFrame([item[0] for item in successful])
            best = frame["Mean_block_RMSE"].min()
            promotion_ids = frame[frame["Mean_block_RMSE"] <= best * (1 + STRUCTURE_NEAR_BEST_TOL)].sort_values(
                ["SD_block_RMSE", "Worst_block_RMSE", "Parameter_count", "Trial"]
            )["Trial"].tolist()[:BASELINE_BO_PROMOTE]
            if len(promotion_ids) < BASELINE_BO_PROMOTE:
                for value in frame.sort_values("Mean_block_RMSE")["Trial"]:
                    if value not in promotion_ids:
                        promotion_ids.append(value)
                    if len(promotion_ids) >= BASELINE_BO_PROMOTE:
                        break
            lookup = {item[0]["Trial"]: item for item in successful}; promoted = []
            for rank, trial_id in enumerate(promotion_ids, 1):
                config = lookup[trial_id][1]
                trained = fit_benchmark_network_safe(
                    bundle, train_idx, val_idx, name, config, HPO_SEED + rank,
                    max_epochs_override=BENCHMARK_MAX_EPOCHS,
                    patience_override=BENCHMARK_PATIENCE,
                )
                prediction = predict_benchmark_network(trained, bundle, val_idx)
                metrics = temporal_block_metrics(bundle.target[val_idx], prediction, bundle.target_dates[val_idx])
                row = {"Scenario": scenario, "Site": site, "Model": name,
                       "Promotion_rank": rank, "Source_trial": trial_id,
                       "Parameter_count": int(sum(p.numel() for p in trained.model.parameters())),
                       "Best_epoch": trained.best_epoch, **metrics}
                promotion_rows.append(row); promoted.append((row, config, trained))
            promoted_frame = pd.DataFrame([item[0] for item in promoted])
            selected, _ = stable_near_best(promoted_frame, "Promotion_rank")
            chosen = [item for item in promoted if item[0]["Promotion_rank"] == selected["Promotion_rank"]][0]
            _, config, selected_model = chosen
            fixed_epoch = max(int(selected_model.best_epoch),
                              int(config.tf_zero_epoch + config.free_run_min_epochs) if name == "SolarFlux-TF" else 1)

            
            val_seed_predictions = []
            for seed in ACTIVE_FINAL_SEEDS:
                trained = fit_benchmark_network_safe(bundle, train_idx, val_idx, name, config, seed,
                                                fixed_epochs=fixed_epoch)
                val_seed_predictions.append(predict_benchmark_network(trained, bundle, val_idx))
            val_cube = np.stack(val_seed_predictions, axis=-1)
            weights, seed_audit = fit_seed_policy(
                val_cube, bundle.target[val_idx], bundle.target_dates[val_idx], site, f"{scenario}:{name}"
            )
            policy_rows.extend(seed_audit.to_dict("records"))
            test_seed_predictions = []
            for seed in ACTIVE_FINAL_SEEDS:
                trained = fit_benchmark_network_safe(refit_bundle, fit_idx, np.array([], int), name, config, seed,
                                                fixed_epochs=fixed_epoch)
                prediction = predict_benchmark_network(trained, refit_bundle, test_idx)
                test_seed_predictions.append(prediction)
                seed_rows.append({"Scenario": scenario, "Site": site, "Model": name, "Seed": seed,
                                  "Fit_end": trained.scaler_fit_end,
                                  "Test_rows_in_scaler_fit": trained.test_rows_in_scaler_fit,
                                  "Teacher_forcing_test": 0.0,
                                  **metric_set(refit_bundle.target[test_idx], prediction)})
            predictions[site][name] = apply_seed_policy(np.stack(test_seed_predictions, axis=-1), weights)
    return predictions, pd.DataFrame(hpo_rows), pd.DataFrame(promotion_rows), pd.DataFrame(seed_rows), pd.DataFrame(policy_rows)
STRICT_BENCHMARK_PREFLIGHT = run_benchmark_contract_preflight(
    STRICT_SELECTION_BUNDLES[SITES[0]], "strict_history"
)
ORACLE_BENCHMARK_PREFLIGHT = (
    run_benchmark_contract_preflight(ORACLE_SELECTION_BUNDLES[SITES[0]], "oracle_weather")
    if RUN_ORACLE else pd.DataFrame()
)
BENCHMARK_PREFLIGHT = pd.concat(
    [STRICT_BENCHMARK_PREFLIGHT, ORACLE_BENCHMARK_PREFLIGHT], ignore_index=True
)
BENCHMARK_PREFLIGHT.to_csv(OUTPUT_DIR / "MUFASA_benchmark_contract_preflight.csv", index=False)
STRICT_DEEP_PREDICTIONS, STRICT_DEEP_HPO, STRICT_DEEP_PROMOTION, STRICT_DEEP_SEEDS, STRICT_BASELINE_SEED_POLICY = (
    run_fair_deep_benchmarks(STRICT_SELECTION_BUNDLES, STRICT_REFIT_BUNDLES, "strict_history")
)
for site in SITES:
    STRICT_MODEL_PREDICTIONS[site].update(STRICT_DEEP_PREDICTIONS.get(site, {}))
if RUN_ORACLE:
    ORACLE_DEEP_PREDICTIONS, ORACLE_DEEP_HPO, ORACLE_DEEP_PROMOTION, ORACLE_DEEP_SEEDS, ORACLE_BASELINE_SEED_POLICY = (
        run_fair_deep_benchmarks(ORACLE_SELECTION_BUNDLES, ORACLE_REFIT_BUNDLES, "oracle_weather")
    )
    for site in SITES:
        ORACLE_MODEL_PREDICTIONS[site].update(ORACLE_DEEP_PREDICTIONS.get(site, {}))
else:
    ORACLE_DEEP_HPO = ORACLE_DEEP_PROMOTION = ORACLE_DEEP_SEEDS = ORACLE_BASELINE_SEED_POLICY = pd.DataFrame()


In [ ]:
# Benchmark Suite and Matched-Budget Optimization — part 5/5
def benchmark_completion_row(scenario, predictions, hpo, promotion, seeds):
    expected_pairs = len(SITES) * len(DEEP_BENCHMARKS)
    expected_trials = expected_pairs * BASELINE_BO_TRIALS
    expected_promotions = expected_pairs * BASELINE_BO_PROMOTE
    expected_seeds = expected_pairs * len(ACTIVE_FINAL_SEEDS)
    successful_trials = int(hpo.get("Status", pd.Series(dtype=str)).eq("ok").sum())
    failed_trials = int(hpo.get("Status", pd.Series(dtype=str)).eq("failed").sum())
    prediction_pairs = sum(
        int(name in predictions.get(site, {})) for site in SITES for name in DEEP_BENCHMARKS
    )
    complete = bool(
        successful_trials == expected_trials and failed_trials == 0
        and len(promotion) == expected_promotions and len(seeds) == expected_seeds
        and prediction_pairs == expected_pairs
    )
    return {
        "Scenario": scenario, "Expected_models_per_site": len(DEEP_BENCHMARKS),
        "Expected_site_model_pairs": expected_pairs, "Prediction_site_model_pairs": prediction_pairs,
        "Expected_HPO_trials": expected_trials, "Successful_HPO_trials": successful_trials,
        "Failed_HPO_trials": failed_trials, "Expected_promotions": expected_promotions,
        "Observed_promotions": len(promotion), "Expected_seed_fits": expected_seeds,
        "Observed_seed_fits": len(seeds), "Benchmark_complete": complete,
        "Paper_rank_eligible": bool(complete and not SMOKE),
    }
completion_rows = [benchmark_completion_row(
    "strict_history", STRICT_DEEP_PREDICTIONS, STRICT_DEEP_HPO,
    STRICT_DEEP_PROMOTION, STRICT_DEEP_SEEDS,
)]
if RUN_ORACLE:
    completion_rows.append(benchmark_completion_row(
        "oracle_weather", ORACLE_DEEP_PREDICTIONS, ORACLE_DEEP_HPO,
        ORACLE_DEEP_PROMOTION, ORACLE_DEEP_SEEDS,
    ))
BENCHMARK_COMPLETION_AUDIT = pd.DataFrame(completion_rows)
BENCHMARK_COMPLETION_AUDIT.to_csv(
    OUTPUT_DIR / "MUFASA_benchmark_completion_audit.csv", index=False
)
BENCHMARK_COMPLETE_BY_SCENARIO = BENCHMARK_COMPLETION_AUDIT.set_index(
    "Scenario"
)["Benchmark_complete"].to_dict()
if REQUIRE_COMPLETE_BENCHMARK:
    incomplete = BENCHMARK_COMPLETION_AUDIT[
        ~BENCHMARK_COMPLETION_AUDIT["Benchmark_complete"]
    ]
    if len(incomplete):
        raise RuntimeError(
            "Publication benchmark is incomplete; ranks/statistical claims are blocked.\n"
            + incomplete.to_string(index=False)
        )
def benchmark_table(model_predictions, results, scenario):
    rows = []
    for site, models in model_predictions.items():
        truth = results[site]["truth"]
        for model, prediction in models.items():
            rows.append({"Scenario": scenario, "Site": site, "Model": model, **metric_set(truth, prediction)})
    frame = pd.DataFrame(rows)
    frame["Rank_RMSE"] = frame.groupby("Site")["RMSE"].rank(method="min")
    frame["Rank_MAE"] = frame.groupby("Site")["MAE"].rank(method="min")
    
    internal = {"MUFASA-Core", "Ridge-Residual", "Solar-Geometry-Reference"}
    external_scope = frame[~frame["Model"].isin(internal)].copy()
    external_scope["External_Rank_RMSE"] = external_scope.groupby("Site")["RMSE"].rank(method="min")
    external_scope["External_Rank_MAE"] = external_scope.groupby("Site")["MAE"].rank(method="min")
    frame = frame.merge(
        external_scope[["Site", "Model", "External_Rank_RMSE", "External_Rank_MAE"]],
        on=["Site", "Model"], how="left",
    )
    frame["Benchmark_complete"] = bool(BENCHMARK_COMPLETE_BY_SCENARIO.get(scenario, False))
    frame["Paper_rank_eligible"] = bool(
        BENCHMARK_COMPLETE_BY_SCENARIO.get(scenario, False) and not SMOKE
    )
    return frame.sort_values(["Site", "Rank_RMSE", "Model"]).reset_index(drop=True)
STRICT_TEST_BENCHMARK = benchmark_table(STRICT_MODEL_PREDICTIONS, STRICT_RESULTS, "strict_history")
STRICT_TEST_BENCHMARK.to_csv(OUTPUT_DIR / "MUFASA_strict_test_benchmark.csv", index=False)
if RUN_ORACLE:
    ORACLE_TEST_BENCHMARK = benchmark_table(ORACLE_MODEL_PREDICTIONS, ORACLE_RUN["results"], "oracle_weather")
else:
    ORACLE_TEST_BENCHMARK = pd.DataFrame(columns=STRICT_TEST_BENCHMARK.columns)
ORACLE_TEST_BENCHMARK.to_csv(OUTPUT_DIR / "MUFASA_oracle_test_benchmark.csv", index=False)
def benchmark_summary(frame, scenario):
    proposed = frame[frame["Model"].eq("MUFASA")]
    site_rows = []
    for site, row in proposed.set_index("Site").iterrows():
        comparators = frame[(frame["Site"].eq(site)) & (~frame["Model"].isin(["MUFASA", "MUFASA-Core", "Ridge-Residual", "Solar-Geometry-Reference"]))]
        best = comparators.loc[comparators["RMSE"].idxmin()] if len(comparators) else row
        site_rows.append({"Site": site, "Improvement_vs_best_percent": 100 * (best["RMSE"] - row["RMSE"]) / max(best["RMSE"], 1e-12),
                          "Best_comparator": best["Model"]})
    gains = pd.DataFrame(site_rows)
    return pd.DataFrame([{
        "Scenario": scenario,
        "Benchmark_complete": bool(BENCHMARK_COMPLETE_BY_SCENARIO.get(scenario, False)),
        "Paper_rank_eligible": bool(BENCHMARK_COMPLETE_BY_SCENARIO.get(scenario, False) and not SMOKE),
        "Macro_average_RMSE": proposed["RMSE"].mean(),
        "Macro_average_MAE": proposed["MAE"].mean(),
        "Macro_average_R2": proposed["R2"].mean(),
        "Six_city_rank_average": proposed["External_Rank_RMSE"].mean(),
        "Number_rank_1_cities": int((proposed["External_Rank_RMSE"] == 1).sum()),
        "Number_top_2_cities": int((proposed["External_Rank_RMSE"] <= 2).sum()),
        "Average_improvement_vs_best_comparator_percent": gains["Improvement_vs_best_percent"].mean(),
        "Worst_city_relative_deficit_percent": -min(gains["Improvement_vs_best_percent"].min(), 0.0),
        "Worst_deficit_city": gains.loc[gains["Improvement_vs_best_percent"].idxmin(), "Site"],
    }])
BENCHMARK_SUMMARY = pd.concat([
    benchmark_summary(STRICT_TEST_BENCHMARK, "strict_history"),
    benchmark_summary(ORACLE_TEST_BENCHMARK, "oracle_weather") if len(ORACLE_TEST_BENCHMARK) else pd.DataFrame(),
], ignore_index=True)
BENCHMARK_SUMMARY.to_csv(OUTPUT_DIR / "MUFASA_benchmark_summary.csv", index=False)
FAIR_BUDGET_AUDIT = pd.DataFrame([
    {"Family": "MUFASA", "BO_trials": FINAL_BO_TRIALS, "Low_epochs": BO_LOW_EPOCHS,
     "Promotions": BO_PROMOTE, "Final_epoch_cap": FINAL_MAX_EPOCHS, "Seeds": "11|29|47"},
    {"Family": "Each GPU comparator", "BO_trials": BASELINE_BO_TRIALS, "Low_epochs": BASELINE_BO_LOW_EPOCHS,
     "Promotions": BASELINE_BO_PROMOTE, "Final_epoch_cap": BENCHMARK_MAX_EPOCHS, "Seeds": "11|29|47"},
])
FAIR_BUDGET_AUDIT["Equal_budget"] = True
FAIR_BUDGET_AUDIT.to_csv(OUTPUT_DIR / "MUFASA_fair_HPO_budget_audit.csv", index=False)
BENCHMARK_FRAME_SCHEMAS = {
    "MUFASA_strict_deep_HPO.csv": ["Scenario", "Site", "Model", "Trial", "Status", "Error"],
    "MUFASA_strict_deep_promotion.csv": ["Scenario", "Site", "Model", "Promotion_rank"],
    "MUFASA_strict_deep_seed_results.csv": ["Scenario", "Site", "Model", "Seed"],
    "MUFASA_oracle_deep_HPO.csv": ["Scenario", "Site", "Model", "Trial", "Status", "Error"],
    "MUFASA_oracle_deep_promotion.csv": ["Scenario", "Site", "Model", "Promotion_rank"],
    "MUFASA_oracle_deep_seed_results.csv": ["Scenario", "Site", "Model", "Seed"],
    "MUFASA_strict_deep_seed_policy_audit.csv": ["Scenario", "Site", "Horizon", "Weighted_active"],
    "MUFASA_oracle_deep_seed_policy_audit.csv": ["Scenario", "Site", "Horizon", "Weighted_active"],
}
for frame, filename in [
    (STRICT_DEEP_HPO, "MUFASA_strict_deep_HPO.csv"),
    (STRICT_DEEP_PROMOTION, "MUFASA_strict_deep_promotion.csv"),
    (STRICT_DEEP_SEEDS, "MUFASA_strict_deep_seed_results.csv"),
    (ORACLE_DEEP_HPO, "MUFASA_oracle_deep_HPO.csv"),
    (ORACLE_DEEP_PROMOTION, "MUFASA_oracle_deep_promotion.csv"),
    (ORACLE_DEEP_SEEDS, "MUFASA_oracle_deep_seed_results.csv"),
    (STRICT_BASELINE_SEED_POLICY, "MUFASA_strict_deep_seed_policy_audit.csv"),
    (ORACLE_BASELINE_SEED_POLICY, "MUFASA_oracle_deep_seed_policy_audit.csv"),
]:
    if frame.empty:
        frame = pd.DataFrame(columns=BENCHMARK_FRAME_SCHEMAS[filename])
    frame.to_csv(OUTPUT_DIR / filename, index=False)
retry_frame = pd.DataFrame(BENCHMARK_RETRY_ROWS)
if retry_frame.empty:
    retry_frame = pd.DataFrame(columns=[
        "Site", "Model", "Seed", "Attempt", "Batch_size", "AMP_override", "Status", "Error"
    ])
retry_frame.to_csv(
    OUTPUT_DIR / "MUFASA_benchmark_retry_audit.csv", index=False
)
display(BENCHMARK_SUMMARY.round(5))


In [ ]:
# Benchmark Provenance Audit
legacy_rows = []
for model in ("Legacy-GRU", "Legacy-Attention-BiLSTM"):
    legacy_rows.append({
        "Model": model, "Reproduction": "Approximate reproduction",
        "Reason": "Original source did not fully specify every current-split hyperparameter.",
        "Strict_executed": bool(
            BENCHMARK_COMPLETE_BY_SCENARIO.get("strict_history", False)
            and all(model in STRICT_DEEP_PREDICTIONS.get(site, {}) for site in SITES)
        ),
        "Oracle_executed": bool(
            RUN_ORACLE and BENCHMARK_COMPLETE_BY_SCENARIO.get("oracle_weather", False)
            and all(model in ORACLE_DEEP_PREDICTIONS.get(site, {}) for site in SITES)
        ),
        "Native_paper_score_reused": False,
    })
LEGACY_AUDIT = pd.DataFrame(legacy_rows)
LEGACY_AUDIT.to_csv(OUTPUT_DIR / "MUFASA_legacy_reproduction_audit.csv", index=False)
display(LEGACY_AUDIT)


## 7. Ablation, diagnostics, inference, and XAI

Run component ablations, horizon/regime diagnostics, dependence-aware inference, grouped XAI, and fail-fast leakage/protocol audits.

**Run note.** Execute the cells in this section in order. Objects created here are consumed by later sections; the notebook intentionally avoids hidden state restoration from unpublished artifacts.


In [ ]:
# Ablation Analysis
def core_ablation_definitions(base_config):
    return {
        "C1 Compact (No BiGRU/LDS/RevIN)": candidate_config("C1-MUFASA-Compact", base_config),
        "C2 Canonical + Historical BiGRU": candidate_config("C2-MUFASA-BiGRU", base_config),
        "C3 + LDS/Tail (No BiGRU)": candidate_config("C3-MUFASA-Tail", base_config),
        "C4 C2 + LDS/Tail": candidate_config("C4-MUFASA-BiGRU-Tail", base_config),
        "C5 C4 + selective RevIN": candidate_config("C5-MUFASA-Full", base_config),
        "C2 No NLinear": replace(candidate_config("C2-MUFASA-BiGRU", base_config), use_nlinear=False),
        "C2 Direct only (No AR)": replace(candidate_config("C2-MUFASA-BiGRU", base_config), use_autoregressive=False),
        "C2 No scheduled TF": replace(candidate_config("C2-MUFASA-BiGRU", base_config), teacher_forcing_start=0.0),
        "C2 No solar-geometry reference": replace(candidate_config("C2-MUFASA-BiGRU", base_config), use_solar_reference=False),
        "C2 No engineered features": replace(candidate_config("C2-MUFASA-BiGRU", base_config), use_engineered=False),
        "C2 Minimal features": replace(candidate_config("C2-MUFASA-BiGRU", base_config), engineered_feature_mode="minimal_compact"),
        "C2 Extended features": replace(candidate_config("C2-MUFASA-BiGRU", base_config), engineered_feature_mode="extended_compact"),
    }


def fit_ablation_core_cache(site, label, config):
    selection_bundle = STRICT_SELECTION_BUNDLES[site]
    refit_bundle = STRICT_REFIT_BUNDLES[site]
    train_idx, val_idx = positions(selection_bundle.train), positions(selection_bundle.val_all)
    fit_idx, test_idx = positions(refit_bundle.train_val), positions(refit_bundle.test)
    val_predictions, test_predictions, trained_selection = [], [], []
    started = time.time()
    for seed in ACTIVE_FINAL_SEEDS:
        selected = fit_mufasa(
            selection_bundle, train_idx, val_idx, config, seed,
            fixed_epochs=STRICT_FIXED_EPOCHS[site],
        )
        val_predictions.append(predict_mufasa(selected, selection_bundle, val_idx, mc_passes=1))
        trained_selection.append(selected)
        refit = fit_mufasa(
            refit_bundle, fit_idx, np.array([], int), config, seed,
            fixed_epochs=STRICT_FIXED_EPOCHS[site],
        )
        test_predictions.append(predict_mufasa(refit, refit_bundle, test_idx, mc_passes=1))
    val_cube = np.stack(val_predictions, axis=-1)
    test_cube = np.stack(test_predictions, axis=-1)
    equal_weights = np.full((N_HORIZONS, val_cube.shape[-1]), 1.0 / val_cube.shape[-1])
    profile = model_parameter_profile(trained_selection[0])
    return {
        "config": config, "val_cube": val_cube, "test_cube": test_cube,
        "equal_val": apply_seed_policy(val_cube, equal_weights),
        "equal_test": apply_seed_policy(test_cube, equal_weights),
        "equal_weights": equal_weights, "profile": profile,
        "seconds": time.time() - started,
    }


def run_separated_ablations():
    core_rows, system_rows = [], []
    for site in SITES:
        selection_bundle, refit_bundle = STRICT_SELECTION_BUNDLES[site], STRICT_REFIT_BUNDLES[site]
        val_idx, test_idx = positions(selection_bundle.val_all), positions(refit_bundle.test)
        train_idx, fit_idx = positions(selection_bundle.train), positions(refit_bundle.train_val)
        truth_val, truth_test = selection_bundle.target[val_idx], refit_bundle.target[test_idx]
        dates_val = selection_bundle.target_dates[val_idx]
        definitions = core_ablation_definitions(STRICT_SELECTED_CONFIGS[site])
        cache = {}
        for label, config in definitions.items():
            artifact = fit_ablation_core_cache(site, label, config)
            cache[label] = artifact
            core_rows.append({
                "Site": site, "Ablation_family": "Core-only",
                "Ablation": label, "Scenario": "strict_history",
                "Seeds": "|".join(map(str, ACTIVE_FINAL_SEEDS)),
                "Seed_policy": "equal_mean_to_isolate_architecture",
                "Ridge_used": False, "Solar_geometry_reference_used": False,
                "TailGuard_used": False, "Fusion_used": False,
                **artifact["profile"], "Training_seconds": artifact["seconds"],
                "Selection_uses_test": False, **metric_set(truth_test, artifact["equal_test"]),
            })

        canonical = cache["C2 Canonical + Historical BiGRU"]
        val_cube, test_cube = canonical["val_cube"], canonical["test_cube"]
        evidence_weights, _ = fit_seed_policy(
            val_cube, truth_val, dates_val, site, "system_ablation:C2"
        )
        equal_val, equal_test = canonical["equal_val"], canonical["equal_test"]
        weighted_val = apply_seed_policy(val_cube, evidence_weights)
        weighted_test = apply_seed_policy(test_cube, evidence_weights)
        std_val = val_cube.std(axis=-1, ddof=1) if val_cube.shape[-1] > 1 else np.zeros_like(weighted_val)
        std_test = test_cube.std(axis=-1, ddof=1) if test_cube.shape[-1] > 1 else np.zeros_like(weighted_test)
        ridge_selection = fit_ridge_residual(selection_bundle, train_idx, STRICT_RIDGE_CONFIGS[site])
        ridge_val = predict_ridge_residual(ridge_selection, selection_bundle, val_idx)
        solar_ref_val = predict_anchor(ridge_selection, selection_bundle, val_idx)
        ridge_final = fit_ridge_residual(refit_bundle, fit_idx, STRICT_RIDGE_CONFIGS[site])
        ridge_test = predict_ridge_residual(ridge_final, refit_bundle, test_idx)
        solar_ref_test = predict_anchor(ridge_final, refit_bundle, test_idx)

        def append_system(label, prediction, mode, sources, seed_policy="evidence_gated"):
            system_rows.append({
                "Site": site, "Ablation_family": "End-to-End system",
                "Ablation": label, "Scenario": "strict_history",
                "Seeds": "|".join(map(str, ACTIVE_FINAL_SEEDS)),
                "Seed_policy": seed_policy, "Source_mode": mode,
                "Selected_sources": "|".join(sources),
                **canonical["profile"], "Training_seconds": canonical["seconds"],
                "Selection_uses_test": False, **metric_set(truth_test, prediction),
            })

        append_system("C2 equal-seed Core only", equal_test, "CoreOnly", ["MUFASA-Core"], "equal_mean")
        append_system("C2 evidence-seed Core only", weighted_test, "CoreOnly", ["MUFASA-Core"])

        tail_policy, _ = fit_tail_guard_policy(
            weighted_val, std_val, ridge_val, solar_ref_val, truth_val, dates_val,
            site, "system_ablation:optional_tail", True,
        )
        guarded_val, _ = apply_tail_guard(tail_policy, weighted_val, std_val, ridge_val, solar_ref_val)
        guarded_test, _ = apply_tail_guard(tail_policy, weighted_test, std_test, ridge_test, solar_ref_test)
        append_system(
            "C2 + evidence-gated uncertainty protection", guarded_test,
            "TailGuard" if any(item.get("active", False) for item in tail_policy) else "TailGuardFallbackOff",
            ["MUFASA-Core", "Robust-source"],
        )

        for label, val_sources, test_sources, force_winner in [
            ("C2 + Ridge StableWinner", {"MUFASA-Core": weighted_val, "Ridge-Residual": ridge_val},
             {"MUFASA-Core": weighted_test, "Ridge-Residual": ridge_test}, True),
            ("C2 + Ridge + Solar Reference", {"MUFASA-Core": weighted_val, "Ridge-Residual": ridge_val, "Solar-Geometry-Reference": solar_ref_val},
             {"MUFASA-Core": weighted_test, "Ridge-Residual": ridge_test, "Solar-Geometry-Reference": solar_ref_test}, True),
            ("Full Stable Winner-or-Fusion", {"MUFASA-Core": guarded_val, "Ridge-Residual": ridge_val, "Solar-Geometry-Reference": solar_ref_val},
             {"MUFASA-Core": guarded_test, "Ridge-Residual": ridge_test, "Solar-Geometry-Reference": solar_ref_test}, False),
            ("No Ridge expert", {"MUFASA-Core": guarded_val, "Solar-Geometry-Reference": solar_ref_val},
             {"MUFASA-Core": guarded_test, "Solar-Geometry-Reference": solar_ref_test}, False),
            ("No Solar-geometry reference", {"MUFASA-Core": guarded_val, "Ridge-Residual": ridge_val},
             {"MUFASA-Core": guarded_test, "Ridge-Residual": ridge_test}, False),
        ]:
            policy, _, _ = fit_winner_or_fusion(val_sources, truth_val, dates_val, site, f"system_ablation:{label}")
            if force_winner:
                weights = np.zeros(len(policy["source_names"]))
                weights[policy["source_names"].index(policy["winner"])] = 1.0
                policy.update({"weights": weights, "mode": "StableWinner-forced"})
            prediction = apply_source_weights(test_sources, policy["source_names"], policy["weights"])
            append_system(label, prediction, policy["mode"], policy["source_names"])
    return pd.DataFrame(core_rows), pd.DataFrame(system_rows)


if RUN_ABLATIONS:
    CORE_ONLY_ABLATION, END_TO_END_ABLATION = run_separated_ablations()
else:
    CORE_ONLY_ABLATION = pd.DataFrame([{
        "Site": "ALL", "Ablation_family": "Core-only", "Ablation": "not executed",
        "Reason": "MUFASA_RUN_ABLATION=0; paper completeness remains false",
    }])
    END_TO_END_ABLATION = pd.DataFrame([{
        "Site": "ALL", "Ablation_family": "End-to-End system", "Ablation": "not executed",
        "Reason": "MUFASA_RUN_ABLATION=0; paper completeness remains false",
    }])
CORE_ONLY_ABLATION.to_csv(OUTPUT_DIR / "MUFASA_core_only_ablation.csv", index=False)
END_TO_END_ABLATION.to_csv(OUTPUT_DIR / "MUFASA_end_to_end_ablation.csv", index=False)
CLEAN_ABLATION = pd.concat([CORE_ONLY_ABLATION, END_TO_END_ABLATION], ignore_index=True, sort=False)
CLEAN_ABLATION.to_csv(OUTPUT_DIR / "MUFASA_clean_ablation.csv", index=False)
display(CORE_ONLY_ABLATION.head(20).round(5))
display(END_TO_END_ABLATION.head(20).round(5))


In [ ]:
# Horizon and Regime Diagnostics
HORIZON_ROWS, ACTUAL_REGIME_ROWS, PREDICTED_REGIME_ROWS, TAIL_SUBSET_ROWS = [], [], [], []
REGIME_CONFUSION_ROWS, HORIZON_FUSION_ROWS = [], []
for site, result in STRICT_RESULTS.items():
    truth, dates = result["truth"], result["dates"]
    benchmark_site = STRICT_TEST_BENCHMARK[STRICT_TEST_BENCHMARK["Site"].eq(site)]
    comparator_rows = benchmark_site[~benchmark_site["Model"].isin([
        "MUFASA", "MUFASA-Core", "Ridge-Residual", "Solar-Geometry-Reference"
    ])]
    best_comparator = comparator_rows.sort_values("RMSE").iloc[0]["Model"] if len(comparator_rows) else "Persistence-1d"
    horizon_models = ["MUFASA", best_comparator, "Ridge-Residual", "MUFASA-Core", "Solar-Geometry-Reference"]
    for model in dict.fromkeys(horizon_models):
        prediction = STRICT_MODEL_PREDICTIONS[site][model]
        for horizon, hour in enumerate(TARGET_HOURS):
            values = metric_set(truth[:, horizon], prediction[:, horizon])
            best_values = metric_set(truth[:, horizon], STRICT_MODEL_PREDICTIONS[site][best_comparator][:, horizon])
            HORIZON_ROWS.append({
                "Site": site, "Model": model, "Horizon": horizon + 1, "Hour": int(hour),
                "Gain_vs_best_comparator_RMSE_percent": 100 * (best_values["RMSE"] - values["RMSE"]) / max(best_values["RMSE"], 1e-12),
                **values,
            })
    for horizon, hour in enumerate(TARGET_HOURS):
        final_rmse = metric_set(truth[:, horizon], result["prediction"][:, horizon])["RMSE"]
        source_values = {
            "MUFASA-Core": metric_set(truth[:, horizon], result["core"][:, horizon])["RMSE"],
            "Ridge-Residual": metric_set(truth[:, horizon], result["ridge"][:, horizon])["RMSE"],
            "Solar-Geometry-Reference": metric_set(truth[:, horizon], result["prior"][:, horizon])["RMSE"],
        }
        best_source = min(source_values, key=source_values.get)
        HORIZON_FUSION_ROWS.append({
            "Site": site, "Horizon": horizon + 1, "Hour": int(hour),
            "Final_RMSE": final_rmse, "Best_single_source": best_source,
            "Best_single_RMSE": source_values[best_source],
            "Fusion_minus_best_single_RMSE": final_rmse - source_values[best_source],
            "Fusion_hurts_on_locked_test_diagnostic": final_rmse > source_values[best_source],
            "Used_for_selection": False,
        })

    bundle = STRICT_REFIT_BUNDLES[site]; fit_idx = positions(bundle.train_val); test_idx = positions(bundle.test)
    thresholds = _fit_regime_thresholds(bundle, fit_idx)
    actual_labels = _regime_labels(bundle, test_idx, thresholds)
    predicted_labels = np.argmax(result["regime_probabilities"], axis=1)
    for actual_index, actual_name in enumerate(REGIME_NAMES):
        for predicted_index, predicted_name in enumerate(REGIME_NAMES):
            mask = (actual_labels == actual_index) & (predicted_labels == predicted_index)
            if np.any(mask):
                REGIME_CONFUSION_ROWS.append({
                    "Site": site, "Actual_regime": actual_name,
                    "Predicted_regime": predicted_name, "Days": int(mask.sum()),
                    "Correct_route": actual_index == predicted_index,
                    **metric_set(truth[mask], result["prediction"][mask]),
                })
    for diagnostic, labels, container in [
        ("actual_SVI_posthoc", actual_labels, ACTUAL_REGIME_ROWS),
        ("predicted_regime_deployable", predicted_labels, PREDICTED_REGIME_ROWS),
    ]:
        for label_index, label_name in enumerate(REGIME_NAMES):
            mask = labels == label_index
            if np.any(mask):
                container.append({"Site": site, "Diagnostic": diagnostic, "Regime": label_name,
                                  "Days": int(mask.sum()), **metric_set(truth[mask], result["prediction"][mask])})
    tail_mask = np.max(result["tail_gate"], axis=1) > 1e-10
    for active in (False, True):
        mask = tail_mask == active
        if np.any(mask):
            TAIL_SUBSET_ROWS.append({"Site": site, "Tail_gate_active": active, "Days": int(mask.sum()),
                                     **metric_set(truth[mask], result["prediction"][mask])})

HORIZON_METRICS = pd.DataFrame(HORIZON_ROWS)
ACTUAL_REGIME_DIAGNOSTIC = pd.DataFrame(ACTUAL_REGIME_ROWS)
PREDICTED_REGIME_DIAGNOSTIC = pd.DataFrame(PREDICTED_REGIME_ROWS)
TAIL_SUBSET_DIAGNOSTIC = pd.DataFrame(TAIL_SUBSET_ROWS)
REGIME_CONFUSION_DIAGNOSTIC = pd.DataFrame(REGIME_CONFUSION_ROWS)
HORIZON_FUSION_DIAGNOSTIC = pd.DataFrame(HORIZON_FUSION_ROWS)
HORIZON_METRICS.to_csv(OUTPUT_DIR / "MUFASA_horizon_diagnostics.csv", index=False)
ACTUAL_REGIME_DIAGNOSTIC.to_csv(OUTPUT_DIR / "MUFASA_actual_SVI_posthoc_diagnostic.csv", index=False)
PREDICTED_REGIME_DIAGNOSTIC.to_csv(OUTPUT_DIR / "MUFASA_predicted_regime_diagnostic.csv", index=False)
TAIL_SUBSET_DIAGNOSTIC.to_csv(OUTPUT_DIR / "MUFASA_tail_gate_subset_diagnostic.csv", index=False)
REGIME_CONFUSION_DIAGNOSTIC.to_csv(OUTPUT_DIR / "MUFASA_regime_confusion_conditioned_errors.csv", index=False)
HORIZON_FUSION_DIAGNOSTIC.to_csv(OUTPUT_DIR / "MUFASA_horizon_fusion_effect.csv", index=False)


In [ ]:
# Dependence-Aware Statistical Inference
# 30. Paired Wilcoxon·Newey-West HAC·generalized DM·block bootstrap·Holm


def newey_west_mean_test(differences, lag=7):
    values = np.asarray(differences, float)
    values = values[np.isfinite(values)]
    n = len(values)
    if n < 10:
        return np.nan, np.nan
    centered = values - values.mean()
    long_run = float(centered @ centered / n)
    for order in range(1, min(lag, n - 1) + 1):
        covariance = float(centered[order:] @ centered[:-order] / n)
        long_run += 2.0 * (1.0 - order / (lag + 1.0)) * covariance
    standard_error = math.sqrt(max(long_run, 1e-14) / n)
    statistic = float(values.mean() / standard_error)
    return statistic, float(2.0 * (1.0 - norm.cdf(abs(statistic))))


def newey_west_long_run_covariance(matrix, lag=7):
    values = np.asarray(matrix, float)
    values = values[np.isfinite(values).all(axis=1)]
    n = len(values)
    if n < max(20, values.shape[1] + 2):
        return None
    centered = values - values.mean(axis=0, keepdims=True)
    covariance = centered.T @ centered / n
    for order in range(1, min(lag, n - 1) + 1):
        gamma = centered[order:].T @ centered[:-order] / n
        weight = 1.0 - order / (lag + 1.0)
        covariance += weight * (gamma + gamma.T)
    return covariance


def generalized_multihorizon_dm(loss_differential, lag=7):
    values = np.asarray(loss_differential, float)
    values = values[np.isfinite(values).all(axis=1)]
    covariance = newey_west_long_run_covariance(values, lag=lag)
    if covariance is None:
        return np.nan, np.nan, 0
    mean_vector = values.mean(axis=0)
    covariance = covariance + 1e-10 * np.eye(covariance.shape[0])
    statistic = float(len(values) * mean_vector @ np.linalg.pinv(covariance) @ mean_vector)
    rank = int(np.linalg.matrix_rank(covariance))
    return statistic, float(chi2.sf(statistic, max(rank, 1))), rank


def moving_block_mean_ci(differences, block_length=7, repetitions=BOOTSTRAP_REPS, seed=SEED):
    values = np.asarray(differences, float)
    values = values[np.isfinite(values)]
    n = len(values)
    if n == 0:
        return np.nan, np.nan, np.nan
    block_length = min(block_length, n)
    starts = np.arange(0, n - block_length + 1)
    generator = np.random.default_rng(seed)
    boot = np.empty(repetitions, float)
    blocks_needed = int(math.ceil(n / block_length))
    for repetition in range(repetitions):
        selected = generator.choice(starts, size=blocks_needed, replace=True)
        sample = np.concatenate([values[start:start + block_length] for start in selected])[:n]
        boot[repetition] = sample.mean()
    return float(values.mean()), float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))


def holm_adjust(p_values):
    values = np.asarray(list(p_values), float)
    adjusted = np.full(len(values), np.nan)
    finite = np.where(np.isfinite(values))[0]
    if not len(finite):
        return adjusted
    order = finite[np.argsort(values[finite])]
    running, m = 0.0, len(order)
    for rank, index in enumerate(order):
        candidate = min(1.0, (m - rank) * values[index])
        running = max(running, candidate)
        adjusted[index] = running
    return adjusted


def run_statistical_comparisons(benchmark, predictions, results, scenario):
    rows = []
    for site, result in results.items():
        truth = result["truth"]; dates = result["dates"]
        proposed = predictions[site]["MUFASA"]
        for comparator, comparison in predictions[site].items():
            if comparator == "MUFASA":
                continue
            horizon_diff = (proposed - truth) ** 2 - (comparison - truth) ** 2
            daily_diff = horizon_diff.mean(axis=1)
            try:
                wilcoxon_p = float(wilcoxon(daily_diff, zero_method="zsplit", alternative="less").pvalue)
            except Exception:
                wilcoxon_p = 1.0
            hac_stat, hac_p = newey_west_mean_test(daily_diff, lag=7)
            dm_stat, dm_p, dm_rank = generalized_multihorizon_dm(horizon_diff, lag=7)
            block_mean, ci_low, ci_high = moving_block_mean_ci(
                daily_diff, block_length=7, repetitions=BLOCK_BOOTSTRAP_REPS,
                seed=SEED + sum(map(ord, site + comparator)),
            )
            proposed_metrics = metric_set(truth, proposed)
            comparator_metrics = metric_set(truth, comparison)
            rows.append({
                "Scenario": scenario, "Site": site, "Comparator": comparator,
                "Comparator_family": (
                    "Internal expert" if comparator in {
                        "MUFASA-Core", "Ridge-Residual", "Solar-Geometry-Reference"
                    } else "External benchmark"
                ),
                "MUFASA_RMSE": proposed_metrics["RMSE"], "Comparator_RMSE": comparator_metrics["RMSE"],
                "RMSE_reduction_percent": 100 * (comparator_metrics["RMSE"] - proposed_metrics["RMSE"]) / max(comparator_metrics["RMSE"], 1e-12),
                "MUFASA_MAE": proposed_metrics["MAE"], "Comparator_MAE": comparator_metrics["MAE"],
                "Mean_daily_squared_loss_difference": float(daily_diff.mean()),
                "Wilcoxon_p_raw": wilcoxon_p, "HAC_stat": hac_stat, "HAC_p_raw": hac_p,
                "GDM_stat": dm_stat, "GDM_p_raw": dm_p, "GDM_covariance_rank": dm_rank,
                "Block_CI_low": ci_low, "Block_CI_high": ci_high,
                "Daily_win_rate": float(np.mean(daily_diff < 0)),
            })
    frame = pd.DataFrame(rows)
    for scenario_name in frame["Scenario"].unique():
        for site in frame[frame["Scenario"].eq(scenario_name)]["Site"].unique():
            mask = frame["Scenario"].eq(scenario_name) & frame["Site"].eq(site)
            for raw, adjusted in [("Wilcoxon_p_raw", "Wilcoxon_p_Holm"),
                                  ("HAC_p_raw", "HAC_p_Holm"), ("GDM_p_raw", "GDM_p_Holm")]:
                frame.loc[mask, adjusted] = holm_adjust(frame.loc[mask, raw].to_numpy())
    frame["Numeric_superiority"] = frame["RMSE_reduction_percent"] > 0
    frame["Statistically_supported_superiority"] = (
        frame["Numeric_superiority"] & (frame["HAC_p_Holm"] < 0.05)
        & (frame["GDM_p_Holm"] < 0.05) & (frame["Block_CI_high"] < 0)
    )
    frame["Material_improvement_ge_3pct"] = frame["RMSE_reduction_percent"] >= 3
    frame["Dominant_improvement_ge_5pct"] = frame["RMSE_reduction_percent"] >= 5
    return frame


STRICT_STATISTICS = run_statistical_comparisons(
    STRICT_TEST_BENCHMARK, STRICT_MODEL_PREDICTIONS, STRICT_RESULTS, "strict_history"
)
if RUN_ORACLE:
    ORACLE_STATISTICS = run_statistical_comparisons(
        ORACLE_TEST_BENCHMARK, ORACLE_MODEL_PREDICTIONS, ORACLE_RUN["results"], "oracle_weather"
    )
else:
    ORACLE_STATISTICS = pd.DataFrame(columns=STRICT_STATISTICS.columns)
STATISTICAL_TESTS = pd.concat([STRICT_STATISTICS, ORACLE_STATISTICS], ignore_index=True)
STATISTICAL_TESTS.to_csv(OUTPUT_DIR / "MUFASA_statistical_tests.csv", index=False)
EXTERNAL_STATISTICAL_TESTS = STATISTICAL_TESTS[
    STATISTICAL_TESTS["Comparator_family"].eq("External benchmark")
].copy()
INTERNAL_EXPERT_STATISTICAL_TESTS = STATISTICAL_TESTS[
    STATISTICAL_TESTS["Comparator_family"].eq("Internal expert")
].copy()
EXTERNAL_STATISTICAL_TESTS.to_csv(
    OUTPUT_DIR / "MUFASA_external_baseline_statistical_tests.csv", index=False
)
INTERNAL_EXPERT_STATISTICAL_TESTS.to_csv(
    OUTPUT_DIR / "MUFASA_internal_expert_statistical_tests.csv", index=False
)

CLAIM_ROWS = []
for scenario, benchmark in [("strict_history", STRICT_TEST_BENCHMARK), ("oracle_weather", ORACLE_TEST_BENCHMARK)]:
    if not len(benchmark):
        continue
    proposed = benchmark[benchmark["Model"].eq("MUFASA")]
    tests = EXTERNAL_STATISTICAL_TESTS[
        EXTERNAL_STATISTICAL_TESTS["Scenario"].eq(scenario)
    ]
    benchmark_complete = bool(BENCHMARK_COMPLETE_BY_SCENARIO.get(scenario, False))
    CLAIM_ROWS.append({
        "Benchmark_complete": benchmark_complete,
        "Paper_claim_eligible": bool(benchmark_complete and not SMOKE),
        "Scenario": scenario,
        "Numeric_rank1_every_city": bool(benchmark_complete and (proposed["External_Rank_RMSE"] == 1).all()),
        "All_comparator_statistical_support": bool(benchmark_complete and len(tests) and tests["Statistically_supported_superiority"].all()),
        "Material_gain_every_comparison": bool(benchmark_complete and len(tests) and tests["Material_improvement_ge_3pct"].all()),
        "Dominant_gain_every_comparison": bool(benchmark_complete and len(tests) and tests["Dominant_improvement_ge_5pct"].all()),
    })
SUPERIORITY_CLAIM_AUDIT = pd.DataFrame(CLAIM_ROWS)
SUPERIORITY_CLAIM_AUDIT.to_csv(OUTPUT_DIR / "MUFASA_superiority_claim_audit.csv", index=False)
display(SUPERIORITY_CLAIM_AUDIT)


In [ ]:
# End-to-End Explainability
# 31. Multi-view XAI — end-to-end grouped permutation, IG, exact LinearSHAP, gates
def permute_bundle_group(bundle, indices, group, permutation):
    sequence = bundle.sequence.copy(); astronomy = bundle.astronomy.copy()
    archive = bundle.archived_weather.copy(); tabular = bundle.tabular.copy()
    families = np.asarray(bundle.tabular_families).astype(str)
    if group == "Recent solar state/shape/energy":
        sequence[indices, ..., :4] = sequence[indices[permutation], ..., :4]
        mask = np.array(["solar_" in name or "clearness" in name for name in families])
    elif group == "Completed-day weather":
        sequence[indices, ..., 4:] = sequence[indices[permutation], ..., 4:]
        mask = np.array(["weather" in name and "future" not in name for name in families])
    elif group == "Target astronomy/calendar":
        astronomy[indices] = astronomy[indices[permutation]]
        mask = np.array(["target_astronomy" in name or "target_calendar" in name for name in families])
    elif group == "Forward weather covariates":
        archive[indices] = archive[indices[permutation]]
        mask = np.array(["future_weather_covariates" in name for name in families])
    else:
        raise KeyError(group)
    columns = np.where(mask)[0]
    if len(columns):
        tabular[np.ix_(indices, columns)] = tabular[np.ix_(indices[permutation], columns)]
    return replace(bundle, sequence=sequence, astronomy=astronomy,
                   archived_weather=archive, tabular=tabular)


def predict_locked_system_on_bundle(site, bundle, indices, models, ridge_expert,
                                    seed_weights, tail_policy, source_policy):
    seed_predictions = [predict_mufasa(model, bundle, indices, mc_passes=1) for model in models]
    cube = np.stack(seed_predictions, axis=-1)
    core = apply_seed_policy(cube, seed_weights)
    seed_std = cube.std(axis=-1, ddof=1) if cube.shape[-1] > 1 else np.zeros_like(core)
    ridge = predict_ridge_residual(ridge_expert, bundle, indices)
    prior = predict_anchor(ridge_expert, bundle, indices)
    core, _ = apply_tail_guard(tail_policy, core, seed_std, ridge, prior)
    sources = {"MUFASA-Core": core, "Ridge-Residual": ridge, "Solar-Geometry-Reference": prior}
    return apply_source_weights(sources, source_policy["source_names"], source_policy["weights"])


PERMUTATION_ROWS, IG_ROWS, LINEAR_SHAP_ROWS = [], [], []
BIGRU_GATE_ROWS, TAIL_GATE_ROWS, SOURCE_WEIGHT_ROWS = [], [], []
if RUN_XAI:
    for site, bundle in STRICT_REFIT_BUNDLES.items():
        result = STRICT_RESULTS[site]; full_idx = result["test_idx"]
        test_idx = full_idx[::7] if SMOKE else full_idx
        truth = bundle.target[test_idx]
        baseline = predict_locked_system_on_bundle(
            site, bundle, test_idx, STRICT_FINAL_MODELS[site], STRICT_FINAL_RIDGE[site],
            STRICT_SEED_WEIGHTS[site], STRICT_TAIL_POLICIES[site], STRICT_SOURCE_POLICIES[site]
        )
        baseline_rmse = metric_set(truth, baseline)["RMSE"]
        permutation = np.random.default_rng(SEED + sum(map(ord, site))).permutation(len(test_idx))
        groups = ["Recent solar state/shape/energy", "Completed-day weather", "Target astronomy/calendar"]
        if np.any(bundle.archived_weather_mask > 0):
            groups.append("Forward weather covariates")
        for group in groups:
            altered = permute_bundle_group(bundle, test_idx, group, permutation)
            prediction = predict_locked_system_on_bundle(
                site, altered, test_idx, STRICT_FINAL_MODELS[site], STRICT_FINAL_RIDGE[site],
                STRICT_SEED_WEIGHTS[site], STRICT_TAIL_POLICIES[site], STRICT_SOURCE_POLICIES[site]
            )
            rmse = metric_set(truth, prediction)["RMSE"]
            PERMUTATION_ROWS.append({"Site": site, "Group": group, "Baseline_RMSE": baseline_rmse,
                                     "Permuted_RMSE": rmse, "RMSE_increase": rmse - baseline_rmse,
                                     "RMSE_increase_percent": 100 * (rmse / max(baseline_rmse, 1e-12) - 1)})

        expert = STRICT_FINAL_RIDGE[site]
        design = expert.scaler.transform(bundle.tabular[np.ix_(test_idx, expert.feature_columns)])
        coefficients = np.asarray(expert.model.coef_)
        contributions = np.abs(design[:, None, :] * coefficients[None, :, :]).mean(axis=(0, 1))
        grouped = pd.DataFrame({"Family": expert.feature_families, "Contribution": contributions}).groupby("Family")["Contribution"].sum()
        for family, value in grouped.items():
            LINEAR_SHAP_ROWS.append({"Site": site, "Expert": "Ridge-Residual",
                                     "Feature_family": family, "Mean_absolute_LinearSHAP": float(value)})

        for horizon, hour in enumerate(TARGET_HOURS):
            BIGRU_GATE_ROWS.append({"Site": site, "Horizon": horizon + 1, "Hour": int(hour),
                                    "Mean": float(result["bigru_gate"][:, horizon].mean()),
                                    "Std": float(result["bigru_gate"][:, horizon].std()),
                                    "Min": float(result["bigru_gate"][:, horizon].min()),
                                    "Max": float(result["bigru_gate"][:, horizon].max())})
            TAIL_GATE_ROWS.append({"Site": site, "Horizon": horizon + 1, "Hour": int(hour),
                                   "Active": bool(STRICT_TAIL_POLICIES[site][horizon]["active"]),
                                   "Mean_gate": float(result["tail_gate"][:, horizon].mean()),
                                   "Max_gate": float(result["tail_gate"][:, horizon].max())})
        policy = STRICT_SOURCE_POLICIES[site]
        for source, weight in zip(policy["source_names"], policy["weights"]):
            SOURCE_WEIGHT_ROWS.append({"Site": site, "Mode": policy["mode"], "Source": source,
                                       "Weight": float(weight), "Selection_uses_test": False})

        
        trained = STRICT_FINAL_MODELS[site][0]
        if TORCH_AVAILABLE and trained.kind == "torch":
            chosen = test_idx[:min(32, len(test_idx))]
            transformed = _transform_mufasa(bundle, chosen, trained.scalers)
            sequence, astronomy, archive, archive_mask, engineered, solar_ref_state = [
                torch.from_numpy(np.asarray(value, np.float32)).to(DEVICE) for value in transformed[:6]
            ]
            seq_base = torch.zeros_like(sequence); astro_base = torch.zeros_like(astronomy)
            eng_base = torch.zeros_like(engineered); solar_ref_base = torch.zeros_like(solar_ref_state)
            totals = [torch.zeros_like(sequence), torch.zeros_like(astronomy),
                      torch.zeros_like(engineered), torch.zeros_like(solar_ref_state)]
            steps = 8 if SMOKE else 32
            trained.model.eval()
            with torch.enable_grad(), torch.backends.cudnn.flags(enabled=False):
                for alpha in torch.linspace(0, 1, steps, device=DEVICE):
                    inputs = [
                        (base + alpha * (value - base)).detach().requires_grad_(True)
                        for base, value in [(seq_base, sequence), (astro_base, astronomy),
                                            (eng_base, engineered), (solar_ref_base, solar_ref_state)]
                    ]
                    state = trained.model(
                        inputs[0], inputs[1], archive, archive_mask, inputs[2], inputs[3],
                        teacher_state=None, teacher_forcing_ratio=0.0,
                    )
                    physical = state * torch.from_numpy(bundle.potential[chosen]).to(DEVICE)
                    gradients = torch.autograd.grad(physical.sum(), inputs, allow_unused=True)
                    totals = [total + (torch.zeros_like(total) if gradient is None else gradient)
                              for total, gradient in zip(totals, gradients)]
            sequence_ig = ((sequence - seq_base) * totals[0] / steps).abs().mean((0, 1, 2)).detach().cpu().numpy()
            astronomy_ig = ((astronomy - astro_base) * totals[1] / steps).abs().mean((0, 1)).detach().cpu().numpy()
            engineered_ig = ((engineered - eng_base) * totals[2] / steps).abs().mean(0).detach().cpu().numpy()
            for feature, value in zip(bundle.sequence_channels, sequence_ig):
                IG_ROWS.append({"Site": site, "Input": "Historical sequence", "Feature": feature,
                                "Mean_absolute_IG": float(value)})
            for feature, value in zip(bundle.astronomy_names, astronomy_ig):
                IG_ROWS.append({"Site": site, "Input": "Target astronomy", "Feature": feature,
                                "Mean_absolute_IG": float(value)})
            active_families = np.asarray(bundle.tabular_families)[trained.scalers["engineered_columns"]]
            family_ig = pd.DataFrame({"Family": active_families, "IG": engineered_ig}).groupby("Family")["IG"].sum()
            for feature, value in family_ig.items():
                IG_ROWS.append({"Site": site, "Input": "Engineered", "Feature": feature,
                                "Mean_absolute_IG": float(value)})

XAI_PERMUTATION = pd.DataFrame(PERMUTATION_ROWS, columns=[
    "Site", "Group", "Baseline_RMSE", "Permuted_RMSE", "RMSE_increase", "RMSE_increase_percent"
])
XAI_IG = pd.DataFrame(IG_ROWS, columns=["Site", "Input", "Feature", "Mean_absolute_IG"])
XAI_LINEAR_SHAP = pd.DataFrame(LINEAR_SHAP_ROWS, columns=[
    "Site", "Expert", "Feature_family", "Mean_absolute_LinearSHAP"
])
XAI_BIGRU = pd.DataFrame(BIGRU_GATE_ROWS, columns=[
    "Site", "Horizon", "Hour", "Mean", "Std", "Min", "Max"
])
XAI_TAIL = pd.DataFrame(TAIL_GATE_ROWS, columns=[
    "Site", "Horizon", "Hour", "Active", "Mean_gate", "Max_gate"
])
XAI_SOURCE = pd.DataFrame(SOURCE_WEIGHT_ROWS, columns=[
    "Site", "Mode", "Source", "Weight", "Selection_uses_test"
])
for frame, filename in [
    (XAI_PERMUTATION, "MUFASA_XAI_grouped_permutation.csv"),
    (XAI_IG, "MUFASA_XAI_integrated_gradients.csv"),
    (XAI_LINEAR_SHAP, "MUFASA_XAI_LinearSHAP.csv"),
    (XAI_BIGRU, "MUFASA_XAI_historical_bigru_gate.csv"),
    (XAI_TAIL, "MUFASA_XAI_tail_guard.csv"),
    (XAI_SOURCE, "MUFASA_XAI_source_weights.csv"),
]:
    frame.to_csv(OUTPUT_DIR / filename, index=False)


In [ ]:
# Leakage and Protocol Audit
INFORMATION_ROWS = []
for scenario, selection_bundles, refit_bundles in [
    ("strict_history", STRICT_SELECTION_BUNDLES, STRICT_REFIT_BUNDLES),
    ("oracle_weather", ORACLE_SELECTION_BUNDLES, ORACLE_REFIT_BUNDLES),
]:
    if scenario == "oracle_weather" and not RUN_ORACLE:
        continue
    for site in SITES:
        selection, refit = selection_bundles[site], refit_bundles[site]
        observed_target_weather = bool(np.any(selection.archived_weather_mask > 0))
        future_family = bool(np.any(np.asarray(selection.tabular_families) == "future_weather_covariates"))
        INFORMATION_ROWS.append({
            "Scenario": scenario, "Site": site,
            "History_ends_before_target": True,
            "Observed_target_weather_used_as_input": observed_target_weather,
            "Oracle_flag": scenario == "oracle_weather",
            "Solar_target_used_as_input": False,
            "Train_end": str(TRAIN_END.date()), "Validation_end": str(VAL_END.date()),
            "Test_start": str(TEST_START.date()),
            "Test_rows_in_HPO": 0,
            "Test_rows_in_scaler_fit": 0,
            "Test_rows_in_LDS_fit": 0,
            "Test_rows_in_seed_weighting": 0,
            "Test_rows_in_TailGuard_fit": 0, "Test_rows_in_fusion_fit": 0,
            "Validation_teacher_forcing": 0.0, "Test_teacher_forcing": 0.0,
            "Future_weather_feature_family": future_family,
            "Retrospective_2020_redevelopment": True,
            "Confirmatory_future_or_external_test_required": True,
            "Canonical_core": CANONICAL_CORE_CANDIDATE,
            "Benchmark_complete": bool(BENCHMARK_COMPLETE_BY_SCENARIO.get(scenario, False)),
        })

INFORMATION_BOUNDARY_AUDIT = pd.DataFrame(INFORMATION_ROWS)
strict_rows = INFORMATION_BOUNDARY_AUDIT[INFORMATION_BOUNDARY_AUDIT["Scenario"].eq("strict_history")]
assert strict_rows["Observed_target_weather_used_as_input"].sum() == 0
assert strict_rows["Future_weather_feature_family"].sum() == 0
assert strict_rows["Solar_target_used_as_input"].sum() == 0
assert strict_rows["Test_rows_in_scaler_fit"].eq(0).all()
assert strict_rows["Test_rows_in_LDS_fit"].eq(0).all()
assert STRICT_FINAL_CONFIG["Selection_uses_test"].eq(False).all()
assert STRICT_PROMOTION_SELECTION["Selected_candidate"].eq(CANONICAL_CORE_CANDIDATE).all()
assert "C0-MUFASA-CurrentFull" not in PRIMARY_CANDIDATES
assert STRICT_CORE_SEEDS["Test_rows_in_scaler_fit"].eq(0).all()
assert STRICT_SEED_POLICY_AUDIT["Selection_uses_test"].eq(False).all()
assert STRICT_TAIL_AUDIT["Selection_uses_test"].eq(False).all()
assert STRICT_FUSION_AUDIT["Selection_uses_test"].eq(False).all()
assert STRICT_CORE_SEEDS["Teacher_forcing_validation"].eq(0.0).all()
for site, models in STRICT_FINAL_MODELS.items():
    for trained in models:
        assert trained.scalers["test_rows_in_fit"] == 0
        assert trained.scalers.get("tail_weight_min", 1.0) >= 1.0
if RUN_ORACLE:
    oracle_rows = INFORMATION_BOUNDARY_AUDIT[INFORMATION_BOUNDARY_AUDIT["Scenario"].eq("oracle_weather")]
    assert oracle_rows["Oracle_flag"].all()
    assert oracle_rows["Observed_target_weather_used_as_input"].all()
    assert (~oracle_rows["Solar_target_used_as_input"]).all()

INFORMATION_BOUNDARY_AUDIT.to_csv(OUTPUT_DIR / "MUFASA_information_boundary_audit.csv", index=False)
EVIDENCE_AUDIT = pd.DataFrame([{
    "Run_label": RUN_LABEL,
    "2020_used_for_architecture_or_HPO": False,
    "2020_used_for_scaler_or_LDS": False,
    "2020_used_for_seed_or_tail_or_fusion": False,
    "Retrospective_redevelopment": True,
    "Confirmatory_test_required": True,
    "Paper_claim_permitted_from_smoke": False if SMOKE else bool(CONFIRMATORY_UNTOUCHED_TEST),
}])
EVIDENCE_AUDIT.to_csv(OUTPUT_DIR / "MUFASA_evidence_status_audit.csv", index=False)
print("All strict/oracle information-boundary assertions passed.")


## 8. Publication artifacts

Generate paper-facing tables/figures only after completeness checks pass, then write an output manifest.

**Run note.** Execute the cells in this section in order. Objects created here are consumed by later sections; the notebook intentionally avoids hidden state restoration from unpublished artifacts.


In [ ]:
# Publication Figures and Core Tables
def save_both(fig, stem):
    from io import BytesIO
    for suffix, fmt, dpi in (("png", "png", FIG_DPI), ("pdf", "pdf", None)):
        buffer = BytesIO()
        kwargs = {"format": fmt, "bbox_inches": "tight"}
        if dpi is not None:
            kwargs["dpi"] = dpi
        fig.savefig(buffer, **kwargs)
        payload = buffer.getvalue()
        if len(payload) < 1024:
            raise RuntimeError(f"{stem}.{suffix}: rendered figure is unexpectedly small")
        (OUTPUT_DIR / f"{stem}.{suffix}").write_bytes(payload)


for site in SITES:
    table = STRICT_CANDIDATE_SCREEN[STRICT_CANDIDATE_SCREEN["Site"].eq(site)].sort_values("Mean_block_RMSE")
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.barh(table["Candidate"], table["Mean_block_RMSE"], xerr=table["SD_block_RMSE"],
            color=["#0f766e" if value else "#94a3b8" for value in table["Selected"]])
    ax.invert_yaxis(); ax.set_xlabel("2019 Q1–Q4 mean RMSE ± SD")
    ax.set_title(f"{site}: Stability-Aware Architecture Screening")
    fig.tight_layout(); save_both(fig, f"Figure_01_{site}_candidate_screen"); plt.show()

fig, ax = plt.subplots(figsize=(14, 8))
summary_plot = STRICT_TEST_BENCHMARK.groupby("Model")["RMSE"].mean().sort_values().head(12)
ax.barh(summary_plot.index, summary_plot.values,
        color=["#c2410c" if name == "MUFASA" else "#64748b" for name in summary_plot.index])
ax.invert_yaxis(); ax.set_xlabel("Six-city macro RMSE"); ax.set_title("Strict-History Benchmark")
fig.tight_layout(); save_both(fig, "Figure_02_strict_macro_benchmark"); plt.show()

fig, axes = plt.subplots(2, 1, figsize=(18, 13), constrained_layout=True)
axes[0].axis("off"); axes[1].axis("off")
axes[0].text(0.5, 0.88, "MUFASA Stable Multi-Candidate Selection", ha="center", fontsize=28, weight="bold")
axes[0].text(0.5, 0.50, "C1–C5 matched seeds/epochs/scalers → 2019 Q1–Q4 diagnostics → C2 + strongest challenger HPO",
             ha="center", fontsize=20, bbox=dict(boxstyle="round,pad=0.6", facecolor="#dbeafe", edgecolor="#1d4ed8"))
axes[1].text(0.5, 0.78, "3-seed core + Ridge residual + Solar-geometry reference", ha="center", fontsize=22,
             bbox=dict(boxstyle="round,pad=0.6", facecolor="#dcfce7", edgecolor="#15803d"))
axes[1].text(0.5, 0.40, "Seed weighting / TailGuard / Fusion only if rolling-OOF 7-day block CI upper < 0",
             ha="center", fontsize=20, bbox=dict(boxstyle="round,pad=0.6", facecolor="#ffedd5", edgecolor="#c2410c"))
axes[1].text(0.5, 0.08, "Lock all policies → refit 2016–2019 → one 2020 inference; strict and oracle reported separately",
             ha="center", fontsize=19)
save_both(fig, "Figure_00_MUFASA_stable_selection_architecture"); plt.show()


In [ ]:
# Fail-Closed Paper Summary
strict_summary = BENCHMARK_SUMMARY[BENCHMARK_SUMMARY["Scenario"].eq("strict_history")].iloc[0]
oracle_summary = (
    BENCHMARK_SUMMARY[BENCHMARK_SUMMARY["Scenario"].eq("oracle_weather")].iloc[0]
    if RUN_ORACLE and (BENCHMARK_SUMMARY["Scenario"] == "oracle_weather").any() else None
)
strict_complete = bool(BENCHMARK_COMPLETE_BY_SCENARIO.get("strict_history", False))
oracle_complete = bool(BENCHMARK_COMPLETE_BY_SCENARIO.get("oracle_weather", False)) if RUN_ORACLE else False
strict_claim = SUPERIORITY_CLAIM_AUDIT[
    SUPERIORITY_CLAIM_AUDIT["Scenario"].eq("strict_history")
].iloc[0]

if not strict_complete or SMOKE:
    claim = "E. Benchmark incomplete or structural-smoke run: publication ranking and superiority claims are disabled."
elif strict_claim["Numeric_rank1_every_city"] and strict_claim["All_comparator_statistical_support"]:
    claim = "A. Unconditional superiority supported against the completed external benchmark family."
elif strict_summary["Number_rank_1_cities"] >= 2:
    claim = "B. Rank-one performance is observed in several sites, but unconditional superiority is not supported."
else:
    claim = "C. The model is competitive but not consistently superior."
if strict_complete and oracle_complete and oracle_summary is not None:
    if oracle_summary["Macro_average_RMSE"] < strict_summary["Macro_average_RMSE"] * 0.95:
        claim += " D. Perfect-weather results identify forward weather information as a major remaining bottleneck."

strict_modes = pd.Series({site: policy["mode"] for site, policy in STRICT_SOURCE_POLICIES.items()})
oracle_modes = (
    pd.Series({site: policy["mode"] for site, policy in ORACLE_RUN["source"].items()})
    if RUN_ORACLE else pd.Series(dtype=str)
)
component_questions = [
    {"Question": "Q1 Canonical neural core", "Answer": CANONICAL_CORE_CANDIDATE},
    {"Question": "Q2 BiGRU", "Answer": "Compare C2 vs C1 in MUFASA_core_only_ablation.csv"},
    {"Question": "Q3 LDS/Tail", "Answer": "Compare C4 vs C2 and C3 vs C1 in core-only results"},
    {"Question": "Q4 RevIN", "Answer": "Compare C5 vs C4 in core-only results"},
    {"Question": "Q5 NLinear/AR/TF", "Answer": "Matched-seed core-only component rows are reported separately"},
    {"Question": "Q6 Ridge strength", "Answer": "Internal expert tests and 2019 source stability are separate from SOTA tests"},
    {"Question": "Q7 Strict fusion", "Answer": f"MUFASA in {(strict_modes == 'MUFASA').sum()}/{len(strict_modes)} cities"},
    {"Question": "Q8 Oracle fusion", "Answer": f"MUFASA in {(oracle_modes == 'MUFASA').sum()}/{len(oracle_modes)} cities" if len(oracle_modes) else "oracle disabled"},
    {"Question": "Q9 strict-oracle gap", "Answer": (
        f"{100*(strict_summary['Macro_average_RMSE']-oracle_summary['Macro_average_RMSE'])/strict_summary['Macro_average_RMSE']:.2f}% macro RMSE"
        if oracle_summary is not None else "oracle disabled"
    )},
    {"Question": "Q10 confirmatory status", "Answer": "2020 retrospective; frozen future/external test required"},
]
COMPONENT_QUESTIONS = pd.DataFrame(component_questions)
COMPONENT_QUESTIONS.to_csv(OUTPUT_DIR / "MUFASA_component_questions_Q1_Q10.csv", index=False)

rank_text = (
    f"{int(strict_summary['Number_rank_1_cities'])}" if strict_complete and not SMOKE else "withheld"
)
top2_text = (
    f"{int(strict_summary['Number_top_2_cities'])}" if strict_complete and not SMOKE else "withheld"
)
lines = [
    "# MUFASA automatic paper-results summary", "",
    f"Run label: **{RUN_LABEL}**", "",
    f"Benchmark contract: **{BENCHMARK_CONTRACT_VERSION}**", "",
    f"Canonical neural core: **{CANONICAL_CORE_CANDIDATE}**", "",
    "## Strict-history result", "",
    f"- Six-city macro RMSE: {strict_summary['Macro_average_RMSE']:.6f}",
    f"- Six-city macro MAE: {strict_summary['Macro_average_MAE']:.6f}",
    f"- Deep benchmark complete: {strict_complete}",
    f"- Publication-eligible rank-1 cities: {rank_text}",
    f"- Publication-eligible top-2 cities: {top2_text}", "",
    "## Oracle / Perfect Weather Assumption", "",
]
if oracle_summary is not None:
    lines.extend([
        f"- Six-city macro RMSE: {oracle_summary['Macro_average_RMSE']:.6f}",
        f"- Six-city macro MAE: {oracle_summary['Macro_average_MAE']:.6f}",
        f"- Deep benchmark complete: {oracle_complete}", "",
    ])
else:
    lines.extend(["- Oracle run disabled.", ""])
lines.extend([
    "## Claim", "", claim, "",
    "The previous 3,888/3,888-failed deep benchmark run is invalidated and is never reused.",
    "2020 remains retrospective development evidence; a frozen future/external confirmatory test is required.",
])
(OUTPUT_DIR / "paper_results_auto.md").write_text("\n".join(lines), encoding="utf-8")
SUMMARY = {
    "run_label": RUN_LABEL, "benchmark_contract": BENCHMARK_CONTRACT_VERSION,
    "canonical_core": CANONICAL_CORE_CANDIDATE,
    "strict_benchmark_complete": strict_complete,
    "oracle_benchmark_complete": oracle_complete,
    "strict": strict_summary.to_dict(),
    "oracle": None if oracle_summary is None else oracle_summary.to_dict(),
    "claim": claim, "retrospective_2020": True,
    "previous_failed_deep_trials_invalidated": INVALIDATED_PREVIOUS_DEEP_TRIALS,
    "confirmatory_future_or_external_test_required": True,
}
(OUTPUT_DIR / "MUFASA_summary.json").write_text(
    json.dumps(SUMMARY, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)
print("\n".join(lines))


In [ ]:
# Output Manifest
import zipfile

required_outputs = [
    "MUFASA_candidate_screen_2019.csv", "MUFASA_candidate_selection_reason.csv",
    "MUFASA_reduced_HPO_trials.csv", "MUFASA_seed_policy_audit.csv",
    "MUFASA_tail_guard_stability_audit.csv", "MUFASA_winner_or_fusion_audit.csv",
    "MUFASA_source_2019_stability.csv", "MUFASA_strict_test_benchmark.csv",
    "MUFASA_oracle_test_benchmark.csv", "MUFASA_core_only_ablation.csv",
    "MUFASA_end_to_end_ablation.csv", "MUFASA_clean_ablation.csv",
    "MUFASA_benchmark_contract_preflight.csv", "MUFASA_benchmark_completion_audit.csv",
    "MUFASA_strict_deep_HPO.csv", "MUFASA_strict_deep_promotion.csv",
    "MUFASA_strict_deep_seed_results.csv", "MUFASA_strict_deep_seed_policy_audit.csv",
    "MUFASA_oracle_deep_HPO.csv", "MUFASA_oracle_deep_promotion.csv",
    "MUFASA_oracle_deep_seed_results.csv", "MUFASA_oracle_deep_seed_policy_audit.csv",
    "MUFASA_benchmark_retry_audit.csv",
    "MUFASA_benchmark_provenance.csv", "MUFASA_external_baseline_statistical_tests.csv",
    "MUFASA_internal_expert_statistical_tests.csv", "MUFASA_information_boundary_audit.csv",
    "MUFASA_final_selected_configuration.csv", "MUFASA_retired_duplicate_candidates.csv",
    "MUFASA_XAI_grouped_permutation.csv", "MUFASA_XAI_integrated_gradients.csv",
    "MUFASA_XAI_LinearSHAP.csv", "MUFASA_XAI_historical_bigru_gate.csv",
    "MUFASA_XAI_tail_guard.csv", "MUFASA_XAI_source_weights.csv",
    "Figure_00_MUFASA_stable_selection_architecture.png",
    "Figure_00_MUFASA_stable_selection_architecture.pdf",
    "Figure_02_strict_macro_benchmark.png", "Figure_02_strict_macro_benchmark.pdf",
    "paper_results_auto.md", "MUFASA_summary.json",
]
required_outputs += [f"Figure_01_{site}_candidate_screen.{suffix}" for site in SITES for suffix in ("png", "pdf")]
missing = [name for name in required_outputs if not (OUTPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Required output files are missing: {missing}")

manifest_rows = []
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        manifest_rows.append({
            "File": str(path.relative_to(OUTPUT_DIR)), "Bytes": path.stat().st_size,
            "SHA256": hashlib.sha256(path.read_bytes()).hexdigest(), "Run_label": RUN_LABEL,
            "Benchmark_contract": BENCHMARK_CONTRACT_VERSION,
        })
OUTPUT_MANIFEST = pd.DataFrame(manifest_rows)
OUTPUT_MANIFEST.to_csv(OUTPUT_DIR / "MUFASA_output_manifest.csv", index=False)

zip_path = OUTPUT_DIR.with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(Path(OUTPUT_DIR.name) / path.relative_to(OUTPUT_DIR)))
print(f"Notebook run complete: {OUTPUT_DIR.resolve()}")
print(f"ZIP: {zip_path.resolve()} ({zip_path.stat().st_size:,} bytes)")


## 9. Oracle-weather manuscript XAI

Generate the manuscript-facing grouped/variable/horizon/site XAI artifacts under the controlled oracle-weather protocol.

**Run note.** Execute the cells in this section in order. Objects created here are consumed by later sections; the notebook intentionally avoids hidden state restoration from unpublished artifacts.


In [ ]:
# Oracle-Weather End-to-End Explainability — part 1/10
# ======================================================================
# ORACLE WEATHER + ALL-CITY MUFASA_FUSION XAI
# ----------------------------------------------------------------------


#

#   - MUFASA validation/test audit
#   - Grouped permutation importance
#   - Future-weather variable importance
#   - Horizon-wise future-weather importance heatmap
#   - 6-city macro figures
#


# ======================================================================

from pathlib import Path
from dataclasses import replace
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------

# ----------------------------------------------------------------------
XAI_PERM_REPEATS = 10
ORACLE_XAI_DIR = OUTPUT_DIR / "oracle_xai"
if ORACLE_XAI_DIR.exists():
    shutil.rmtree(ORACLE_XAI_DIR)
ORACLE_XAI_DIR.mkdir(parents=True, exist_ok=True)
FIG_DPI_XAI = 600
print("=" * 78)
print("MUFASA — Oracle Weather / All-City MUFASA XAI")
print(f"Output directory : {ORACLE_XAI_DIR.resolve()}")
print(f"Permutation repeats = {XAI_PERM_REPEATS}")
print("=" * 78)


# ----------------------------------------------------------------------

# ----------------------------------------------------------------------
_required_objects = [
    "ORACLE_RUN",
    "ORACLE_SELECTION_BUNDLES",
    "ORACLE_REFIT_BUNDLES",
    "SITES",
    "TARGET_HOURS",
    "N_HORIZONS",
    "metric_set",
    "positions",
    "apply_seed_policy",
    "predict_mufasa",
    "predict_ridge_residual",
    "predict_anchor",
    "fit_simplex_fusion",
    "apply_source_weights",
    "source_stability_table",
    "stable_near_best",
]
_missing = [name for name in _required_objects if name not in globals()]
if _missing:
    raise RuntimeError(
        "Oracle-weather prerequisite is missing. Run the preceding oracle-weather stage first."
        f"Missing objects = {_missing}"
    )
if ORACLE_RUN is None:
    raise RuntimeError(
        "Oracle-weather prerequisite is missing. Run the preceding oracle-weather stage first."
        "Oracle-weather prerequisite is missing. Run the preceding oracle-weather stage first."
    )
if len(SITES) != 6:
    print(
        f"[Warning] Current sites={len(SITES)}: {SITES}\n"
        "Public-release execution note."
        "Public-release execution note."
    )
print("Oracle objects verified.")
print("Cities:", SITES)


# ======================================================================

# ======================================================================
#

#   MUFASA-Core
#   Ridge-Residual
#   Solar-Geometry-Reference
#

#


#


# ======================================================================
ORACLE_ALLCITY_FUSION_POLICIES = {}
fusion_policy_rows = []
fusion_validation_rows = []


In [ ]:
# Oracle-Weather End-to-End Explainability — part 2/10
for site in SITES:

    bundle = ORACLE_SELECTION_BUNDLES[site]
    val_idx = positions(bundle.val_all)

    truth_val = bundle.target[val_idx]
    dates_val = bundle.target_dates[val_idx]

    # --------------------------------------------------------------
    # Neural Core validation prediction
    # --------------------------------------------------------------
    cube = ORACLE_RUN["val_cubes"][site]

    core_val = apply_seed_policy(
        cube,
        ORACLE_RUN["seed_weights"][site],
    )

    
    

    # --------------------------------------------------------------
    # Ridge / Astronomy validation predictions
    # --------------------------------------------------------------
    ridge_val = np.asarray(
        ORACLE_RUN["ridge_val"][site],
        dtype=np.float32,
    )

    solar_ref_val = np.asarray(
        ORACLE_RUN["prior_val"][site],
        dtype=np.float32,
    )

    sources_val = {
        "MUFASA-Core": core_val,
        "Ridge-Residual": ridge_val,
        "Solar-Geometry-Reference": solar_ref_val,
    }

    source_names = list(sources_val.keys())

    # --------------------------------------------------------------
    
    
    # --------------------------------------------------------------
    stability = source_stability_table(
        sources_val,
        truth_val,
        dates_val,
        site,
        "oracle_weather_allcity_stablefusion",
    )

    winner_row, near_best = stable_near_best(
        stability,
        "Source",
    )

    stable_winner = winner_row["Source"]

    # --------------------------------------------------------------
    
    # --------------------------------------------------------------
    weights = fit_simplex_fusion(
        sources_val,
        truth_val,
        source_names,
        stable_winner,
    )

    weights = np.asarray(weights, dtype=float)

    if not np.isfinite(weights).all():
        raise FloatingPointError(
            f"{site}: MUFASA weight contains NaN/Inf."
        )

    if np.any(weights < -1e-8):
        raise AssertionError(
            f"{site}: MUFASA contains negative weight: {weights}"
        )

    if not np.isclose(weights.sum(), 1.0, atol=1e-5):
        raise AssertionError(
            f"{site}: MUFASA weights do not sum to 1: {weights}"
        )

    fusion_val = apply_source_weights(
        sources_val,
        source_names,
        weights,
    )

    fusion_metrics = metric_set(
        truth_val,
        fusion_val,
    )

    winner_metrics = metric_set(
        truth_val,
        sources_val[stable_winner],
    )

    policy = {
        "mode": "MUFASA",
        "source_names": source_names,
        "weights": weights,
        "winner_anchor": stable_winner,
        "selection_uses_test": False,
    }

    ORACLE_ALLCITY_FUSION_POLICIES[site] = policy

    for source, weight in zip(source_names, weights):
        fusion_policy_rows.append({
            "Site": site,
            "Mode": "MUFASA",
            "Source": source,
            "Weight": float(weight),
            "Validation_anchor": stable_winner,
            "Selection_uses_test": False,
        })

    fusion_validation_rows.append({
        "Site": site,
        "Stable_winner_anchor": stable_winner,
        "Winner_val_RMSE": winner_metrics["RMSE"],
        "MUFASA_val_RMSE": fusion_metrics["RMSE"],
        "MUFASA_val_MAE": fusion_metrics["MAE"],
        "MUFASA_val_R2": fusion_metrics["R2"],
        "Selection_uses_test": False,
    })
ORACLE_ALLCITY_FUSION_POLICY_DF = pd.DataFrame(
    fusion_policy_rows
)
ORACLE_ALLCITY_FUSION_VALIDATION_DF = pd.DataFrame(
    fusion_validation_rows
)
ORACLE_ALLCITY_FUSION_POLICY_DF.to_csv(
    ORACLE_XAI_DIR / "00_Oracle_MUFASA_weights.csv",
    index=False,
)
ORACLE_ALLCITY_FUSION_VALIDATION_DF.to_csv(
    ORACLE_XAI_DIR / "00_Oracle_MUFASA_validation_audit.csv",
    index=False,
)
print("\n[All-city Oracle MUFASA weights]")
display(
    ORACLE_ALLCITY_FUSION_POLICY_DF.round(6)
)


# ======================================================================

# ======================================================================


In [ ]:
# Oracle-Weather End-to-End Explainability — part 3/10
def predict_oracle_stablefusion(
    site,
    bundle,
    indices,
):
    "Oracle-weather prerequisite is missing. Run the preceding oracle-weather stage first."

    indices = np.asarray(
        indices,
        dtype=int,
    )

    models = ORACLE_RUN["final_models"][site]
    ridge_expert = ORACLE_RUN["final_ridge"][site]

    # --------------------------------------------------------------
    # 3-seed MUFASA Core
    # --------------------------------------------------------------
    seed_predictions = [
        predict_mufasa(
            model,
            bundle,
            indices,
            mc_passes=1,
        )
        for model in models
    ]

    cube = np.stack(
        seed_predictions,
        axis=-1,
    )

    core = apply_seed_policy(
        cube,
        ORACLE_RUN["seed_weights"][site],
    )

    # --------------------------------------------------------------
    # Ridge / Astronomy experts
    # --------------------------------------------------------------
    ridge = predict_ridge_residual(
        ridge_expert,
        bundle,
        indices,
    )

    prior = predict_anchor(
        ridge_expert,
        bundle,
        indices,
    )

    sources = {
        "MUFASA-Core": core,
        "Ridge-Residual": ridge,
        "Solar-Geometry-Reference": prior,
    }

    policy = ORACLE_ALLCITY_FUSION_POLICIES[site]

    prediction = apply_source_weights(
        sources,
        policy["source_names"],
        policy["weights"],
    )

    if prediction.shape != (
        len(indices),
        N_HORIZONS,
    ):
        raise RuntimeError(
            f"{site}: prediction shape={prediction.shape}; "
            f"expected {(len(indices), N_HORIZONS)}"
        )

    if not np.isfinite(prediction).all():
        raise FloatingPointError(
            f"{site}: Oracle MUFASA prediction has NaN/Inf."
        )

    return prediction


# ======================================================================

# ======================================================================
#

# ======================================================================
stablefusion_test_rows = []
ORACLE_MUFASA_FUSION_TEST_PREDICTIONS = {}
for site in SITES:

    bundle = ORACLE_REFIT_BUNDLES[site]

    test_idx = positions(
        bundle.test
    )

    truth = bundle.target[test_idx]

    prediction = predict_oracle_stablefusion(
        site,
        bundle,
        test_idx,
    )

    ORACLE_MUFASA_FUSION_TEST_PREDICTIONS[site] = prediction

    metrics = metric_set(
        truth,
        prediction,
    )

    stablefusion_test_rows.append({
        "Site": site,
        "Model": "MUFASA-Oracle",
        **metrics,
    })
ORACLE_MUFASA_FUSION_TEST_DF = pd.DataFrame(
    stablefusion_test_rows
)
ORACLE_MUFASA_FUSION_TEST_DF.to_csv(
    ORACLE_XAI_DIR / "00_Oracle_MUFASA_test_performance.csv",
    index=False,
)

# ======================================================================

# ======================================================================
#


#


# ======================================================================
all_prediction_rows = []


In [ ]:
# Oracle-Weather End-to-End Explainability — part 4/10
for site in SITES:

    bundle = ORACLE_REFIT_BUNDLES[site]

    test_idx = positions(
        bundle.test
    )

    dates = pd.DatetimeIndex(
        bundle.target_dates[test_idx]
    )

    truth = np.asarray(
        bundle.target[test_idx],
        dtype=float,
    )

    prediction = np.asarray(
        ORACLE_MUFASA_FUSION_TEST_PREDICTIONS[site],
        dtype=float,
    )

    # --------------------------------------------------------------
    # Wide format:
    
    # --------------------------------------------------------------
    wide_rows = []

    for i, date in enumerate(dates):

        row = {
            "Site": site,
            "Date": pd.Timestamp(date).strftime("%Y-%m-%d"),
            "Model": "MUFASA-Oracle",
        }

        for h, hour in enumerate(TARGET_HOURS):

            actual_value = float(truth[i, h])
            predicted_value = float(prediction[i, h])

            row[f"Actual_{int(hour):02d}"] = actual_value
            row[f"Pred_{int(hour):02d}"] = predicted_value
            row[f"Error_{int(hour):02d}"] = predicted_value - actual_value
            row[f"AbsError_{int(hour):02d}"] = abs(
                predicted_value - actual_value
            )

        wide_rows.append(row)

    wide_df = pd.DataFrame(
        wide_rows
    )

    wide_df.to_csv(
        ORACLE_XAI_DIR
        / f"Prediction_{site}_Oracle_MUFASA_WIDE.csv",
        index=False,
    )

    # --------------------------------------------------------------
    # Long format:
    
    
    # --------------------------------------------------------------
    long_rows = []

    for i, date in enumerate(dates):

        for h, hour in enumerate(TARGET_HOURS):

            actual_value = float(
                truth[i, h]
            )

            predicted_value = float(
                prediction[i, h]
            )

            error = (
                predicted_value
                - actual_value
            )

            long_rows.append({
                "Site": site,
                "Date": pd.Timestamp(date).strftime("%Y-%m-%d"),
                "Hour": int(hour),
                "Horizon": h + 1,
                "Actual": actual_value,
                "Prediction": predicted_value,
                "Error": error,
                "Absolute_Error": abs(error),
                "Squared_Error": error ** 2,
                "Model": "MUFASA-Oracle",
            })

    long_df = pd.DataFrame(
        long_rows
    )

    long_df.to_csv(
        ORACLE_XAI_DIR
        / f"Prediction_{site}_Oracle_MUFASA_LONG.csv",
        index=False,
    )

    all_prediction_rows.append(
        long_df
    )

# ----------------------------------------------------------------------

# ----------------------------------------------------------------------
ORACLE_MUFASA_FUSION_ALL_PREDICTIONS = pd.concat(
    all_prediction_rows,
    ignore_index=True,
)
ORACLE_MUFASA_FUSION_ALL_PREDICTIONS.to_csv(
    ORACLE_XAI_DIR
    / "00_ALL_6Cities_Oracle_MUFASA_Predictions_LONG.csv",
    index=False,
)
print(
    "\nMUFASA prediction files saved."
)
print(
    "Total prediction rows =",
    len(
        ORACLE_MUFASA_FUSION_ALL_PREDICTIONS
    ),
)
display(
    ORACLE_MUFASA_FUSION_ALL_PREDICTIONS.head(20)
)
print("\n[Oracle all-city MUFASA test performance]")
display(
    ORACLE_MUFASA_FUSION_TEST_DF.round(6)
)
print(
    "\nMacro-average RMSE =",
    ORACLE_MUFASA_FUSION_TEST_DF["RMSE"].mean(),
)
print(
    "Macro-average MAE  =",
    ORACLE_MUFASA_FUSION_TEST_DF["MAE"].mean(),
)
print(
    "Macro-average R2   =",
    ORACLE_MUFASA_FUSION_TEST_DF["R2"].mean(),
)


# ======================================================================
# 5. Oracle bundle perturbation helper
# ======================================================================
def _future_weather_columns(bundle):
    "Oracle-weather prerequisite is missing. Run the preceding oracle-weather stage first."

    families = np.asarray(
        bundle.tabular_families
    ).astype(str)

    columns = np.where(
        families == "future_weather_covariates"
    )[0]

    expected = (
        N_HORIZONS
        * len(bundle.target_weather_names)
    )

    if len(columns) != expected:
        raise RuntimeError(
            f"{bundle.site}: expected {expected} future weather tabular "
            f"columns but found {len(columns)}."
        )

    return columns


In [ ]:
# Oracle-Weather End-to-End Explainability — part 5/10
def permute_oracle_group(
    bundle,
    indices,
    group,
    permutation,
):
    """Public-release implementation note."""

    indices = np.asarray(
        indices,
        dtype=int,
    )

    permutation = np.asarray(
        permutation,
        dtype=int,
    )

    source_indices = indices[permutation]

    sequence = bundle.sequence.copy()
    astronomy = bundle.astronomy.copy()
    archive = bundle.archived_weather.copy()
    tabular = bundle.tabular.copy()

    families = np.asarray(
        bundle.tabular_families
    ).astype(str)

    # --------------------------------------------------------------
    # A. Oracle target-day meteorology
    # --------------------------------------------------------------
    if group == "Forward meteorological conditions":

        archive[indices] = archive[source_indices]

        future_cols = _future_weather_columns(
            bundle
        )

        tabular[
            np.ix_(indices, future_cols)
        ] = tabular[
            np.ix_(source_indices, future_cols)
        ]

    # --------------------------------------------------------------
    # B. Historical Solar dynamics
    # --------------------------------------------------------------
    elif group == "Historical solar dynamics":

        # sequence first 4 channels:
        # clear-sky state, normalized shape, relative energy, ramp
        sequence[
            indices,
            ...,
            :4
        ] = sequence[
            source_indices,
            ...,
            :4
        ]

        mask = np.array([
            (
                "solar_" in family
                or "clearness" in family
            )
            for family in families
        ])

        columns = np.where(mask)[0]

        if len(columns):
            tabular[
                np.ix_(indices, columns)
            ] = tabular[
                np.ix_(source_indices, columns)
            ]

    # --------------------------------------------------------------
    # C. Historical meteorology
    # --------------------------------------------------------------
    elif group == "Historical meteorology":

        
        sequence[
            indices,
            ...,
            4:
        ] = sequence[
            source_indices,
            ...,
            4:
        ]

        mask = np.array([
            (
                (
                    "weather" in family.lower()
                    or "tempchange" in family.lower()
                    or "rhchange" in family.lower()
                )
                and family != "future_weather_covariates"
            )
            for family in families
        ])

        columns = np.where(mask)[0]

        if len(columns):
            tabular[
                np.ix_(indices, columns)
            ] = tabular[
                np.ix_(source_indices, columns)
            ]

    # --------------------------------------------------------------
    # D. Solar geometry/calendar
    # --------------------------------------------------------------
    elif group == "Solar geometry and calendar":

        astronomy[
            indices
        ] = astronomy[
            source_indices
        ]

        mask = np.array([
            (
                "target_astronomy" in family
                or "target_calendar" in family
            )
            for family in families
        ])

        columns = np.where(mask)[0]

        if len(columns):
            tabular[
                np.ix_(indices, columns)
            ] = tabular[
                np.ix_(source_indices, columns)
            ]

    else:
        raise KeyError(
            f"Unknown XAI group: {group}"
        )

    return replace(
        bundle,
        sequence=sequence,
        astronomy=astronomy,
        archived_weather=archive,
        tabular=tabular,
    )


# ======================================================================
# 6. End-to-end Grouped Permutation Importance
# ======================================================================
#

#


# ======================================================================
XAI_GROUPS = [
    "Forward meteorological conditions",
    "Historical solar dynamics",
    "Historical meteorology",
    "Solar geometry and calendar",
]
group_rows = []


In [ ]:
# Oracle-Weather End-to-End Explainability — part 6/10
for site in SITES:

    print(
        f"\n[Grouped permutation] {site}"
    )

    bundle = ORACLE_REFIT_BUNDLES[site]

    test_idx = positions(
        bundle.test
    )

    truth = bundle.target[test_idx]

    baseline = (
        ORACLE_MUFASA_FUSION_TEST_PREDICTIONS[site]
    )

    baseline_rmse = metric_set(
        truth,
        baseline,
    )["RMSE"]

    for group_idx, group in enumerate(XAI_GROUPS):

        for repeat in range(
            XAI_PERM_REPEATS
        ):

            rng = np.random.default_rng(
                SEED
                + 10000
                + 1000 * group_idx
                + 100 * repeat
                + sum(map(ord, site))
            )

            permutation = rng.permutation(
                len(test_idx)
            )

            altered = permute_oracle_group(
                bundle,
                test_idx,
                group,
                permutation,
            )

            prediction = predict_oracle_stablefusion(
                site,
                altered,
                test_idx,
            )

            permuted_rmse = metric_set(
                truth,
                prediction,
            )["RMSE"]

            group_rows.append({
                "Site": site,
                "Group": group,
                "Repeat": repeat + 1,
                "Baseline_RMSE": baseline_rmse,
                "Permuted_RMSE": permuted_rmse,
                "RMSE_increase": (
                    permuted_rmse
                    - baseline_rmse
                ),
                "RMSE_increase_percent": (
                    100.0
                    * (
                        permuted_rmse
                        / max(
                            baseline_rmse,
                            1e-12,
                        )
                        - 1.0
                    )
                ),
            })
ORACLE_XAI_GROUP_DETAIL = pd.DataFrame(
    group_rows
)
ORACLE_XAI_GROUP_SUMMARY = (
    ORACLE_XAI_GROUP_DETAIL
    .groupby(
        ["Site", "Group"],
        as_index=False,
    )
    .agg(
        Baseline_RMSE=(
            "Baseline_RMSE",
            "first",
        ),
        RMSE_increase_percent_mean=(
            "RMSE_increase_percent",
            "mean",
        ),
        RMSE_increase_percent_std=(
            "RMSE_increase_percent",
            "std",
        ),
        RMSE_increase_mean=(
            "RMSE_increase",
            "mean",
        ),
    )
)
ORACLE_XAI_GROUP_MACRO = (
    ORACLE_XAI_GROUP_SUMMARY
    .groupby(
        "Group",
        as_index=False,
    )
    .agg(
        Mean_RMSE_increase_percent=(
            "RMSE_increase_percent_mean",
            "mean",
        ),
        SD_across_sites=(
            "RMSE_increase_percent_mean",
            "std",
        ),
    )
    .sort_values(
        "Mean_RMSE_increase_percent",
        ascending=False,
    )
)
ORACLE_XAI_GROUP_DETAIL.to_csv(
    ORACLE_XAI_DIR
    / "01_grouped_permutation_detail.csv",
    index=False,
)
ORACLE_XAI_GROUP_SUMMARY.to_csv(
    ORACLE_XAI_DIR
    / "01_grouped_permutation_by_site.csv",
    index=False,
)
ORACLE_XAI_GROUP_MACRO.to_csv(
    ORACLE_XAI_DIR
    / "01_grouped_permutation_macro.csv",
    index=False,
)
print("\n[Grouped permutation macro importance]")
display(
    ORACLE_XAI_GROUP_MACRO.round(4)
)


# ======================================================================
# 7. Figure A — Global grouped permutation importance
# ======================================================================
_plot = ORACLE_XAI_GROUP_MACRO.sort_values(
    "Mean_RMSE_increase_percent",
    ascending=True,
)
fig, ax = plt.subplots(
    figsize=(9.0, 5.4)
)
ax.barh(
    _plot["Group"],
    _plot["Mean_RMSE_increase_percent"],
    xerr=_plot["SD_across_sites"].fillna(0.0),
    capsize=3,
)
ax.axvline(
    0.0,
    linewidth=0.8,
)
ax.set_xlabel(
    "Increase in RMSE after permutation (%)"
)
ax.set_ylabel(
    ""
)
ax.set_title(
    "Oracle-Weather MUFASA: Grouped Permutation Importance"
)
ax.grid(
    axis="x",
    alpha=0.25,
)
fig.tight_layout()
fig.savefig(
    ORACLE_XAI_DIR
    / "Figure_XAI_A_Grouped_Permutation.png",
    dpi=FIG_DPI_XAI,
    bbox_inches="tight",
)
fig.savefig(
    ORACLE_XAI_DIR
    / "Figure_XAI_A_Grouped_Permutation.pdf",
    bbox_inches="tight",
)
plt.show()


In [ ]:
# Oracle-Weather End-to-End Explainability — part 7/10
plt.close(fig)


# ======================================================================
# 8. Future-weather variable-level permutation
# ======================================================================
#

#

# Temp
# Humi
# WS
# wind_u
# wind_v
# DewPoint
# VaporPressure
# DewPointDepression
#

#


#

# ======================================================================
def permute_future_weather_variable(
    bundle,
    indices,
    weather_index,
    permutation,
):
    """Public-release implementation note."""

    indices = np.asarray(
        indices,
        dtype=int,
    )

    permutation = np.asarray(
        permutation,
        dtype=int,
    )

    source_indices = indices[permutation]

    archive = bundle.archived_weather.copy()
    tabular = bundle.tabular.copy()

    weather_names = tuple(
        bundle.target_weather_names
    )

    n_weather = len(
        weather_names
    )

    future_cols = _future_weather_columns(
        bundle
    )

    # --------------------------------------------------------------
    # Neural archive input
    # [N, horizon, weather]
    # --------------------------------------------------------------
    archive[
        indices,
        :,
        weather_index
    ] = archive[
        source_indices,
        :,
        weather_index
    ]

    # --------------------------------------------------------------
    # Tabular appended Oracle weather
    # order = horizon-major × weather-minor
    # --------------------------------------------------------------
    original_block = (
        tabular[
            np.ix_(
                indices,
                future_cols,
            )
        ]
        .reshape(
            len(indices),
            N_HORIZONS,
            n_weather,
        )
    )

    source_block = (
        tabular[
            np.ix_(
                source_indices,
                future_cols,
            )
        ]
        .reshape(
            len(indices),
            N_HORIZONS,
            n_weather,
        )
    )

    changed_block = original_block.copy()

    changed_block[
        :,
        :,
        weather_index
    ] = source_block[
        :,
        :,
        weather_index
    ]

    tabular[
        np.ix_(
            indices,
            future_cols,
        )
    ] = changed_block.reshape(
        len(indices),
        -1,
    )

    return replace(
        bundle,
        archived_weather=archive,
        tabular=tabular,
    )
weather_rows = []
weather_horizon_rows = []


In [ ]:
# Oracle-Weather End-to-End Explainability — part 8/10
for site in SITES:

    print(
        f"\n[Future weather variable XAI] {site}"
    )

    bundle = ORACLE_REFIT_BUNDLES[site]

    test_idx = positions(
        bundle.test
    )

    truth = bundle.target[test_idx]

    baseline = (
        ORACLE_MUFASA_FUSION_TEST_PREDICTIONS[site]
    )

    weather_names = tuple(
        bundle.target_weather_names
    )

    baseline_total_rmse = metric_set(
        truth,
        baseline,
    )["RMSE"]

    baseline_horizon_rmse = np.array([
        np.sqrt(
            np.mean(
                (
                    truth[:, h]
                    - baseline[:, h]
                ) ** 2
            )
        )
        for h in range(
            N_HORIZONS
        )
    ])

    for weather_index, weather_name in enumerate(
        weather_names
    ):

        for repeat in range(
            XAI_PERM_REPEATS
        ):

            rng = np.random.default_rng(
                SEED
                + 30000
                + 1000 * weather_index
                + 100 * repeat
                + sum(map(ord, site))
            )

            permutation = rng.permutation(
                len(test_idx)
            )

            altered = permute_future_weather_variable(
                bundle,
                test_idx,
                weather_index,
                permutation,
            )

            prediction = predict_oracle_stablefusion(
                site,
                altered,
                test_idx,
            )

            # ------------------------------------------------------
            
            # ------------------------------------------------------
            permuted_total_rmse = metric_set(
                truth,
                prediction,
            )["RMSE"]

            weather_rows.append({
                "Site": site,
                "Variable": weather_name,
                "Repeat": repeat + 1,
                "Baseline_RMSE": baseline_total_rmse,
                "Permuted_RMSE": permuted_total_rmse,
                "RMSE_increase_percent": (
                    100.0
                    * (
                        permuted_total_rmse
                        / max(
                            baseline_total_rmse,
                            1e-12,
                        )
                        - 1.0
                    )
                ),
            })

            # ------------------------------------------------------
            
            # ------------------------------------------------------
            for h, hour in enumerate(
                TARGET_HOURS
            ):

                permuted_h_rmse = np.sqrt(
                    np.mean(
                        (
                            truth[:, h]
                            - prediction[:, h]
                        ) ** 2
                    )
                )

                weather_horizon_rows.append({
                    "Site": site,
                    "Variable": weather_name,
                    "Hour": int(hour),
                    "Horizon": h + 1,
                    "Repeat": repeat + 1,
                    "Baseline_RMSE": float(
                        baseline_horizon_rmse[h]
                    ),
                    "Permuted_RMSE": float(
                        permuted_h_rmse
                    ),
                    "RMSE_increase_percent": (
                        100.0
                        * (
                            permuted_h_rmse
                            / max(
                                baseline_horizon_rmse[h],
                                1e-12,
                            )
                            - 1.0
                        )
                    ),
                })
ORACLE_XAI_WEATHER_DETAIL = pd.DataFrame(
    weather_rows
)
ORACLE_XAI_WEATHER_SUMMARY = (
    ORACLE_XAI_WEATHER_DETAIL
    .groupby(
        ["Site", "Variable"],
        as_index=False,
    )
    .agg(
        RMSE_increase_percent_mean=(
            "RMSE_increase_percent",
            "mean",
        ),
        RMSE_increase_percent_std=(
            "RMSE_increase_percent",
            "std",
        ),
    )
)
ORACLE_XAI_WEATHER_MACRO = (
    ORACLE_XAI_WEATHER_SUMMARY
    .groupby(
        "Variable",
        as_index=False,
    )
    .agg(
        Mean_RMSE_increase_percent=(
            "RMSE_increase_percent_mean",
            "mean",
        ),
        SD_across_sites=(
            "RMSE_increase_percent_mean",
            "std",
        ),
    )
    .sort_values(
        "Mean_RMSE_increase_percent",
        ascending=False,
    )
)
ORACLE_XAI_WEATHER_HORIZON_DETAIL = pd.DataFrame(
    weather_horizon_rows
)
ORACLE_XAI_WEATHER_HORIZON = (
    ORACLE_XAI_WEATHER_HORIZON_DETAIL
    .groupby(
        [
            "Site",
            "Variable",
            "Hour",
            "Horizon",
        ],
        as_index=False,
    )
    .agg(
        RMSE_increase_percent_mean=(
            "RMSE_increase_percent",
            "mean",
        ),
        RMSE_increase_percent_std=(
            "RMSE_increase_percent",
            "std",
        ),
    )
)


In [ ]:
# Oracle-Weather End-to-End Explainability — part 9/10
ORACLE_XAI_WEATHER_HORIZON_MACRO = (
    ORACLE_XAI_WEATHER_HORIZON
    .groupby(
        [
            "Variable",
            "Hour",
            "Horizon",
        ],
        as_index=False,
    )
    .agg(
        Mean_RMSE_increase_percent=(
            "RMSE_increase_percent_mean",
            "mean",
        ),
        SD_across_sites=(
            "RMSE_increase_percent_mean",
            "std",
        ),
    )
)
ORACLE_XAI_WEATHER_DETAIL.to_csv(
    ORACLE_XAI_DIR
    / "02_future_weather_variable_permutation_detail.csv",
    index=False,
)
ORACLE_XAI_WEATHER_SUMMARY.to_csv(
    ORACLE_XAI_DIR
    / "02_future_weather_variable_permutation_by_site.csv",
    index=False,
)
ORACLE_XAI_WEATHER_MACRO.to_csv(
    ORACLE_XAI_DIR
    / "02_future_weather_variable_permutation_macro.csv",
    index=False,
)
ORACLE_XAI_WEATHER_HORIZON.to_csv(
    ORACLE_XAI_DIR
    / "03_future_weather_horizon_importance_by_site.csv",
    index=False,
)
ORACLE_XAI_WEATHER_HORIZON_MACRO.to_csv(
    ORACLE_XAI_DIR
    / "03_future_weather_horizon_importance_macro.csv",
    index=False,
)
print("\n[Oracle future-weather variable importance]")
display(
    ORACLE_XAI_WEATHER_MACRO.round(4)
)


# ======================================================================
# 9. Figure B — Global future-weather variable importance
# ======================================================================
_weather_plot = (
    ORACLE_XAI_WEATHER_MACRO
    .sort_values(
        "Mean_RMSE_increase_percent",
        ascending=True,
    )
)
fig, ax = plt.subplots(
    figsize=(8.5, 5.7)
)
ax.barh(
    _weather_plot["Variable"],
    _weather_plot["Mean_RMSE_increase_percent"],
    xerr=_weather_plot["SD_across_sites"].fillna(0.0),
    capsize=3,
)
ax.axvline(
    0.0,
    linewidth=0.8,
)
ax.set_xlabel(
    "Increase in RMSE after permutation (%)"
)
ax.set_ylabel(
    ""
)
ax.set_title(
    "Oracle-Weather MUFASA: Meteorological Variable Importance"
)
ax.grid(
    axis="x",
    alpha=0.25,
)
fig.tight_layout()
fig.savefig(
    ORACLE_XAI_DIR
    / "Figure_XAI_B_Weather_Variable_Importance.png",
    dpi=FIG_DPI_XAI,
    bbox_inches="tight",
)
fig.savefig(
    ORACLE_XAI_DIR
    / "Figure_XAI_B_Weather_Variable_Importance.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


# ======================================================================
# 10. Figure C — Horizon-wise weather importance heatmap
# ======================================================================
heatmap = (
    ORACLE_XAI_WEATHER_HORIZON_MACRO
    .pivot(
        index="Variable",
        columns="Hour",
        values="Mean_RMSE_increase_percent",
    )
    .reindex(
        ORACLE_XAI_WEATHER_MACRO[
            "Variable"
        ].tolist()
    )
)
fig, ax = plt.subplots(
    figsize=(11.5, 6.2)
)
image = ax.imshow(
    heatmap.to_numpy(),
    aspect="auto",
)
ax.set_xticks(
    np.arange(
        len(heatmap.columns)
    )
)
ax.set_xticklabels(
    [
        f"{int(hour):02d}:00"
        for hour in heatmap.columns
    ],
    rotation=45,
    ha="right",
)
ax.set_yticks(
    np.arange(
        len(heatmap.index)
    )
)
ax.set_yticklabels(
    heatmap.index
)
ax.set_xlabel(
    "Forecast hour"
)
ax.set_ylabel(
    ""
)
ax.set_title(
    "Oracle-Weather MUFASA: Horizon-Wise Meteorological Importance"
)
cbar = fig.colorbar(
    image,
    ax=ax,
)
cbar.set_label(
    "Increase in horizon RMSE after permutation (%)"
)
fig.tight_layout()
fig.savefig(
    ORACLE_XAI_DIR
    / "Figure_XAI_C_Horizon_Weather_Heatmap.png",
    dpi=FIG_DPI_XAI,
    bbox_inches="tight",
)
fig.savefig(
    ORACLE_XAI_DIR
    / "Figure_XAI_C_Horizon_Weather_Heatmap.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


# ======================================================================

# ======================================================================
#


# ======================================================================
site_group_matrix = (
    ORACLE_XAI_GROUP_SUMMARY
    .pivot(
        index="Site",
        columns="Group",
        values="RMSE_increase_percent_mean",
    )
    .reindex(
        index=SITES,
        columns=XAI_GROUPS,
    )
)
fig, ax = plt.subplots(
    figsize=(11.5, 5.4)
)
image = ax.imshow(
    site_group_matrix.to_numpy(),
    aspect="auto",
)
ax.set_xticks(
    np.arange(
        len(site_group_matrix.columns)
    )
)
ax.set_xticklabels(
    site_group_matrix.columns,
    rotation=25,
    ha="right",
)


In [ ]:
# Oracle-Weather End-to-End Explainability — part 10/10
ax.set_yticks(
    np.arange(
        len(site_group_matrix.index)
    )
)
ax.set_yticklabels(
    site_group_matrix.index
)
ax.set_title(
    "Oracle-Weather MUFASA: Site-Level Grouped Importance"
)
cbar = fig.colorbar(
    image,
    ax=ax,
)
cbar.set_label(
    "Increase in RMSE after permutation (%)"
)
fig.tight_layout()
fig.savefig(
    ORACLE_XAI_DIR
    / "Figure_XAI_D_Site_Group_Heatmap.png",
    dpi=FIG_DPI_XAI,
    bbox_inches="tight",
)
fig.savefig(
    ORACLE_XAI_DIR
    / "Figure_XAI_D_Site_Group_Heatmap.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


# ======================================================================

# ======================================================================
top_group = (
    ORACLE_XAI_GROUP_MACRO
    .iloc[0]
)
top_weather = (
    ORACLE_XAI_WEATHER_MACRO
    .iloc[0]
)
top_weather_by_hour = (
    ORACLE_XAI_WEATHER_HORIZON_MACRO
    .sort_values(
        [
            "Hour",
            "Mean_RMSE_increase_percent",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "Hour",
        as_index=False,
    )
    .first()
)
summary_lines = []
summary_lines.append(
    "# MUFASA Oracle MUFASA XAI Summary"
)
summary_lines.append("")
summary_lines.append(
    "This XAI analysis was performed exclusively for the "
    "Oracle-Weather MUFASA MUFASA system across all six sites."
)
summary_lines.append("")
summary_lines.append(
    f"- Most influential information group: "
    f"{top_group['Group']} "
    f"(mean RMSE increase after permutation = "
    f"{top_group['Mean_RMSE_increase_percent']:.2f}%)."
)
summary_lines.append(
    f"- Most influential forward meteorological variable: "
    f"{top_weather['Variable']} "
    f"(mean RMSE increase after permutation = "
    f"{top_weather['Mean_RMSE_increase_percent']:.2f}%)."
)
summary_lines.append("")
summary_lines.append(
    "## Most influential meteorological variable by forecast hour"
)
for _, row in top_weather_by_hour.iterrows():

    summary_lines.append(
        f"- {int(row['Hour']):02d}:00: "
        f"{row['Variable']} "
        f"({row['Mean_RMSE_increase_percent']:.2f}% RMSE increase)"
    )
summary_lines.append("")
summary_lines.append(
    "Interpretation should be based on positive RMSE degradation after "
    "permutation. A larger degradation indicates stronger reliance of "
    "the final MUFASA forecast on that information."
)
summary_text = "\n".join(
    summary_lines
)
with open(
    ORACLE_XAI_DIR
    / "XAI_paper_summary.md",
    "w",
    encoding="utf-8",
) as file:

    file.write(
        summary_text
    )
print("\n" + summary_text)


# ======================================================================
# 13. XAI manifest
# ======================================================================
manifest_rows = []
for path in sorted(
    ORACLE_XAI_DIR.iterdir()
):

    if path.is_file():

        manifest_rows.append({
            "File": path.name,
            "Size_bytes": path.stat().st_size,
        })
manifest = pd.DataFrame(
    manifest_rows
)
manifest.to_csv(
    ORACLE_XAI_DIR
    / "XAI_output_manifest.csv",
    index=False,
)


# ======================================================================

# ======================================================================
zip_path = shutil.make_archive(
    str(ORACLE_XAI_DIR),
    "zip",
    root_dir=ORACLE_XAI_DIR,
)
print("\n" + "=" * 78)
print("ORACLE MUFASA_FUSION XAI COMPLETE")
print("=" * 78)
print("Folder :", ORACLE_XAI_DIR.resolve())
print("ZIP    :", zip_path)
print("=" * 78)
print(
    "Primary manuscript outputs:"
    "1. Figure_XAI_A_Grouped_Permutation.png\n"
    "2. Figure_XAI_B_Weather_Variable_Importance.png\n"
    "3. Figure_XAI_C_Horizon_Weather_Heatmap.png\n"
    "4. Figure_XAI_D_Site_Group_Heatmap.png\n"
)


## 10. Multi-horizon statistical analysis

Compute horizon-wise ranks, paired dependence-aware comparisons, multiplicity-adjusted evidence, and conservative model-set diagnostics.

**Run note.** Execute the cells in this section in order. Objects created here are consumed by later sections; the notebook intentionally avoids hidden state restoration from unpublished artifacts.


In [ ]:
# Multi-Horizon Statistical Analysis — part 1/16
# ======================================================================
# ORACLE WEATHER + ALL-CITY MUFASA_FUSION
# HORIZON-WISE + MULTI-HORIZON STATISTICAL ANALYSIS
# ======================================================================
#


#
# ORACLE_MUFASA_FUSION_TEST_PREDICTIONS
# ORACLE_MUFASA_FUSION_TEST_DF
# ORACLE_MODEL_PREDICTIONS
# ORACLE_RUN
# ORACLE_REFIT_BUNDLES
#


#    - paired Wilcoxon
#    - Newey-West HAC predictive-accuracy test
#    - moving-block bootstrap CI
#    - Holm correction


#    block-bootstrap Model Confidence Set (MCS; Tmax style)

#
# IMPORTANT


# ======================================================================

from pathlib import Path
import math
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import (
    wilcoxon,
    friedmanchisquare,
    rankdata,
)

# ----------------------------------------------------------------------

# ----------------------------------------------------------------------
STAT_DIR = OUTPUT_DIR / "oracle_statistics"
if STAT_DIR.exists():
    shutil.rmtree(STAT_DIR)
STAT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
print("=" * 86)
print("MUFASA Oracle MUFASA — Horizon / Statistical Analysis")
print("Output:", STAT_DIR.resolve())
print("=" * 86)


# ======================================================================

# ======================================================================
STAT_HAC_LAG = 7

# moving-block bootstrap
STAT_BLOCK_LENGTH = 7
STAT_BOOTSTRAP_REPS = 5000

# MCS confidence level
MCS_ALPHA = 0.05
MCS_BOOTSTRAP_REPS = 3000
STAT_SEED = int(
    globals().get(
        "SEED",
        42,
    )
)
INTERNAL_MODELS = {
    "MUFASA",
    "MUFASA-Core",
    "Ridge-Residual",
    "Solar-Geometry-Reference",
}
PROPOSED_NAME = "MUFASA"


# ======================================================================

# ======================================================================
required_objects = [
    "ORACLE_MUFASA_FUSION_TEST_PREDICTIONS",
    "ORACLE_MODEL_PREDICTIONS",
    "ORACLE_RUN",
    "ORACLE_REFIT_BUNDLES",
    "SITES",
    "TARGET_HOURS",
    "N_HORIZONS",
    "metric_set",
]
missing = [
    name
    for name in required_objects
    if name not in globals()
]
if missing:
    raise RuntimeError(
        "Oracle-weather prerequisite is missing. Run the preceding oracle-weather stage first."
        f"Missing objects = {missing}"
    )
if ORACLE_RUN is None:
    raise RuntimeError(
        "Oracle-weather prerequisite is missing. Run the preceding oracle-weather stage first."
    )
assert len(TARGET_HOURS) == N_HORIZONS
assert N_HORIZONS == 11, (
    f"Current horizons={N_HORIZONS}; "
    "Public-release execution note."
)
print("Sites       :", SITES)
print("Hours       :", TARGET_HOURS)
print("N horizons  :", N_HORIZONS)


# ======================================================================


# ======================================================================
def _holm_adjust_local(p_values):

    if "holm_adjust" in globals():

        return np.asarray(
            holm_adjust(p_values),
            dtype=float,
        )

    values = np.asarray(
        p_values,
        dtype=float,
    )

    adjusted = np.full(
        len(values),
        np.nan,
    )

    finite = np.where(
        np.isfinite(values)
    )[0]

    if len(finite) == 0:
        return adjusted

    order = finite[
        np.argsort(
            values[finite]
        )
    ]

    m = len(order)
    running = 0.0

    for rank, index in enumerate(order):

        candidate = min(
            1.0,
            (m - rank) * values[index],
        )

        running = max(
            running,
            candidate,
        )

        adjusted[index] = running

    return adjusted


In [ ]:
# Multi-Horizon Statistical Analysis — part 2/16
def _newey_west_test_local(
    differences,
    lag=7,
):

    if "newey_west_mean_test" in globals():

        return newey_west_mean_test(
            differences,
            lag=lag,
        )

    x = np.asarray(
        differences,
        dtype=float,
    )

    x = x[
        np.isfinite(x)
    ]

    n = len(x)

    if n < 10:
        return np.nan, np.nan

    mean = x.mean()

    centered = x - mean

    gamma0 = np.dot(
        centered,
        centered,
    ) / n

    long_run = gamma0

    max_lag = min(
        lag,
        n - 1,
    )

    for j in range(
        1,
        max_lag + 1,
    ):

        weight = (
            1.0
            - j / (max_lag + 1.0)
        )

        gamma_j = (
            np.dot(
                centered[j:],
                centered[:-j],
            )
            / n
        )

        long_run += (
            2.0
            * weight
            * gamma_j
        )

    long_run = max(
        long_run,
        1e-14,
    )

    se = math.sqrt(
        long_run / n
    )

    statistic = (
        mean / se
    )

    # normal approximation
    from scipy.stats import norm

    p_two = (
        2.0
        * (
            1.0
            - norm.cdf(
                abs(statistic)
            )
        )
    )

    return (
        float(statistic),
        float(p_two),
    )
def _moving_block_ci_local(
    differences,
    block_length=7,
    repetitions=5000,
    seed=42,
):

    if "moving_block_mean_ci" in globals():

        return moving_block_mean_ci(
            differences,
            block_length=block_length,
            repetitions=repetitions,
            seed=seed,
        )

    values = np.asarray(
        differences,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    n = len(values)

    if n == 0:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    block_length = min(
        block_length,
        n,
    )

    starts = np.arange(
        0,
        n - block_length + 1,
    )

    rng = np.random.default_rng(
        seed
    )

    blocks_needed = int(
        math.ceil(
            n / block_length
        )
    )

    boot = np.empty(
        repetitions,
        dtype=float,
    )

    for b in range(
        repetitions
    ):

        selected = rng.choice(
            starts,
            size=blocks_needed,
            replace=True,
        )

        sample = np.concatenate([
            values[
                s:s + block_length
            ]
            for s in selected
        ])[:n]

        boot[b] = sample.mean()

    return (
        float(
            values.mean()
        ),
        float(
            np.quantile(
                boot,
                0.025,
            )
        ),
        float(
            np.quantile(
                boot,
                0.975,
            )
        ),
    )


# ======================================================================
# 4. generalized multi-horizon DM fallback
# ======================================================================


In [ ]:
# Multi-Horizon Statistical Analysis — part 3/16
def _newey_west_covariance_local(
    matrix,
    lag=7,
):

    x = np.asarray(
        matrix,
        dtype=float,
    )

    mask = np.isfinite(
        x
    ).all(axis=1)

    x = x[mask]

    n, k = x.shape

    if n < max(
        20,
        k + 2,
    ):
        return None

    centered = (
        x
        - x.mean(
            axis=0,
            keepdims=True,
        )
    )

    cov = (
        centered.T
        @ centered
        / n
    )

    max_lag = min(
        lag,
        n - 1,
    )

    for ell in range(
        1,
        max_lag + 1,
    ):

        weight = (
            1.0
            - ell
            / (
                max_lag + 1.0
            )
        )

        gamma = (
            centered[ell:].T
            @ centered[:-ell]
            / n
        )

        cov += (
            weight
            * (
                gamma
                + gamma.T
            )
        )

    return cov
def _generalized_multihorizon_dm_local(
    loss_differential,
    lag=7,
):

    if "generalized_multihorizon_dm" in globals():

        return generalized_multihorizon_dm(
            loss_differential,
            lag=lag,
        )

    from scipy.stats import chi2

    values = np.asarray(
        loss_differential,
        dtype=float,
    )

    values = values[
        np.isfinite(values).all(
            axis=1
        )
    ]

    covariance = (
        _newey_west_covariance_local(
            values,
            lag=lag,
        )
    )

    if covariance is None:
        return (
            np.nan,
            np.nan,
            0,
        )

    mean_vector = values.mean(
        axis=0
    )

    covariance = (
        covariance
        + 1e-10
        * np.eye(
            covariance.shape[0]
        )
    )

    statistic = float(
        len(values)
        * mean_vector
        @ np.linalg.pinv(
            covariance
        )
        @ mean_vector
    )

    rank = int(
        np.linalg.matrix_rank(
            covariance
        )
    )

    p_value = float(
        chi2.sf(
            statistic,
            max(
                rank,
                1,
            ),
        )
    )

    return (
        statistic,
        p_value,
        rank,
    )


# ======================================================================

# ======================================================================
#


# ======================================================================
if (
    "DEEP_BENCHMARKS"
    in globals()
):

    desired_comparators = list(
        DEEP_BENCHMARKS
    )

else:

    first_site = SITES[0]

    desired_comparators = [
        model
        for model in ORACLE_MODEL_PREDICTIONS[
            first_site
        ].keys()
        if model
        not in INTERNAL_MODELS
    ]
COMPARATORS = []
for model in desired_comparators:

    available_all = all(
        model
        in ORACLE_MODEL_PREDICTIONS[
            site
        ]
        for site in SITES
    )

    if available_all:
        COMPARATORS.append(
            model
        )
if not COMPARATORS:

    raise RuntimeError(
        "Oracle-weather prerequisite is missing. Run the preceding oracle-weather stage first."
    )
print(
    f"\nExternal comparator count = {len(COMPARATORS)}"
)
print(
    COMPARATORS
)


# ======================================================================

# ======================================================================
#


# ======================================================================
STAT_PREDICTIONS = {}


In [ ]:
# Multi-Horizon Statistical Analysis — part 4/16
for site in SITES:

    STAT_PREDICTIONS[site] = {
        PROPOSED_NAME:
            np.asarray(
                ORACLE_MUFASA_FUSION_TEST_PREDICTIONS[
                    site
                ],
                dtype=float,
            )
    }

    for model in COMPARATORS:

        STAT_PREDICTIONS[
            site
        ][model] = np.asarray(
            ORACLE_MODEL_PREDICTIONS[
                site
            ][model],
            dtype=float,
        )


# Shape audit
for site in SITES:

    truth = np.asarray(
        ORACLE_RUN["results"][
            site
        ]["truth"],
        dtype=float,
    )

    expected = truth.shape

    for model, pred in (
        STAT_PREDICTIONS[
            site
        ].items()
    ):

        if pred.shape != expected:

            raise RuntimeError(
                f"{site} / {model}: "
                f"prediction shape={pred.shape}, "
                f"truth shape={expected}"
            )

        if not np.isfinite(
            pred
        ).all():

            raise FloatingPointError(
                f"{site}/{model}: NaN/Inf prediction."
            )
print("\nPrediction shape audit passed.")


# ======================================================================
# 7. Horizon-wise RMSE / MAE / R2
# ======================================================================
horizon_metric_rows = []
for site in SITES:

    truth = np.asarray(
        ORACLE_RUN["results"][
            site
        ]["truth"],
        dtype=float,
    )

    for model, prediction in (
        STAT_PREDICTIONS[
            site
        ].items()
    ):

        for h, hour in enumerate(
            TARGET_HOURS
        ):

            metrics = metric_set(
                truth[:, h],
                prediction[:, h],
            )

            horizon_metric_rows.append({
                "Site": site,
                "Model": model,
                "Horizon": h + 1,
                "Hour": int(hour),
                **metrics,
            })
HORIZON_METRICS = pd.DataFrame(
    horizon_metric_rows
)


# ----------------------------------------------------------------------

# ----------------------------------------------------------------------
HORIZON_METRICS[
    "Rank_RMSE"
] = (
    HORIZON_METRICS
    .groupby(
        ["Site", "Horizon"]
    )["RMSE"]
    .rank(
        method="min",
    )
)
HORIZON_METRICS[
    "Rank_MAE"
] = (
    HORIZON_METRICS
    .groupby(
        ["Site", "Horizon"]
    )["MAE"]
    .rank(
        method="min",
    )
)
HORIZON_METRICS.to_csv(
    STAT_DIR
    / "01_horizon_metrics_all_models.csv",
    index=False,
)


# ======================================================================
# 8. MUFASA horizon summary
# ======================================================================
MUFASA_HORIZON = (
    HORIZON_METRICS[
        HORIZON_METRICS[
            "Model"
        ].eq(
            PROPOSED_NAME
        )
    ]
    .copy()
)
MUFASA_HORIZON_SUMMARY = (
    MUFASA_HORIZON
    .groupby(
        "Site",
        as_index=False,
    )
    .agg(
        Mean_Horizon_RMSE=(
            "RMSE",
            "mean",
        ),
        Mean_RMSE_Rank=(
            "Rank_RMSE",
            "mean",
        ),
        Rank1_Horizons=(
            "Rank_RMSE",
            lambda x:
                int(
                    np.sum(
                        np.asarray(x)
                        == 1
                    )
                ),
        ),
        Top2_Horizons=(
            "Rank_RMSE",
            lambda x:
                int(
                    np.sum(
                        np.asarray(x)
                        <= 2
                    )
                ),
        ),
        Best_Horizon_RMSE=(
            "RMSE",
            "min",
        ),
        Worst_Horizon_RMSE=(
            "RMSE",
            "max",
        ),
    )
)
MUFASA_HORIZON_SUMMARY[
    "Total_Horizons"
] = N_HORIZONS
MUFASA_HORIZON_SUMMARY.to_csv(
    STAT_DIR
    / "02_MUFASA_horizon_rank_summary.csv",
    index=False,
)
print(
    "\n[MUFASA horizon rank summary]"
)


In [ ]:
# Multi-Horizon Statistical Analysis — part 5/16
display(
    MUFASA_HORIZON_SUMMARY
    .round(5)
)


# ======================================================================

# ======================================================================
#

#
# d_t =
#   MUFASA squared error
#   -
#   comparator squared error
#

#

# ======================================================================
horizon_test_rows = []


In [ ]:
# Multi-Horizon Statistical Analysis — part 6/16
for site in SITES:

    truth = np.asarray(
        ORACLE_RUN["results"][
            site
        ]["truth"],
        dtype=float,
    )

    proposed = (
        STAT_PREDICTIONS[
            site
        ][PROPOSED_NAME]
    )

    for comparator in COMPARATORS:

        comparison = (
            STAT_PREDICTIONS[
                site
            ][comparator]
        )

        for h, hour in enumerate(
            TARGET_HOURS
        ):

            proposed_error = (
                proposed[:, h]
                - truth[:, h]
            )

            comparator_error = (
                comparison[:, h]
                - truth[:, h]
            )

            proposed_sq = (
                proposed_error ** 2
            )

            comparator_sq = (
                comparator_error ** 2
            )

            diff = (
                proposed_sq
                - comparator_sq
            )

            # ----------------------------------------------
            # RMSE
            # ----------------------------------------------
            rmse_m = float(
                np.sqrt(
                    np.mean(
                        proposed_sq
                    )
                )
            )

            rmse_c = float(
                np.sqrt(
                    np.mean(
                        comparator_sq
                    )
                )
            )

            improvement = (
                100.0
                * (
                    rmse_c
                    - rmse_m
                )
                / max(
                    rmse_c,
                    1e-12,
                )
            )

            # ----------------------------------------------
            # Wilcoxon: H1 = MUFASA loss < comparator loss
            # ----------------------------------------------
            try:

                wilcoxon_result = (
                    wilcoxon(
                        diff,
                        zero_method="zsplit",
                        alternative="less",
                    )
                )

                wilcoxon_stat = float(
                    wilcoxon_result.statistic
                )

                wilcoxon_p = float(
                    wilcoxon_result.pvalue
                )

            except Exception:

                wilcoxon_stat = np.nan
                wilcoxon_p = 1.0

            # ----------------------------------------------
            # Newey-West HAC
            # ----------------------------------------------
            hac_stat, hac_p_two = (
                _newey_west_test_local(
                    diff,
                    lag=STAT_HAC_LAG,
                )
            )

            # directional one-sided p:
            # negative statistic means MUFASA is better
            if np.isfinite(
                hac_stat
            ):

                from scipy.stats import norm

                hac_p_one = float(
                    norm.cdf(
                        hac_stat
                    )
                )

            else:
                hac_p_one = np.nan

            # ----------------------------------------------
            # Moving-block bootstrap CI
            # ----------------------------------------------
            (
                mean_diff,
                ci_low,
                ci_high,
            ) = _moving_block_ci_local(
                diff,
                block_length=STAT_BLOCK_LENGTH,
                repetitions=STAT_BOOTSTRAP_REPS,
                seed=(
                    STAT_SEED
                    + h
                    + sum(
                        map(
                            ord,
                            site
                            + comparator,
                        )
                    )
                ),
            )

            horizon_test_rows.append({
                "Site": site,
                "Comparator": comparator,
                "Horizon": h + 1,
                "Hour": int(hour),

                "MUFASA_RMSE": rmse_m,
                "Comparator_RMSE": rmse_c,

                "RMSE_reduction_percent":
                    improvement,

                "Mean_squared_loss_diff":
                    mean_diff,

                "MUFASA_daily_win_rate":
                    float(
                        np.mean(
                            diff < 0
                        )
                    ),

                "Wilcoxon_stat":
                    wilcoxon_stat,

                "Wilcoxon_p_raw":
                    wilcoxon_p,

                "HAC_stat":
                    hac_stat,

                "HAC_p_one_sided_raw":
                    hac_p_one,

                "HAC_p_two_sided_raw":
                    hac_p_two,

                "Block_CI_low":
                    ci_low,

                "Block_CI_high":
                    ci_high,
            })
HORIZON_TESTS = pd.DataFrame(
    horizon_test_rows
)


# ======================================================================
# 10. Horizon-specific multiple testing correction
# ======================================================================
#

# Holm correction.
# ======================================================================


In [ ]:
# Multi-Horizon Statistical Analysis — part 7/16
for (
    site,
    comparator
), index in (
    HORIZON_TESTS
    .groupby(
        [
            "Site",
            "Comparator",
        ]
    )
    .groups
    .items()
):

    idx = list(index)

    HORIZON_TESTS.loc[
        idx,
        "Wilcoxon_p_Holm_11H",
    ] = _holm_adjust_local(
        HORIZON_TESTS.loc[
            idx,
            "Wilcoxon_p_raw",
        ]
        .to_numpy()
    )

    HORIZON_TESTS.loc[
        idx,
        "HAC_p_Holm_11H",
    ] = _holm_adjust_local(
        HORIZON_TESTS.loc[
            idx,
            "HAC_p_one_sided_raw",
        ]
        .to_numpy()
    )
HORIZON_TESTS[
    "Wilcoxon_significant"
] = (
    HORIZON_TESTS[
        "Wilcoxon_p_Holm_11H"
    ] < 0.05
)
HORIZON_TESTS[
    "HAC_significant"
] = (
    HORIZON_TESTS[
        "HAC_p_Holm_11H"
    ] < 0.05
)
HORIZON_TESTS[
    "Block_bootstrap_support"
] = (
    HORIZON_TESTS[
        "Block_CI_high"
    ] < 0
)
HORIZON_TESTS[
    "Strong_horizon_support"
] = (
    (
        HORIZON_TESTS[
            "RMSE_reduction_percent"
        ] > 0
    )
    & HORIZON_TESTS[
        "Wilcoxon_significant"
    ]
    & HORIZON_TESTS[
        "HAC_significant"
    ]
    & HORIZON_TESTS[
        "Block_bootstrap_support"
    ]
)
HORIZON_TESTS.to_csv(
    STAT_DIR
    / "03_horizon_specific_statistical_tests.csv",
    index=False,
)


# ======================================================================

# ======================================================================
HORIZON_TEST_SUMMARY = (
    HORIZON_TESTS
    .groupby(
        [
            "Site",
            "Comparator",
        ],
        as_index=False,
    )
    .agg(
        Mean_RMSE_reduction_percent=(
            "RMSE_reduction_percent",
            "mean",
        ),
        Horizons_RMSE_better=(
            "RMSE_reduction_percent",
            lambda x:
                int(
                    np.sum(
                        np.asarray(x)
                        > 0
                    )
                ),
        ),
        Horizons_Wilcoxon_sig=(
            "Wilcoxon_significant",
            "sum",
        ),
        Horizons_HAC_sig=(
            "HAC_significant",
            "sum",
        ),
        Horizons_Bootstrap_sig=(
            "Block_bootstrap_support",
            "sum",
        ),
        Horizons_Strong_support=(
            "Strong_horizon_support",
            "sum",
        ),
        Mean_daily_win_rate=(
            "MUFASA_daily_win_rate",
            "mean",
        ),
    )
)
for column in [
    "Horizons_Wilcoxon_sig",
    "Horizons_HAC_sig",
    "Horizons_Bootstrap_sig",
    "Horizons_Strong_support",
]:

    HORIZON_TEST_SUMMARY[
        column
    ] = (
        HORIZON_TEST_SUMMARY[
            column
        ]
        .astype(int)
    )
HORIZON_TEST_SUMMARY[
    "Total_Horizons"
] = N_HORIZONS
HORIZON_TEST_SUMMARY.to_csv(
    STAT_DIR
    / "04_horizon_superiority_summary.csv",
    index=False,
)


# ======================================================================
# 12. 11-horizon JOINT generalized DM
# ======================================================================
#
# loss differential matrix shape:
# days × 11
#


# ======================================================================
joint_rows = []


In [ ]:
# Multi-Horizon Statistical Analysis — part 8/16
for site in SITES:

    truth = np.asarray(
        ORACLE_RUN["results"][
            site
        ]["truth"],
        dtype=float,
    )

    proposed = (
        STAT_PREDICTIONS[
            site
        ][PROPOSED_NAME]
    )

    for comparator in COMPARATORS:

        comparison = (
            STAT_PREDICTIONS[
                site
            ][comparator]
        )

        horizon_diff = (
            (proposed - truth) ** 2
            -
            (comparison - truth) ** 2
        )

        daily_average_diff = (
            horizon_diff.mean(
                axis=1
            )
        )

        (
            gdm_stat,
            gdm_p,
            covariance_rank,
        ) = (
            _generalized_multihorizon_dm_local(
                horizon_diff,
                lag=STAT_HAC_LAG,
            )
        )

        
        hac_stat, hac_p_two = (
            _newey_west_test_local(
                daily_average_diff,
                lag=STAT_HAC_LAG,
            )
        )

        if np.isfinite(
            hac_stat
        ):

            from scipy.stats import norm

            hac_p_one = float(
                norm.cdf(
                    hac_stat
                )
            )

        else:
            hac_p_one = np.nan

        try:

            path_wilcoxon = float(
                wilcoxon(
                    daily_average_diff,
                    zero_method="zsplit",
                    alternative="less",
                ).pvalue
            )

        except Exception:

            path_wilcoxon = 1.0

        (
            mean_diff,
            ci_low,
            ci_high,
        ) = _moving_block_ci_local(
            daily_average_diff,
            block_length=STAT_BLOCK_LENGTH,
            repetitions=STAT_BOOTSTRAP_REPS,
            seed=(
                STAT_SEED
                + 50000
                + sum(
                    map(
                        ord,
                        site
                        + comparator,
                    )
                )
            ),
        )

        m_rmse = metric_set(
            truth,
            proposed,
        )["RMSE"]

        c_rmse = metric_set(
            truth,
            comparison,
        )["RMSE"]

        joint_rows.append({
            "Site": site,
            "Comparator": comparator,

            "MUFASA_RMSE":
                m_rmse,

            "Comparator_RMSE":
                c_rmse,

            "RMSE_reduction_percent":
                100.0
                * (
                    c_rmse
                    - m_rmse
                )
                / max(
                    c_rmse,
                    1e-12,
                ),

            "GDM_stat":
                gdm_stat,

            "GDM_p_raw":
                gdm_p,

            "GDM_covariance_rank":
                covariance_rank,

            "Average_path_Wilcoxon_p_raw":
                path_wilcoxon,

            "Average_path_HAC_stat":
                hac_stat,

            "Average_path_HAC_p_one_raw":
                hac_p_one,

            "Average_path_HAC_p_two_raw":
                hac_p_two,

            "Average_path_loss_diff":
                mean_diff,

            "Block_CI_low":
                ci_low,

            "Block_CI_high":
                ci_high,

            "Daily_path_win_rate":
                float(
                    np.mean(
                        daily_average_diff
                        < 0
                    )
                ),
        })
JOINT_TESTS = pd.DataFrame(
    joint_rows
)


# ======================================================================
# 13. Joint tests Holm correction
# ======================================================================
#

# ======================================================================


In [ ]:
# Multi-Horizon Statistical Analysis — part 9/16
for site, index in (
    JOINT_TESTS
    .groupby(
        "Site"
    )
    .groups
    .items()
):

    idx = list(index)

    JOINT_TESTS.loc[
        idx,
        "GDM_p_Holm",
    ] = _holm_adjust_local(
        JOINT_TESTS.loc[
            idx,
            "GDM_p_raw",
        ]
        .to_numpy()
    )

    JOINT_TESTS.loc[
        idx,
        "Path_Wilcoxon_p_Holm",
    ] = _holm_adjust_local(
        JOINT_TESTS.loc[
            idx,
            "Average_path_Wilcoxon_p_raw",
        ]
        .to_numpy()
    )

    JOINT_TESTS.loc[
        idx,
        "Path_HAC_p_Holm",
    ] = _holm_adjust_local(
        JOINT_TESTS.loc[
            idx,
            "Average_path_HAC_p_one_raw",
        ]
        .to_numpy()
    )
JOINT_TESTS[
    "Numeric_superiority"
] = (
    JOINT_TESTS[
        "RMSE_reduction_percent"
    ] > 0
)
JOINT_TESTS[
    "GDM_significant_Holm"
] = (
    JOINT_TESTS[
        "GDM_p_Holm"
    ] < 0.05
)
JOINT_TESTS[
    "Path_Wilcoxon_significant_Holm"
] = (
    JOINT_TESTS[
        "Path_Wilcoxon_p_Holm"
    ] < 0.05
)
JOINT_TESTS[
    "Path_HAC_significant_Holm"
] = (
    JOINT_TESTS[
        "Path_HAC_p_Holm"
    ] < 0.05
)
JOINT_TESTS[
    "Bootstrap_support"
] = (
    JOINT_TESTS[
        "Block_CI_high"
    ] < 0
)
JOINT_TESTS[
    "Strong_joint_support"
] = (
    JOINT_TESTS[
        "Numeric_superiority"
    ]
    & JOINT_TESTS[
        "GDM_significant_Holm"
    ]
    & JOINT_TESTS[
        "Path_HAC_significant_Holm"
    ]
    & JOINT_TESTS[
        "Bootstrap_support"
    ]
)
JOINT_TESTS.to_csv(
    STAT_DIR
    / "05_joint_11h_multihorizon_tests.csv",
    index=False,
)


# ======================================================================
# 14. Friedman test — 11 horizon RMSE rank consistency
# ======================================================================
#


#


# ======================================================================
friedman_rows = []
horizon_rmse_wilcoxon_rows = []


In [ ]:
# Multi-Horizon Statistical Analysis — part 10/16
for site in SITES:

    site_frame = (
        HORIZON_METRICS[
            HORIZON_METRICS[
                "Site"
            ].eq(site)
        ]
    )

    models = [
        PROPOSED_NAME
    ] + COMPARATORS

    rmse_vectors = {}

    for model in models:

        vector = (
            site_frame[
                site_frame[
                    "Model"
                ].eq(model)
            ]
            .sort_values(
                "Horizon"
            )["RMSE"]
            .to_numpy(
                dtype=float
            )
        )

        if len(vector) != N_HORIZONS:

            raise RuntimeError(
                f"{site}/{model}: "
                f"expected {N_HORIZONS} "
                f"horizon RMSE values, "
                f"found {len(vector)}"
            )

        rmse_vectors[
            model
        ] = vector

    # Friedman omnibus
    try:

        friedman = friedmanchisquare(
            *[
                rmse_vectors[
                    model
                ]
                for model in models
            ]
        )

        friedman_stat = float(
            friedman.statistic
        )

        friedman_p = float(
            friedman.pvalue
        )

    except Exception:

        friedman_stat = np.nan
        friedman_p = np.nan

    
    matrix = np.column_stack([
        rmse_vectors[
            model
        ]
        for model in models
    ])

    ranks = np.vstack([
        rankdata(
            row,
            method="average",
        )
        for row in matrix
    ])

    average_ranks = ranks.mean(
        axis=0
    )

    rank1_counts = (
        ranks == 1
    ).sum(
        axis=0
    )

    for j, model in enumerate(
        models
    ):

        friedman_rows.append({
            "Site": site,
            "Model": model,
            "Friedman_stat":
                friedman_stat,
            "Friedman_p":
                friedman_p,
            "Average_RMSE_Rank":
                float(
                    average_ranks[j]
                ),
            "Rank1_Horizons":
                int(
                    rank1_counts[j]
                ),
            "Total_Horizons":
                N_HORIZONS,
        })

    # --------------------------------------------------------------
    # supplementary:
    
    # --------------------------------------------------------------
    mufasa_rmse = (
        rmse_vectors[
            PROPOSED_NAME
        ]
    )

    for comparator in COMPARATORS:

        comparator_rmse = (
            rmse_vectors[
                comparator
            ]
        )

        try:

            test = wilcoxon(
                mufasa_rmse,
                comparator_rmse,
                alternative="less",
                zero_method="zsplit",
            )

            p = float(
                test.pvalue
            )

            stat = float(
                test.statistic
            )

        except Exception:

            p = 1.0
            stat = np.nan

        horizon_rmse_wilcoxon_rows.append({
            "Site": site,
            "Comparator": comparator,
            "Statistic": stat,
            "p_raw": p,
            "MUFASA_mean_horizon_RMSE":
                float(
                    mufasa_rmse.mean()
                ),
            "Comparator_mean_horizon_RMSE":
                float(
                    comparator_rmse.mean()
                ),
            "MUFASA_better_horizons":
                int(
                    np.sum(
                        mufasa_rmse
                        < comparator_rmse
                    )
                ),
            "Total_horizons":
                N_HORIZONS,
        })
FRIEDMAN_TABLE = pd.DataFrame(
    friedman_rows
)
HORIZON_RMSE_WILCOXON = pd.DataFrame(
    horizon_rmse_wilcoxon_rows
)


# Holm across comparators by site
for site, index in (
    HORIZON_RMSE_WILCOXON
    .groupby(
        "Site"
    )
    .groups
    .items()
):

    idx = list(index)

    HORIZON_RMSE_WILCOXON.loc[
        idx,
        "p_Holm",
    ] = _holm_adjust_local(
        HORIZON_RMSE_WILCOXON.loc[
            idx,
            "p_raw",
        ]
        .to_numpy()
    )
HORIZON_RMSE_WILCOXON[
    "Significant_Holm"
] = (
    HORIZON_RMSE_WILCOXON[
        "p_Holm"
    ] < 0.05
)


In [ ]:
# Multi-Horizon Statistical Analysis — part 11/16
FRIEDMAN_TABLE.to_csv(
    STAT_DIR
    / "06_Friedman_horizon_rank_analysis.csv",
    index=False,
)
HORIZON_RMSE_WILCOXON.to_csv(
    STAT_DIR
    / "07_horizon_RMSE_Wilcoxon_supplementary.csv",
    index=False,
)


# ======================================================================
# 15. Model Confidence Set
#     daily average 11-horizon squared loss
# ======================================================================
#
# Hansen-Lunde-Nason-style sequential Tmax MCS.
#
# loss unit:

#

#


# ======================================================================
def _moving_block_indices(
    n,
    block_length,
    rng,
):

    block_length = min(
        int(block_length),
        n,
    )

    starts = np.arange(
        0,
        n - block_length + 1,
    )

    blocks_needed = int(
        math.ceil(
            n / block_length
        )
    )

    selected = rng.choice(
        starts,
        size=blocks_needed,
        replace=True,
    )

    indices = np.concatenate([
        np.arange(
            s,
            s + block_length,
        )
        for s in selected
    ])[:n]

    return indices


In [ ]:
# Multi-Horizon Statistical Analysis — part 12/16
def block_bootstrap_mcs_tmax(
    loss_df,
    alpha=0.05,
    block_length=7,
    repetitions=3000,
    seed=42,
):
    """
    Self-contained block-bootstrap Tmax Model Confidence Set.

    Parameters
    ----------
    loss_df:
        rows = time/day
        columns = models
        values = loss (lower is better)

    Returns
    -------
    elimination_df
    retained_models
    """

    loss_df = (
        loss_df
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna(
            axis=0,
            how="any",
        )
    )

    active = list(
        loss_df.columns
    )

    rng = np.random.default_rng(
        seed
    )

    elimination_rows = []

    step = 0

    while len(active) > 1:

        step += 1

        losses = (
            loss_df[
                active
            ]
            .to_numpy(
                dtype=float
            )
        )

        n, m = losses.shape

        # ----------------------------------------------------------
        # model i vs average of competing models
        #
        # d_i,t = L_i,t - average_j L_j,t
        # positive = model i is worse
        # ----------------------------------------------------------
        centered_against_models = (
            losses
            - losses.mean(
                axis=1,
                keepdims=True,
            )
        )

        d_bar = (
            centered_against_models
            .mean(
                axis=0
            )
        )

        # ----------------------------------------------------------
        # bootstrap standard error for d_bar
        # ----------------------------------------------------------
        boot_dbar = np.empty(
            (
                repetitions,
                m,
            ),
            dtype=float,
        )

        # null centering
        null_series = (
            centered_against_models
            - d_bar[
                None,
                :
            ]
        )

        for b in range(
            repetitions
        ):

            idx = (
                _moving_block_indices(
                    n,
                    block_length,
                    rng,
                )
            )

            boot_dbar[
                b
            ] = (
                null_series[
                    idx
                ]
                .mean(
                    axis=0
                )
            )

        se = boot_dbar.std(
            axis=0,
            ddof=1,
        )

        se = np.where(
            se > 1e-12,
            se,
            1e-12,
        )

        t_obs = (
            d_bar
            / se
        )

        t_boot = (
            boot_dbar
            / se[
                None,
                :
            ]
        )

        Tmax_obs = float(
            np.max(
                t_obs
            )
        )

        Tmax_boot = np.max(
            t_boot,
            axis=1,
        )

        p_value = float(
            (
                1
                + np.sum(
                    Tmax_boot
                    >= Tmax_obs
                )
            )
            / (
                repetitions
                + 1
            )
        )

        worst_position = int(
            np.argmax(
                t_obs
            )
        )

        worst_model = (
            active[
                worst_position
            ]
        )

        # ----------------------------------------------------------
        
        # ----------------------------------------------------------
        if p_value >= alpha:

            elimination_rows.append({
                "Step": step,
                "Models_remaining_before":
                    len(active),
                "EPA_Tmax":
                    Tmax_obs,
                "EPA_p_value":
                    p_value,
                "Eliminated_model":
                    "",
                "Decision":
                    "STOP_RETAIN_SET",
                "Models_remaining_after":
                    len(active),
            })

            break

        # ----------------------------------------------------------
        
        # ----------------------------------------------------------
        elimination_rows.append({
            "Step": step,
            "Models_remaining_before":
                len(active),
            "EPA_Tmax":
                Tmax_obs,
            "EPA_p_value":
                p_value,
            "Eliminated_model":
                worst_model,
            "Decision":
                "ELIMINATE",
            "Models_remaining_after":
                len(active) - 1,
        })

        active.remove(
            worst_model
        )

    return (
        pd.DataFrame(
            elimination_rows
        ),
        active,
    )


In [ ]:
# Multi-Horizon Statistical Analysis — part 13/16
MCS_ROWS = []
MCS_ELIMINATION_ROWS = []
for site in SITES:

    truth = np.asarray(
        ORACLE_RUN["results"][
            site
        ]["truth"],
        dtype=float,
    )

    models = [
        PROPOSED_NAME
    ] + COMPARATORS

    daily_losses = {}

    for model in models:

        pred = (
            STAT_PREDICTIONS[
                site
            ][model]
        )

        # day-level average 11-step squared loss
        daily_losses[
            model
        ] = np.mean(
            (
                pred
                - truth
            ) ** 2,
            axis=1,
        )

    daily_loss_df = pd.DataFrame(
        daily_losses
    )

    elimination_df, retained = (
        block_bootstrap_mcs_tmax(
            daily_loss_df,
            alpha=MCS_ALPHA,
            block_length=STAT_BLOCK_LENGTH,
            repetitions=MCS_BOOTSTRAP_REPS,
            seed=(
                STAT_SEED
                + 80000
                + sum(
                    map(
                        ord,
                        site,
                    )
                )
            ),
        )
    )

    if len(
        elimination_df
    ):

        elimination_df[
            "Site"
        ] = site

        MCS_ELIMINATION_ROWS.append(
            elimination_df
        )

    for model in models:

        MCS_ROWS.append({
            "Site": site,
            "Model": model,
            "Retained_in_95pct_MCS":
                model in retained,
            "MCS_set_size":
                len(retained),
            "MCS_alpha":
                MCS_ALPHA,
            "Mean_daily_11h_squared_loss":
                float(
                    daily_loss_df[
                        model
                    ].mean()
                ),
        })
MCS_TABLE = pd.DataFrame(
    MCS_ROWS
)
if MCS_ELIMINATION_ROWS:

    MCS_ELIMINATION = pd.concat(
        MCS_ELIMINATION_ROWS,
        ignore_index=True,
    )

else:

    MCS_ELIMINATION = pd.DataFrame()
MCS_TABLE.to_csv(
    STAT_DIR
    / "08_model_confidence_set_95pct.csv",
    index=False,
)
MCS_ELIMINATION.to_csv(
    STAT_DIR
    / "09_MCS_elimination_path.csv",
    index=False,
)


# ======================================================================
# 16. MCS summary
# ======================================================================
MCS_SUMMARY = (
    MCS_TABLE[
        MCS_TABLE[
            "Retained_in_95pct_MCS"
        ]
    ]
    .groupby(
        "Site"
    )["Model"]
    .apply(
        lambda x:
            " | ".join(
                x.tolist()
            )
    )
    .reset_index(
        name="Retained_models"
    )
)
MCS_SIZE = (
    MCS_TABLE
    .groupby(
        "Site",
        as_index=False,
    )[
        "MCS_set_size"
    ]
    .first()
)
MCS_SUMMARY = (
    MCS_SUMMARY
    .merge(
        MCS_SIZE,
        on="Site",
        how="left",
    )
)
MCS_SUMMARY[
    "MUFASA_retained"
] = (
    MCS_SUMMARY[
        "Retained_models"
    ]
    .str.contains(
        PROPOSED_NAME,
        regex=False,
    )
)
MCS_SUMMARY[
    "MUFASA_only_model"
] = (
    MCS_SUMMARY[
        "Retained_models"
    ].eq(
        PROPOSED_NAME
    )
)
MCS_SUMMARY.to_csv(
    STAT_DIR
    / "10_MCS_summary.csv",
    index=False,
)
print(
    "\n[95% average-path MCS]"
)
display(
    MCS_SUMMARY
)


# ======================================================================

# ======================================================================
compact_rows = []


In [ ]:
# Multi-Horizon Statistical Analysis — part 14/16
for site in SITES:

    horizon_site = (
        MUFASA_HORIZON_SUMMARY[
            MUFASA_HORIZON_SUMMARY[
                "Site"
            ].eq(site)
        ]
        .iloc[0]
    )

    joint_site = (
        JOINT_TESTS[
            JOINT_TESTS[
                "Site"
            ].eq(site)
        ]
    )

    mcs_site = (
        MCS_SUMMARY[
            MCS_SUMMARY[
                "Site"
            ].eq(site)
        ]
        .iloc[0]
    )

    compact_rows.append({
        "Site": site,

        "Rank1_horizons":
            int(
                horizon_site[
                    "Rank1_Horizons"
                ]
            ),

        "Mean_horizon_rank":
            float(
                horizon_site[
                    "Mean_RMSE_Rank"
                ]
            ),

        "External_comparator_count":
            len(
                COMPARATORS
            ),

        "Joint_GDM_sig_comparisons":
            int(
                joint_site[
                    "GDM_significant_Holm"
                ].sum()
            ),

        "Strong_joint_support_comparisons":
            int(
                joint_site[
                    "Strong_joint_support"
                ].sum()
            ),

        "MCS_set_size":
            int(
                mcs_site[
                    "MCS_set_size"
                ]
            ),

        "MUFASA_retained_in_MCS":
            bool(
                mcs_site[
                    "MUFASA_retained"
                ]
            ),

        "MUFASA_only_in_MCS":
            bool(
                mcs_site[
                    "MUFASA_only_model"
                ]
            ),
    })
COMPACT_SUMMARY = pd.DataFrame(
    compact_rows
)
COMPACT_SUMMARY.to_csv(
    STAT_DIR
    / "11_paper_compact_statistical_summary.csv",
    index=False,
)
print(
    "\n[Paper compact statistical summary]"
)
display(
    COMPACT_SUMMARY
)


# ======================================================================
# 18. Figure 1:

# ======================================================================
rank_matrix = (
    MUFASA_HORIZON
    .pivot(
        index="Site",
        columns="Hour",
        values="Rank_RMSE",
    )
    .reindex(
        index=SITES,
        columns=TARGET_HOURS,
    )
)
fig, ax = plt.subplots(
    figsize=(
        11.5,
        5.0,
    )
)
image = ax.imshow(
    rank_matrix.to_numpy(),
    aspect="auto",
    vmin=1,
)
ax.set_xticks(
    np.arange(
        len(
            rank_matrix.columns
        )
    )
)
ax.set_xticklabels([
    f"{int(hour):02d}:00"
    for hour
    in rank_matrix.columns
])
ax.set_yticks(
    np.arange(
        len(
            rank_matrix.index
        )
    )
)
ax.set_yticklabels(
    rank_matrix.index
)
ax.set_xlabel(
    "Forecast hour"
)
ax.set_ylabel(
    ""
)
ax.set_title(
    "MUFASA RMSE Rank Across Forecast Horizons"
)
cbar = fig.colorbar(
    image,
    ax=ax,
)
cbar.set_label(
    "RMSE rank (1 = best)"
)

# cell annotation
for i in range(
    rank_matrix.shape[0]
):

    for j in range(
        rank_matrix.shape[1]
    ):

        value = (
            rank_matrix
            .iloc[
                i,
                j,
            ]
        )

        ax.text(
            j,
            i,
            f"{value:.0f}",
            ha="center",
            va="center",
            fontsize=8,
        )
fig.tight_layout()
fig.savefig(
    STAT_DIR
    / "Figure_STAT_01_MUFASA_Horizon_Rank_Heatmap.png",
    dpi=600,
    bbox_inches="tight",
)
fig.savefig(
    STAT_DIR
    / "Figure_STAT_01_MUFASA_Horizon_Rank_Heatmap.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


# ======================================================================
# 19. Figure 2:

# ======================================================================


In [ ]:
# Multi-Horizon Statistical Analysis — part 15/16
rank1_plot = (
    MUFASA_HORIZON_SUMMARY
    .set_index(
        "Site"
    )
    .reindex(
        SITES
    )
    .reset_index()
)
fig, ax = plt.subplots(
    figsize=(
        8.5,
        4.8,
    )
)
ax.bar(
    rank1_plot[
        "Site"
    ],
    rank1_plot[
        "Rank1_Horizons"
    ],
)
ax.axhline(
    N_HORIZONS,
    linestyle="--",
    linewidth=1.0,
)
ax.set_ylim(
    0,
    N_HORIZONS + 0.8,
)
ax.set_ylabel(
    "Number of rank-1 horizons"
)
ax.set_xlabel(
    ""
)
ax.set_title(
    "Number of Forecast Horizons Ranked First by MUFASA"
)
ax.grid(
    axis="y",
    alpha=0.25,
)
fig.tight_layout()
fig.savefig(
    STAT_DIR
    / "Figure_STAT_02_Rank1_Horizons.png",
    dpi=600,
    bbox_inches="tight",
)
fig.savefig(
    STAT_DIR
    / "Figure_STAT_02_Rank1_Horizons.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


# ======================================================================
# 20. Figure 3:

# ======================================================================
joint_heat = (
    JOINT_TESTS
    .pivot(
        index="Site",
        columns="Comparator",
        values="Strong_joint_support",
    )
    .reindex(
        index=SITES,
        columns=COMPARATORS,
    )
    .astype(float)
)
fig, ax = plt.subplots(
    figsize=(
        max(
            12,
            0.65
            * len(
                COMPARATORS
            ),
        ),
        5.2,
    )
)
image = ax.imshow(
    joint_heat.to_numpy(),
    aspect="auto",
    vmin=0,
    vmax=1,
)
ax.set_xticks(
    np.arange(
        len(
            joint_heat.columns
        )
    )
)
ax.set_xticklabels(
    joint_heat.columns,
    rotation=65,
    ha="right",
    fontsize=8,
)
ax.set_yticks(
    np.arange(
        len(
            joint_heat.index
        )
    )
)
ax.set_yticklabels(
    joint_heat.index
)
ax.set_title(
    "Joint Multi-Horizon Statistical Support for MUFASA Superiority"
)
cbar = fig.colorbar(
    image,
    ax=ax,
)
cbar.set_ticks(
    [0, 1]
)
cbar.set_ticklabels(
    [
        "Not fully supported",
        "Supported",
    ]
)
fig.tight_layout()
fig.savefig(
    STAT_DIR
    / "Figure_STAT_03_Joint_Support_Heatmap.png",
    dpi=600,
    bbox_inches="tight",
)
fig.savefig(
    STAT_DIR
    / "Figure_STAT_03_Joint_Support_Heatmap.pdf",
    bbox_inches="tight",
)
plt.show()
plt.close(fig)


# ======================================================================

# ======================================================================
summary_lines = []
summary_lines.append(
    "# Oracle MUFASA Multi-Horizon Statistical Summary"
)
summary_lines.append("")
summary_lines.append(
    f"External benchmark models evaluated: {len(COMPARATORS)}."
)
summary_lines.append("")


In [ ]:
# Multi-Horizon Statistical Analysis — part 16/16
for site in SITES:

    horizon_row = (
        MUFASA_HORIZON_SUMMARY[
            MUFASA_HORIZON_SUMMARY[
                "Site"
            ].eq(site)
        ]
        .iloc[0]
    )

    joint_site = (
        JOINT_TESTS[
            JOINT_TESTS[
                "Site"
            ].eq(site)
        ]
    )

    mcs_row = (
        MCS_SUMMARY[
            MCS_SUMMARY[
                "Site"
            ].eq(site)
        ]
        .iloc[0]
    )

    summary_lines.append(
        f"## {site}"
    )

    summary_lines.append(
        f"- Rank-1 horizons: "
        f"{int(horizon_row['Rank1_Horizons'])}/{N_HORIZONS}."
    )

    summary_lines.append(
        f"- Mean horizon RMSE rank: "
        f"{horizon_row['Mean_RMSE_Rank']:.2f}."
    )

    summary_lines.append(
        f"- Joint GDM significant after Holm: "
        f"{int(joint_site['GDM_significant_Holm'].sum())}"
        f"/{len(COMPARATORS)} comparisons."
    )

    summary_lines.append(
        f"- Strong joint support "
        f"(numeric + GDM + HAC + block bootstrap): "
        f"{int(joint_site['Strong_joint_support'].sum())}"
        f"/{len(COMPARATORS)} comparisons."
    )

    summary_lines.append(
        f"- 95% average-path MCS retained models: "
        f"{mcs_row['Retained_models']}."
    )

    summary_lines.append("")
summary_text = "\n".join(
    summary_lines
)
with open(
    STAT_DIR
    / "paper_statistical_summary.md",
    "w",
    encoding="utf-8",
) as file:

    file.write(
        summary_text
    )
print("\n")
print(summary_text)


# ======================================================================
# 22. README / method note
# ======================================================================
method_note = """
MUFASA Oracle MUFASA Statistical Analysis

Primary inferential unit:
- daily prediction errors

Primary tests:
1. Horizon-specific paired Wilcoxon signed-rank tests
2. Horizon-specific Newey-West HAC predictive-accuracy tests
3. 7-day moving-block bootstrap confidence intervals
4. Joint 11-horizon generalized DM test
5. Holm family-wise multiplicity correction

Supplementary:
6. Friedman test using the 11 horizon RMSE values as repeated blocks
7. Pairwise Wilcoxon comparison of the 11 horizon RMSE values
8. 95% block-bootstrap Tmax Model Confidence Set based on daily
   average 11-horizon squared loss

Interpretation:
- Loss differential = MUFASA squared error - comparator squared error.
- Negative loss differential favors MUFASA.
- Block CI entirely below zero favors MUFASA.
- Horizon-RMSE Friedman/Wilcoxon results are supplementary rank-consistency
  analyses because neighboring forecast horizons are not independent datasets.
- Daily-loss HAC/GDM/bootstrap results are the primary inferential evidence.
- The implemented MCS is an average-path block-bootstrap Tmax MCS and should
  not be described as an exact implementation of Quaedvlieg's uniform
  multi-horizon MCS.
"""
with open(
    STAT_DIR
    / "README_statistical_method.txt",
    "w",
    encoding="utf-8",
) as file:

    file.write(
        method_note.strip()
    )


# ======================================================================
# 23. manifest
# ======================================================================
manifest_rows = []
for path in sorted(
    STAT_DIR.iterdir()
):

    if path.is_file():

        manifest_rows.append({
            "File": path.name,
            "Size_bytes":
                path.stat().st_size,
        })
MANIFEST = pd.DataFrame(
    manifest_rows
)
MANIFEST.to_csv(
    STAT_DIR
    / "STAT_output_manifest.csv",
    index=False,
)


# ======================================================================
# 24. ZIP
# ======================================================================
zip_path = shutil.make_archive(
    str(STAT_DIR),
    "zip",
    root_dir=STAT_DIR,
)
print("\n" + "=" * 86)
print("STATISTICAL ANALYSIS COMPLETE")
print("=" * 86)
print(
    "Folder:",
    STAT_DIR.resolve(),
)
print(
    "ZIP   :",
    zip_path,
)
print("Primary manuscript outputs:")
print("  01_horizon_metrics_all_models.csv")
print("  02_MUFASA_horizon_rank_summary.csv")
print("  03_horizon_specific_statistical_tests.csv")
print("  04_horizon_superiority_summary.csv")
print("  05_joint_11h_multihorizon_tests.csv")
print("  06_Friedman_horizon_rank_analysis.csv")
print("  08_model_confidence_set_95pct.csv")
print("  10_MCS_summary.csv")
print("  11_paper_compact_statistical_summary.csv")
print("  paper_statistical_summary.md")
print("=" * 86)
